# BC-01. Prognoza popytu tygodniowego dla dosyłek do sklepów Nordvik

| | |
|---|---|
| **Autor** | agent AI (materiał szkoleniowy) |
| **Zespół** | Data Science, zlecenie dla merchandisingu i alokacji |
| **Data wykonania notebooka** | 2026-11-10 (stan danych: do niedzieli 2026-09-20; punkt prognozy: poniedziałek 2026-09-21) |
| **Dane** | syntetyczne (wyciąg z hurtowni Nordvik), patrz sekcja 3 |
| **Język / środowisko** | polski; Python 3.12, wersje bibliotek w sekcji 2 |

## 1.1. Zlecenie biznesowe (własnymi słowami)

**Kto pyta:** kierownik merchandisingu i alokacji w Nordvik, polskiej sieci odzieżowej (5 sklepów stacjonarnych i sklep internetowy, własna marka, sezony SS i AW).

**O co pyta:** dziś dosyłki do sklepów planuje się na podstawie średniej sprzedaży z ostatnich 4 tygodni, korygowanej „na oko" przez planistów. Przy zmianie sezonu i wejściu nowych kolekcji ta metoda zawodzi: raz brakuje towaru (np. kurtek), raz jest go za dużo. Pytanie: **czy da się prognozować lepiej i o ile lepiej?**

**Jaka decyzja za tym stoi:** ile sztuk wysłać do każdej lokalizacji (sklepy i online) na najbliższe 4 tygodnie.

**Termin:** poniedziałek 2026-09-21. Plan dosyłek ma objąć tygodnie **2026-W39 do 2026-W42** (21 IX – 18 X 2026).

**Zastrzeżenia od zleceniodawcy:** dział cen potwierdził, że w tym okresie nie planuje akcji rabatowych. Zlecenie dotyczy głównie najlepiej sprzedających się modeli (top 20).

**Czego zleceniodawca nie określił** (to pytania do niego; na czas pracy przyjmuję jawne założenia): poziom agregacji prognozy, sposób liczenia błędu, czas realizacji dosyłki. Pełna lista pytań jest w dodatku (sekcja 8).

## 1.2. Cel analizy

1. Zmierzyć, **jak dobra jest dzisiejsza praktyka** (średnia z 4 tygodni) w uczciwym teście na przeszłych danych.
2. Sprawdzić, czy prostsze lub bardziej złożone metody dają **zmierzoną** poprawę, i z jaką niepewnością.
3. Dać zleceniodawcy **prognozę 4 tygodni dla top 20 modeli × lokalizacja** wraz z przedziałem niepewności oraz **rekomendację na poniedziałek**, razem z listą rzeczy, których prognoza nie rozstrzyga.

## 1.3. Spis treści i jak czytać ten notebook

1. Metadata & Purpose (tu jesteś; zlecenie, doprecyzowanie problemu, hipotezy, wyniki)
2. Setup
3. Load Data (opis danych, definicje metryk, wczytanie)
4. Explore (audyt jakości danych, eksploracja pod hipotezy)
5. Transform (panel tygodniowy, kalendarz, flagi)
6. Analyze (hipotezy i walidacja, baseline'y, modele, diagnostyka, przegląd krytyczny)
7. Results & Conclusions (prognoza na W39–W42, rekomendacja, ograniczenia, przekazanie)
8. Appendix (dziennik pracy, pytania do zleceniodawcy, post-mortem, ślepe uliczki)

Konwencja: przed każdym krokiem jest akapit **Po co**, po wyniku **Co widzę** (fakty z wyniku) i **Wniosek / decyzja** (interpretacja i dalszy ruch). Założenia i niepewności są oznaczone słowami **ZAŁOŻENIE** i **NIEPEWNOŚĆ**. Komórki z interpretacją zostały napisane po zobaczeniu wyników odpowiedniego kroku, w kolejności pracy. Gdzie coś mnie zaskoczyło lub pomysł się nie sprawdził, jest to opisane, a nie usunięte.

## 1.4. Pojęcia używane w notebooku

- **Tydzień handlowy:** pełny tydzień ISO-8601 (poniedziałek–niedziela). Grupuję po parze (`IsoYear`, `IsoWeek`), nigdy po `Year` + `IsoWeek`, bo na przełomie roku to daje błędne tygodnie.
- **Origin:** poniedziałek, w którym „stoimy" i robimy prognozę. Wolno użyć danych tylko do niedzieli poprzedzającej origin.
- **Rolling-origin backtest:** ocena na przeszłości, która powtarza prawdziwą sytuację: dla kolejnych poniedziałków (origin) uczymy się na danych do tego dnia, prognozujemy kolejne 4 tygodnie i porównujemy z tym, co się faktycznie sprzedało. Żadnego losowego podziału, bo mieszałby przyszłość z przeszłością.
- **WAPE:** suma błędów bezwzględnych podzielona przez sumę sprzedaży rzeczywistej, liczona na całym zbiorze (nie średnia z WAPE serii). Intuicja: „o ile procent sztuk w sumie się mylimy". Źródło definicji: standardowa miara prognostyczna przyjęta w tej analizie (słownik metryk Nordvik definiuje metryki sprzedażowe, nie błędy prognoz).
- **Bias (obciążenie):** (suma prognoz − suma rzeczywistych) / suma rzeczywistych. Dodatni = zawyżamy (ryzyko nadmiaru towaru), ujemny = zaniżamy (ryzyko braków).
- **Seria:** sprzedaż tygodniowa jednego modelu (`StyleCode`) w jednej lokalizacji. Model = styl produktu; SKU = model × kolor × rozmiar.
- **Lift, ablacja, pooling, zimny start:** wyjaśnione przy pierwszym użyciu.

## 1.5. Przełożenie zlecenia na problem prognostyczny (doprecyzowanie)

**ZAŁOŻENIE (poziom prognozy):** prognozuję **sprzedane sztuki w tygodniu ISO na poziomie `StyleCode` × lokalizacja** (5 sklepów + online). Uzasadnienie:
- poziom SKU × dzień jest zbyt rzadki (dużo zer, szum losowy dominuje nad sygnałem), a rozkład na kolory i rozmiary można zrobić później prostym kluczem udziałów;
- poziom regionu lub kategorii jest zbyt zgrubny: dosyłka jedzie do konkretnego sklepu i konkretnego modelu.

**Horyzont:** tygodnie t+1 … t+4 od poniedziałku-origin, oraz **suma 4 tygodni** (to ona odpowiada na pytanie „ile wysłać na 4 tygodnie").

**Miara sukcesu (główna):** **WAPE sumy 4 tygodni na poziomie `StyleCode` × lokalizacja**, agregowane po wszystkich seriach i punktach origin. Dodatkowo: WAPE osobno dla horyzontów 1–4, bias, pokrycie i szerokość przedziału 80%.

**ZAŁOŻENIE (top 20):** 20 modeli o największej liczbie sprzedanych sztuk w **8 tygodniach przed origin** (łącznie po wszystkich lokalizacjach). Czułość wyniku na tę definicję sprawdzę w diagnostyce.

**Baseline = dzisiejsza praktyka:** średnia tygodniowa z ostatnich 4 pełnych tygodni (B1). Odtwarzam ją dokładnie i mierzę.

**Kryterium wewnętrzne „WAPE < 30% dla top 20":** nie przyjmuję go z góry. Sprawdzę, czy jest osiągalne na poziomie model × lokalizacja (małe liczby sztuk na serię = duży szum losowy) i czy ma sens jako cel.

**ZAŁOŻENIE (czas realizacji dosyłki):** nieznany; zakładam, że decyzja dotyczy całego okna 4 tygodni naraz (tydzień po tygodniu nie da się już korygować). To zawęża interpretację: liczy się suma 4 tygodni, a błędy poszczególnych tygodni są pomocnicze.

**ZAŁOŻENIE (brak akcji rabatowych):** zgodnie z deklaracją działu cen w W39–W42 nie ma promocji, więc prognozuję popyt w warunkach „bez nowych akcji". Rabaty sezonowe wynikające z cyklu życia produktu (po ok. 8 tygodniach od premiery) to inna rzecz; będę sprawdzał, czy dane pokazują takie rabaty i jak je traktować.

**Czego prognoza nie rozstrzyga:** rozbicia na rozmiary i kolory, minimalnych stanów ekspozycyjnych, stanów na magazynach (dane o dostawach i przesunięciach nie istnieją).

# 2. Setup

**Po co:** zamrożenie środowiska i ziaren losowych, żeby ktoś inny odtworzył wynik z pustego środowiska. Jeśli ustawiona jest zmienna środowiskowa `MLFLOW_TRACKING_URI`, eksperymenty mogłyby być logowane do MLflow; w przeciwnym razie wyniki zbieram w tabeli w notebooku (tak jest tutaj, patrz wynik poniżej). Żadnych haseł ani adresów usług w kodzie.

In [ ]:
import os, sys, platform, warnings, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib
import matplotlib.pyplot as plt
import scipy, sklearn, statsmodels, lightgbm, holidays
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 20260921                       # jedno ziarno dla wszystkiego, co losowe (bootstrap, modele)
RNG = np.random.default_rng(SEED)

# Korzeń repozytorium wyznaczam względem notebooka, żeby uruchomienie działało z dowolnego katalogu roboczego
HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "datasets" / "synthetic" / "bc01").exists())
DATA = REPO / "datasets" / "synthetic" / "bc01"
CACHE = REPO / "analizy" / "bc01-prognoza-popytu" / "cache"   # ciężkie wyniki pośrednie
SQLDIR = REPO / "analizy" / "bc01-prognoza-popytu" / "sql"
CACHE.mkdir(exist_ok=True)

# Stałe sytuacji decyzyjnej
LAST_DATA_DAY = pd.Timestamp("2026-09-20")   # ostatni dzień danych sprzedażowych
ORIGIN_NOW = pd.Timestamp("2026-09-21")      # poniedziałek, w którym planujemy dosyłki
H = 4                                        # horyzont w tygodniach

plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3, "figure.figsize": (9, 4)})
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.3f}".format)

print("Python", platform.python_version())
for m in (np, pd, duckdb, matplotlib, scipy, sklearn, statsmodels, lightgbm, holidays):
    print(f"{m.__name__:12s}", m.__version__)
print("MLFLOW_TRACKING_URI ustawiona:", bool(os.getenv("MLFLOW_TRACKING_URI")))
print("Dane:", DATA.relative_to(REPO))

: 

# 3. Load Data

## 3.1. Opis danych

Wyciąg z hurtowni danych Nordvik w plikach Parquet (jeden plik na tabelę). Waluta PLN, VAT 23%, ceny `UnitPrice` i `ListPrice` są brutto.

| Tabela | Ziarno (jeden wiersz to…) | Kluczowe kolumny |
|---|---|---|
| `dim_date` | dzień kalendarzowy | `DateKey` (yyyymmdd), `Date`, `IsoYear`, `IsoWeek`, `YearWeek` (np. `2026-W38`), `DayOfWeek` (1 = poniedziałek), `Season` |
| `dim_store` | lokalizacja (5 sklepów + online) | `StoreKey`, `StoreCode`, `City`, `Channel` (`STORE`/`ONLINE`), `SalesAreaM2` (metraż **bieżący**), `OpenedDate`, `RemodelDate` |
| `dim_product` | SKU (model × kolor × rozmiar) | `ProductKey`, `SKU`, `StyleCode` (model), `Category`, `Color`, `Size`, `ListPrice` (bieżący), `UnitCost`, `IsActive` |
| `merch_plan` | model | `StyleCode`, `Collection` (`SS25`, `AW25`, `SS26`, `AW26`, `BASIC`), `LaunchDate` (**planowana** premiera) |
| `fact_sales` | pozycja paragonu lub zamówienia | `DateKey`, `ProductKey`, `StoreKey`, `TransactionNo`, `LineNumber`, `Quantity`, `UnitPrice`, `DiscountAmount`, `GrossAmount`, `NetAmount` |
| `fact_inventory_daily` | dzień × SKU × lokalizacja (migawka na koniec dnia) | `DateKey`, `ProductKey`, `StoreKey`, `StockQuantity` |

**Zakres i znane ograniczenia (udokumentowane przez firmę):**
- Sprzedaż: 2025-01-01 … 2026-09-20. Stany magazynowe: tylko od sierpnia 2026 (bieżący sezon), więc nie da się nimi sprawdzić braków towaru w całej historii.
- `fact_inventory_daily` to **migawka**: nie sumuję stanów z kolejnych dni, a różnica stanów nie jest sprzedażą (wpływają na nią dostawy i przesunięcia, o których nie ma danych).
- `ListPrice` i `SalesAreaM2` są bieżące, nie historyczne. Brak historii cennika, kalendarza promocji, danych o klientach, ruchu na stronie i pogodzie. Rabaty widać wyłącznie w `DiscountAmount`.
- `merch_plan` pochodzi z systemu planowania; daty premier są planowane, faktyczne mogą się różnić.
- Wiersze sprzedaży powstają przy pełnym przeładowaniu hurtowni, więc duplikaty są możliwe (sprawdzam w audycie).
- Przebudowa sklepu Kraków Rynek około 2026-04-01; przebieg prac nieopisany.
- Sklepy stacjonarne nie sprzedają w święta państwowe i w większość niedziel (otwarte są niedziele handlowe); sklep internetowy działa codziennie. **To deklaracja firmy, sprawdzam ją w danych.**

## 3.2. Definicje metryk i ich źródło

| Metryka | Definicja | Źródło definicji |
|---|---|---|
| Sztuki | `SUM(Quantity)` | słownik metryk Nordvik (sekcja „Sprzedaż") |
| Sprzedaż netto | `SUM(NetAmount)`; „sprzedaż" bez doprecyzowania = netto przed zwrotami | słownik metryk Nordvik |
| Stopa rabatu | `SUM(DiscountAmount) / SUM(Quantity × UnitPrice)` | słownik metryk Nordvik |
| Tydzień handlowy | pełny tydzień ISO, pon–niedz.; grupowanie po `IsoYear` + `IsoWeek` | słownik metryk Nordvik („Kalendarz") |
| Stan magazynowy | `StockQuantity`: sztuki na koniec dnia, migawka | słownik metryk Nordvik („Zapas i dostępność") |
| WAPE, bias, pokrycie przedziału | jak w sekcji 1.4 | definicje przyjęte w tej analizie (miary prognostyczne, spoza słownika metryk) |

Zwroty (`fact_returns`) nie są częścią tego wyciągu, więc **„popyt" oznacza tu sprzedane sztuki przed zwrotami**.

## 3.3. Wczytanie

**Po co:** wczytuję tabele (pliki są małe, ok. 0,56 mln wierszy sprzedaży), wypisuję wymiary i podgląd. Agregacje robię w DuckDB (SQL na plikach Parquet), a wynik analizuję w pandas.

In [ ]:
con = duckdb.connect()   # baza w pamięci; nic nie zapisuję poza plikami w analizy/
tables = ["dim_date", "dim_store", "dim_product", "merch_plan", "fact_sales", "fact_inventory_daily"]
for t in tables:
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{(DATA / (t + '.parquet')).as_posix()}')")

dfs = {t: con.execute(f"SELECT * FROM {t}").df() for t in tables}
for t, d in dfs.items():
    print(f"{t:22s} wierszy={len(d):>8,}  kolumn={d.shape[1]}")
dim_date, dim_store, dim_product, merch_plan, fact_sales, fact_inv = (dfs[t] for t in tables)
display(dim_store)
display(merch_plan.head(5))
display(fact_sales.head(5))
display(fact_inv.head(5))
display(dim_product.head(5))

**Co widzę:** sprzedaż liczy ok. 562 tys. pozycji paragonowych, katalog to 312 SKU, plan kolekcji (`merch_plan`) ma 36 wierszy (modeli), a lokalizacji jest 6 (5 sklepów + online). Stany magazynowe to tylko ok. 60 tys. wierszy (sierpień–wrzesień 2026). Cena i koszt są na SKU, a tygodniowe serie trzeba będzie zbudować na poziomie modelu (`StyleCode`). Sklep Kraków Rynek ma w wymiarze metraż 320 m² (bieżący po przebudowie) i datę przebudowy 2026-04-01.

**Wniosek:** jeśli 36 modeli z planu pokrywa się z katalogiem (sprawdzę w audycie), panel to maksymalnie 36 × 6 = 216 serii. To mało, więc „top 20 modeli" to ponad połowa katalogu, a nie filtr o dużej selektywności. Zanim cokolwiek zagreguję, robię audyt jakości danych.

# 4. Explore

## 4.1. Audyt: ziarno, klucze, duplikaty, zgodność arytmetyczna

**Po co:** pliki powstają przy pełnym przeładowaniu hurtowni, więc mogą zawierać zdublowane wiersze; zdublowana sprzedaż zawyżyłaby popyt i zepsuła zarówno model, jak i ocenę. Sprawdzam też, czy kwoty zgadzają się z regułami: `GrossAmount = Quantity × UnitPrice − DiscountAmount` oraz `NetAmount = GrossAmount / 1,23`.

In [ ]:
s = fact_sales
print("Zakres DateKey:", s.DateKey.min(), "…", s.DateKey.max())
print("SalesKey unikalny:", s.SalesKey.is_unique)
print("Braki w kolumnach (liczba NULL):", s.isna().sum()[lambda x: x > 0].to_dict() or "brak")
print("LoadId:", s.LoadId.value_counts().to_dict())

# Duplikat biznesowy: ta sama linia tego samego paragonu
d1 = s.duplicated(["TransactionNo", "LineNumber"], keep=False)
# Duplikat pełny: identyczny wiersz poza technicznym SalesKey
cols_wo_key = [c for c in s.columns if c != "SalesKey"]
d2 = s.duplicated(cols_wo_key, keep=False)
print(f"Wiersze w grupach zdublowanych wg (TransactionNo, LineNumber): {d1.sum()}")
print(f"Wiersze w grupach zdublowanych wg wszystkich kolumn poza SalesKey: {d2.sum()}")

# Zgodność arytmetyczna (tolerancja na zaokrąglenia do grosza)
gross_calc = s.Quantity * s.UnitPrice - s.DiscountAmount
net_calc = s.GrossAmount / 1.23
print("max |Gross - (Q*P - Disc)|:", float((gross_calc - s.GrossAmount).abs().max()))
print("max |Net - Gross/1.23|   :", float((net_calc - s.NetAmount).abs().max()))
print("Quantity:", s.Quantity.describe().round(2).to_dict())
print("Quantity <= 0:", int((s.Quantity <= 0).sum()), "| Discount < 0:", int((s.DiscountAmount < 0).sum()))

# Integralność kluczy obcych
print("DateKey bez dim_date   :", int((~s.DateKey.isin(dim_date.DateKey)).sum()))
print("ProductKey bez dim_prod:", int((~s.ProductKey.isin(dim_product.ProductKey)).sum()))
print("StoreKey bez dim_store :", int((~s.StoreKey.isin(dim_store.StoreKey)).sum()))

# Czy UnitPrice == ListPrice (README mówi, że cena jest stała w czasie)?
m = s.merge(dim_product[["ProductKey", "ListPrice"]], on="ProductKey")
print("UnitPrice != ListPrice:", int((m.UnitPrice != m.ListPrice).sum()))

**Co widzę:** zero duplikatów (ani wg klucza biznesowego `TransactionNo` + `LineNumber`, ani wg pełnych wierszy), brak NULL, kwoty zgodne z regułami (największa różnica `NetAmount` względem `GrossAmount / 1,23` to 0,005 zł, czyli zaokrąglenie do grosza), wszystkie klucze obce się łączą, a `UnitPrice` zawsze równa się `ListPrice`. Cała sprzedaż ma jeden `LoadId` (jedno pełne przeładowanie, 2026-09-21). Ilość na pozycji to zwykle 1 szt. (mediana i kwartyle 1), maksimum 12.

**Wniosek:** w tym wyciągu nie znalazłem problemów typu duplikaty ani błędy arytmetyczne, więc nie czyszczę wierszy. Zostaje mi sprawdzić pozycje o większej ilości (max 12) i, co ważniejsze, zagadnienia, które nie wychodzą z reguł arytmetycznych: kalendarz otwarcia, daty premier i pokrycie serii.

## 4.2. Audyt: kalendarz otwarcia lokalizacji

**Po co:** tydzień bez sprzedaży może znaczyć trzy różne rzeczy: sklep był zamknięty (święto, niedziela niehandlowa), nie było towaru, albo naprawdę nikt nie kupił. Od tego zależy, czy brak wiersza to zero popytu, czy brak informacji. Zaczynam od pytania, które dni sklepy w ogóle mają zerową sprzedaż i czy zgadza się to z deklaracją firmy (zamknięte w święta i w większość niedziel).

**Skąd kalendarz świąt:** hurtownia nie ma tabeli świąt, więc korzystam z **danych zewnętrznych**: biblioteki Pythona `holidays` (święta państwowe Polski, w tym ruchome: Wielkanoc, Boże Ciało; Wigilia jako dzień wolny od 2025 r. wg przepisów). Niedziele handlowe wyznaczam z przepisów ustawy o ograniczeniu handlu w niedziele (ostatnia niedziela stycznia, kwietnia, czerwca i sierpnia, niedziela przed Wielkanocą, od 2025 r. trzy niedziele przed Wigilią). To wiedza zewnętrzna wobec danych Nordvik; poniżej porównuję ją z tym, co faktycznie widać w sprzedaży.

In [ ]:
daily = con.execute("""
    SELECT d."Date" AS date, f.StoreKey AS store_key, SUM(f.Quantity) AS qty
    FROM fact_sales f JOIN dim_date d ON d.DateKey = f.DateKey
    GROUP BY d."Date", f.StoreKey
""").df()
daily["date"] = pd.to_datetime(daily["date"])
code = dim_store.set_index("StoreKey")["StoreCode"]
all_days = pd.date_range("2025-01-01", LAST_DATA_DAY)
q = daily.pivot(index="date", columns="store_key", values="qty").reindex(all_days).fillna(0)
q.columns = code.loc[q.columns].values

pl_hol = holidays.country_holidays("PL", years=[2025, 2026], language="pl")
hol_days = pd.DatetimeIndex(sorted(pd.Timestamp(d) for d in pl_hol))
is_sun = all_days.day_of_week == 6
is_hol = all_days.isin(hol_days)

rows = []
for st in q.columns:
    sold = q[st].values > 0
    rows.append({
        "lokalizacja": st,
        "dni_z_zerem_pon_sob": int((~sold & ~is_sun).sum()),
        "z_tego_swieta": int((~sold & ~is_sun & is_hol).sum()),
        "z_tego_inne": int((~sold & ~is_sun & ~is_hol).sum()),
        "niedziele_ze_sprzedaza": int((sold & is_sun).sum()),
        "niedziele_razem": int(is_sun.sum()),
        "swieta_ze_sprzedaza": int((sold & is_hol & ~is_sun).sum()),
    })
display(pd.DataFrame(rows))

# Niedziele ze sprzedażą w sklepach (które daty), żeby porównać z przepisami
sun_open = {st: [d.strftime("%Y-%m-%d") for d in all_days[(q[st].values > 0) & is_sun]] for st in q.columns}
for st, v in sun_open.items():
    print(st, len(v), v if len(v) <= 14 else v[:3] + ["…"] + v[-3:])

**Co widzę:** wszystkie 5 sklepów stacjonarnych ma identyczny wzorzec: zero sprzedaży w 18 dniach świątecznych przypadających od poniedziałku do soboty (i **tylko** wtedy: dni z zerem „z innych powodów" jest 0) oraz sprzedaż w dokładnie 13 niedzielach spośród 90. Sklep internetowy sprzedaje codziennie, także w 18 świętach i w 90 niedzielach. Daty 13 niedziel ze sprzedażą wyglądają jak niedziele handlowe z przepisów (ostatnie niedziele stycznia/kwietnia/czerwca/sierpnia, niedziela przed Wielkanocą, trzy niedziele grudnia 2025).

**Wniosek (do potwierdzenia kodem niżej):** zerowa sprzedaż sklepu w dzień ustawowo wolny to **zamknięcie, nie brak popytu**. Przy agregacji do tygodni nie ma „dziur", ale tygodnie ze świętem lub niedzielą handlową mają inną liczbę dni otwarcia, więc kalendarz otwarcia będzie cechą modelu (liczba dni handlowych w tygodniu). Sprawdzam jeszcze kodem, że obserwowane niedziele równają się zbiorowi wynikającemu z przepisów, a nie tylko „wyglądają podobnie".

In [ ]:
from dateutil.easter import easter

def last_sunday(year, month):
    end = pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)
    return end - pd.Timedelta(days=(end.day_of_week + 1) % 7)   # niedziela ma day_of_week = 6

def trading_sundays(year):
    """Niedziele handlowe wg przepisów: ostatnie niedziele I, IV, VI, VIII, niedziela przed Wielkanocą,
    od 2025 r. trzy niedziele przed Wigilią (zakładam, że przepis obowiązuje w całym 2025 i 2026)."""
    out = {last_sunday(year, m) for m in (1, 4, 6, 8)}
    out.add(pd.Timestamp(easter(year)) - pd.Timedelta(days=7))
    xmas_eve = pd.Timestamp(year, 12, 24)
    sundays_before = [xmas_eve - pd.Timedelta(days=d) for d in range(1, 30) if (xmas_eve - pd.Timedelta(days=d)).day_of_week == 6]
    out.update(sundays_before[:3])
    return out

rule_sundays = {d for y in (2025, 2026) for d in trading_sundays(y) if d <= LAST_DATA_DAY}
obs_sundays = {st: set(pd.to_datetime(v)) for st, v in sun_open.items() if st != "S-ONL-01"}
for st, v in obs_sundays.items():
    print(st, "obserwowane == z przepisów:", v == rule_sundays,
          "| tylko w danych:", sorted(v - rule_sundays), "| tylko w przepisach:", sorted(rule_sundays - v))

## 4.3. Audyt: daty premier (`LaunchDate`) a pierwsza sprzedaż, wygaszanie (`IsActive`)

**Po co:** zasada „seria zaczyna się w premierze" wymaga wiedzy, od kiedy model naprawdę jest w sprzedaży. `LaunchDate` to data planowana, więc porównuję ją z pierwszą faktyczną sprzedażą. Modele sprzedawane przed premierą lub po wygaszeniu (`IsActive = 0`) wymagają osobnej reguły. Sprawdzam też, czy katalog i plan kolekcji obejmują te same modele.

In [ ]:
style_info = (dim_product.groupby("StyleCode")
              .agg(Category=("Category", "first"), n_sku=("ProductKey", "size"), n_active_sku=("IsActive", "sum"),
                   list_price=("ListPrice", "first")).reset_index())
print("modele w dim_product:", style_info.StyleCode.nunique(), "| w merch_plan:", merch_plan.StyleCode.nunique(),
      "| wspólne:", len(set(style_info.StyleCode) & set(merch_plan.StyleCode)))

fs = con.execute("""
    SELECT p.StyleCode, MIN(d."Date") AS first_sale, MAX(d."Date") AS last_sale, SUM(f.Quantity) AS qty
    FROM fact_sales f JOIN dim_product p ON p.ProductKey = f.ProductKey JOIN dim_date d ON d.DateKey = f.DateKey
    GROUP BY p.StyleCode
""").df()
for c in ("first_sale", "last_sale"):
    fs[c] = pd.to_datetime(fs[c])
lp = merch_plan.assign(LaunchDate=pd.to_datetime(merch_plan.LaunchDate)).merge(fs, on="StyleCode", how="outer").merge(style_info, on="StyleCode", how="left")
lp["days_first_sale_minus_launch"] = (lp.first_sale - lp.LaunchDate).dt.days
lp["inactive"] = lp.n_active_sku < lp.n_sku

print("\nLiczba modeli wg kolekcji i kategorii:")
display(lp.groupby(["Collection"]).agg(modele=("StyleCode", "size"), start_min=("LaunchDate", "min"), start_max=("LaunchDate", "max"),
                                       pierwsza_sprzedaz_min=("first_sale", "min")))
print("Różnica (pierwsza sprzedaż − premiera planowana) w dniach, wg kolekcji:")
display(lp.groupby("Collection")["days_first_sale_minus_launch"].describe()[["count", "min", "50%", "max"]])
print("Modele z różnicą spoza [-1, 7] dni lub nieaktywne:")
display(lp[(lp.days_first_sale_minus_launch.abs() > 7) | (lp.days_first_sale_minus_launch < -1) | lp.inactive]
        [["StyleCode", "Category", "Collection", "LaunchDate", "first_sale", "last_sale", "days_first_sale_minus_launch", "n_active_sku", "n_sku"]])

**Co widzę:**
- Niedziele handlowe obserwowane w sklepach pokrywają się **dokładnie** z niedzielami wynikającymi z przepisów (5 sklepów × 13 dat, zero różnic w obie strony). Kalendarz otwarcia jest więc zrozumiany i zweryfikowany w danych.
- Katalog i plan kolekcji obejmują te same 36 modeli.
- 12 modeli `BASIC` ma premiery planowane w 2024 r., czyli przed początkiem danych (2025-01-01). Dla nich „pierwsza sprzedaż minus premiera" (100–340 dni) nie jest opóźnieniem, tylko **ucięciem danych z lewej strony**.
- Dla nowych kolekcji (SS25, AW25, SS26, AW26) pierwsza sprzedaż mieści się między 2 dniami **przed** planowaną premierą a 5 dniami po niej. Sprzedaż „przed premierą" to najwyżej 2 dni, przypadające na weekend poprzedzający poniedziałek premiery.
- Flaga `IsActive = 0` mają wszystkie SKU 14 modeli ze starszych kolekcji (SS25, AW25, SS26), a **mimo to 12 z nich sprzedaje się jeszcze we wrześniu 2026** (ostatnia sprzedaż 2026-09-15…20; pozostałe dwa w sierpniu). Flaga nie znaczy więc „model już nie sprzedawany".

**Wniosek / decyzja:**
1. Datę premiery z planu traktuję jako wiarygodną z dokładnością do kilku dni; start serii wyznaczam jednak z **pierwszej faktycznej sprzedaży** (tydzień, w którym seria ma pierwszą sprzedaż), nie z planu. Wiek modelu (tygodnie od premiery) liczę z planu, bo dla `BASIC` to jedyne źródło.
2. `IsActive` nie używam do filtrowania: jest to znacznik katalogowy, który nie opisuje faktycznej sprzedaży. Które modele wygasają, rozstrzygam po sprzedaży (patrz pokrycie serii niżej).
3. **NIEPEWNOŚĆ:** nie wiem, czy modele ze starszych kolekcji sprzedają się „ogonem" po obniżonych cenach (to sprawdzę w stopie rabatu), czy z innego powodu.

## 4.4. Audyt: pokrycie serii i znaczenie braku wiersza w danych

**Po co:** zanim uzupełnię brakujące tygodnie zerami, muszę rozstrzygnąć, co oznacza brak wiersza. Buduję tygodniowy panel `StyleCode × lokalizacja × tydzień ISO` (zapytanie w pliku `sql/weekly_sales_style_store.sql`; dane sprzedażowe przed zwrotami, sztuki zgodnie z definicją w słowniku metryk) i dla każdej serii liczę: pierwszy i ostatni tydzień ze sprzedażą oraz tygodnie z zerem pomiędzy nimi.

In [ ]:
sql_weekly = (SQLDIR / "weekly_sales_style_store.sql").read_text(encoding="utf-8")
wk = con.execute(sql_weekly, {"start_date": "2025-01-01", "end_date": "2026-09-21"}).df()
wk["week_start"] = pd.to_datetime(wk["week_start"])
print(f"Wiersze tygodniowe (model x lokalizacja x tydzień, tylko tam, gdzie była sprzedaż): {len(wk):,}")
print("Tygodni ISO w danych:", wk.week_start.nunique(), "| od", wk.week_start.min().date(), "do", wk.week_start.max().date())

weeks = pd.date_range(wk.week_start.min(), wk.week_start.max(), freq="W-MON")
styles = sorted(style_info.StyleCode)
stores_keys = list(dim_store.StoreKey)
grid = pd.MultiIndex.from_product([weeks, styles, stores_keys], names=["week_start", "style_code", "store_key"]).to_frame(index=False)
panel0 = grid.merge(wk[["week_start", "style_code", "store_key", "qty"]], how="left", on=["week_start", "style_code", "store_key"])
print(f"Pełna siatka: {len(panel0):,} wierszy ({len(styles)} modeli x {len(stores_keys)} lokalizacji x {len(weeks)} tygodni); "
      f"z danymi: {panel0.qty.notna().mean():.1%}")

# Pokrycie serii: pierwszy/ostatni tydzień ze sprzedażą, zera wewnątrz zakresu, najdłuższa przerwa
def coverage(g):
    a = g.sort_values("week_start")
    sold = a.qty.fillna(0).values > 0
    if not sold.any():
        return pd.Series({"first": pd.NaT, "last": pd.NaT, "weeks_sold": 0, "zeros_inside": 0, "longest_gap": 0})
    i0, i1 = np.where(sold)[0][[0, -1]]
    inside = ~sold[i0:i1 + 1]
    run = best = 0
    for z in inside:
        run = run + 1 if z else 0
        best = max(best, run)
    return pd.Series({"first": a.week_start.iloc[i0], "last": a.week_start.iloc[i1], "weeks_sold": int(sold.sum()),
                      "zeros_inside": int(inside.sum()), "longest_gap": best})

cov = panel0.groupby(["style_code", "store_key"]).apply(coverage, include_groups=False).reset_index()
cov = cov.merge(style_info[["StyleCode", "Category"]], left_on="style_code", right_on="StyleCode").drop(columns="StyleCode")
cov = cov.merge(merch_plan[["StyleCode", "Collection"]], left_on="style_code", right_on="StyleCode").drop(columns="StyleCode")
print("\nSerie bez żadnej sprzedaży:", int((cov.weeks_sold == 0).sum()), "z", len(cov))
print("Rozkład najdłuższej przerwy (tygodnie zera między pierwszą a ostatnią sprzedażą):")
print(cov.longest_gap.value_counts().sort_index().to_string())
display(cov.groupby(["store_key"]).agg(serie=("style_code", "size"), bez_przerw=("longest_gap", lambda x: int((x == 0).sum())),
                                       maks_przerwa=("longest_gap", "max"), zera_w_srodku=("zeros_inside", "sum")))

**Co widzę:** na 216 możliwych serii (36 modeli × 6 lokalizacji) **każda** ma jakąś sprzedaż, ale w siatce 90 tygodni wypełnionych danymi jest tylko 52% komórek. 166 serii nie ma żadnej przerwy między pierwszą a ostatnią sprzedażą, a 50 ma co najmniej jeden tydzień zera w środku; najdłuższa przerwa to 32 tygodnie (Kraków Rynek, `store_key` 3), przy czym przerwy zdarzają się we wszystkich lokalizacjach, nie tylko w jednej.

**Wniosek (częściowy):** „tydzień bez wiersza" w środku życia serii zdarza się często i nie jest wyjaśniony samym kalendarzem otwarcia (zamknięcia z kalendarza to pojedyncze dni, nie tygodnie). Zanim uzupełnię te tygodnie zerami, sprawdzam, co je powoduje: (a) model sezonowy, który po sezonie ma zerowy popyt, (b) przerwa dotykająca wszystkich lokalizacji naraz (decyzja asortymentowa), (c) przerwa w jednej lokalizacji przy normalnej sprzedaży w pozostałych (podejrzenie braku towaru).

In [ ]:
# Przebiegi zer wewnątrz życia serii (od pierwszej do ostatniej sprzedaży) o długości >= 3 tygodni
pn = panel0.assign(qty=panel0.qty.fillna(0)).sort_values(["style_code", "store_key", "week_start"])
runs = []
for (sc, sk), g in pn.groupby(["style_code", "store_key"]):
    v, w = g.qty.values, g.week_start.values
    nz = np.where(v > 0)[0]
    i0, i1 = nz[0], nz[-1]
    j = i0
    while j <= i1:
        if v[j] == 0:
            k = j
            while k <= i1 and v[k] == 0:
                k += 1
            level = v[max(i0, j - 8):j].mean()           # poziom 8 tygodni przed przerwą
            runs.append((sc, sk, pd.Timestamp(w[j]), pd.Timestamp(w[k - 1]), k - j, level))
            j = k
        else:
            j += 1
runs = pd.DataFrame(runs, columns=["style_code", "store_key", "gap_start", "gap_end", "gap_weeks", "level_before"])
runs = runs.merge(style_info[["StyleCode", "Category"]], left_on="style_code", right_on="StyleCode").drop(columns="StyleCode")
long_runs = runs[runs.gap_weeks >= 3].copy()
print("Przebiegi zer >= 3 tyg.:", len(long_runs), "w", long_runs.groupby(["style_code", "store_key"]).ngroups, "seriach;",
      "w tym z poziomem przed przerwą >= 2 szt./tydz.:", int((long_runs.level_before >= 2).sum()))
print("\nWg kategorii (liczba przebiegów >= 3 tyg., mediana długości, średni poziom przed):")
display(long_runs.groupby("Category").agg(przebiegi=("gap_weeks", "size"), mediana_dl=("gap_weeks", "median"),
                                           sr_poziom_przed=("level_before", "mean")).round(2))
print("Największe przerwy (z dużym poziomem sprzedaży przed):")
display(long_runs.sort_values("level_before", ascending=False).head(12))

**Co widzę:** przebiegi zer o długości ≥ 3 tygodni to 77 przypadków w 37 seriach (z 216). Występują tylko w czterech kategoriach: Akcesoria, Kurtki, Sukienki i T-shirty; nie ma ich w Jeansach, Koszulach i Swetrach. Poziom sprzedaży przed przerwą jest niski (średnio 0,7–1,4 szt./tydz. na serię), a tylko 8 z 77 przebiegów poprzedzał poziom ≥ 2 szt./tydz. Największe przerwy dotyczą kurtek (np. `M-JKT-002` w kilku sklepach równocześnie od kwietnia do lipca 2026).

**Wniosek (robocza interpretacja, nie dowód):** przy sprzedaży 1–3 sztuk na serię na tydzień zero jest po prostu możliwym wynikiem losowym (dla średniej 1 szt./tydz. szansa zera w tygodniu to ok. 37%, czyli seria trzech zer z rzędu zdarza się przypadkiem w ok. 5% trójek), a nakłada się na to **sezonowy koniec życia** modeli (kurtki od kwietnia do lipca). Nie widzę w tym zestawieniu wyraźnego śladu „wyprzedane przy wysokiej sprzedaży" (tylko 8 przebiegów z wyższym poziomem, głównie kurtki, więc sezonowo). Do tego wrócę po zbudowaniu panelu i po sprawdzeniu stanów magazynowych (od sierpnia 2026).

**Reguła uzupełniania zerami (robocza, do ostatecznego potwierdzenia po sprawdzeniu stanów):**
- **przed pierwszym tygodniem sprzedaży serii:** NIE zero, tylko „seria nie istnieje" (model nie jest jeszcze w sprzedaży w tej lokalizacji);
- **od pierwszej sprzedaży do końca danych:** brak wiersza = **zero sprzedaży** (sklepy są otwarte w każdym tygodniu co najmniej kilka dni, bo zamknięcia z kalendarza to pojedyncze dni, więc własne zero w tygodniu jest informacją o popycie, a nie o zamknięciu), z flagą „podejrzenie braku towaru" tam, gdzie da się to wesprzeć stanami;
- **pierwszy tydzień danych (2025-W01)** jest niepełny (dane od środy 2025-01-01), więc wykluczam go z treningu i oceny.

## 4.5. Skala i zarys sezonowości: sprzedaż tygodniowa wg kategorii i lokalizacji

**Po co:** odpowiedzieć na pytania: jak duże są serie (to determinuje szum losowy), czy widać sezonowość roczną (porównanie tych samych tygodni 2025 i 2026), jaki jest wzrost rok do roku, jak wyglądają kategorie i lokalizacje na tle siebie.

In [ ]:
cat_of = style_info.set_index("StyleCode").Category
pn["Category"] = pn.style_code.map(cat_of)
pn["store"] = pn.store_key.map(code)
iso = dim_date[["Date", "IsoYear", "IsoWeek"]].assign(Date=lambda d: pd.to_datetime(d.Date))
wk_iso = iso.groupby(["IsoYear", "IsoWeek"]).Date.min().rename("week_start").reset_index()   # poniedziałek tygodnia ISO
pn = pn.merge(wk_iso, on="week_start", how="left")

by_cat = pn.groupby(["week_start", "Category"]).qty.sum().unstack()
by_store = pn.groupby(["week_start", "store"]).qty.sum().unstack()

fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
by_cat.iloc[1:].plot(ax=ax[0], lw=1.4)
ax[0].set_title("Sprzedaż tygodniowa wg kategorii (wszystkie lokalizacje; bez niepełnego 2025-W01)")
ax[0].set_ylabel("sztuki / tydzień"); ax[0].legend(ncol=4, fontsize=8)
by_store.iloc[1:].plot(ax=ax[1], lw=1.4)
ax[1].axvline(pd.Timestamp("2026-04-01"), color="k", ls=":", lw=1); ax[1].text(pd.Timestamp("2026-04-05"), by_store.iloc[1:].max().max() * .95, "przebudowa Kraków", fontsize=8)
ax[1].set_title("Sprzedaż tygodniowa wg lokalizacji (wszystkie kategorie)")
ax[1].set_ylabel("sztuki / tydzień"); ax[1].set_xlabel("tydzień (poniedziałek)"); ax[1].legend(ncol=6, fontsize=8)
plt.tight_layout(); plt.show()

tot = pn.groupby("week_start").qty.sum()
print("Sprzedaż łączna: mediana tygodnia (bez W01) =", int(tot.iloc[1:].median()), "szt.;",
      "mediana serii w tygodniu (wszystkie serie, w tym zera) =", pn.qty.median(), "szt.")
print("Średnia sprzedaż serii (szt./tydz., ostatnie 8 pełnych tygodni, 216 serii):")
last8 = pn[pn.week_start >= pd.Timestamp("2026-07-27")]
ser8 = last8.groupby(["style_code", "store_key"]).qty.sum() / 8
print(ser8.describe().round(2).to_dict())

**Co widzę (wykres i liczby):**
- Skala: łącznie ok. 7 tys. szt. tygodniowo (mediana), z czego dominują T-shirty (3–5 tys. szt./tydz. w sezonie letnim). Mediana pojedynczej serii w tygodniu to 2 szt. (wliczając tygodnie przed premierą i zera), a średnia sprzedaż serii w ostatnich 8 tygodniach to ok. 41 szt./tydz. przy medianie ok. 22 i maksimum 353. Rozkład jest więc **silnie prawoskośny**: kilka serii niesie większość wolumenu, a wiele serii to kilka sztuk tygodniowo.
- Sezonowość roczna jest wyraźna: T-shirty mają szczyt w czerwcu i dołek w styczniu–lutym, Kurtki i Swetry ledwo istnieją latem i startują w sierpniu–wrześniu; wspólny szczyt widać pod koniec listopada (Black Friday) i spadki w tygodniach z dniami wolnymi (Boże Narodzenie, Nowy Rok).
- W ostatnich 3–4 tygodniach (od połowy sierpnia 2026) Swetry gwałtownie rosną: do ok. 3 tys. szt./tydz., podczas gdy w 2025 r. w tym samym okresie były na poziomie poniżej 1 tys. Kurtki też startują, ale mniej gwałtownie.
- Sklepy mają zbliżony kształt sezonowości; Kraków Rynek jest najniższy, a po przebudowie (kwiecień 2026) jego sprzedaż rośnie i zbliża się do poziomu pozostałych sklepów. Spadki wspólne dla wszystkich lokalizacji (np. początek stycznia) wyglądają na kalendarz, nie na zdarzenie lokalne.

**Wniosek:** (1) wolumen jest skupiony w niewielu seriach, więc ocena WAPE będzie zdominowana przez największe serie; (2) przy prognozie na W39–W42 kluczowa jest **zmiana sezonu** (Swetry, Kurtki rosną, T-shirty spadają); (3) skok Swetrów jest podejrzany co do przyczyny. Sprawdzam go na poziomie modeli i lokalizacji: jedno-modelowy efekt (premiera hitu) czy zjawisko całej kategorii, i czy występuje we wszystkich lokalizacjach naraz.

In [ ]:
# Rozkład skoku Swetrów: model x tydzień (wszystkie lokalizacje), potem lokalizacja x tydzień
sw = pn[(pn.Category == "Swetry") & (pn.week_start >= "2026-07-20")]
print("Swetry wg modelu (sztuki / tydzień, wszystkie lokalizacje):")
display(sw.pivot_table(index="style_code", columns="week_start", values="qty", aggfunc="sum").rename(columns=lambda c: c.strftime("%m-%d")))
print("Swetry wg lokalizacji (sztuki / tydzień, wszystkie modele):")
display(sw.pivot_table(index="store", columns="week_start", values="qty", aggfunc="sum").rename(columns=lambda c: c.strftime("%m-%d")))
# Ten sam okres rok wcześniej (tygodnie ISO 31–38)
sw25 = pn[(pn.Category == "Swetry") & (pn.IsoYear == 2025) & (pn.IsoWeek.between(31, 38))]
print("Swetry 2025, tygodnie ISO 31–38 (szt./tydz., wszystkie modele i lokalizacje):",
      sw25.groupby("IsoWeek").qty.sum().to_dict())

## 4.6. Wzrost rok do roku (te same tygodnie ISO) i wzór dnia tygodnia

**Po co:** sprawdzić (a) czy 2026 sprzedaje się lepiej niż 2025 i w jakich kategoriach (to wpływa na to, czy poziom z 2025 można przenosić na 2026), (b) czy kanał online i sklepy różnią się wzorem dnia tygodnia (to ma znaczenie dla interpretacji tygodni ze świętami i niedzielami handlowymi). Porównuję **te same tygodnie ISO 2026-W02…W37 i 2025-W02…W37** (pomijam W01, bo w 2025 jest niepełny, i W38, bo to ostatni pełny tydzień danych, w którym trwa zmiana sezonu).

In [ ]:
yoy = pn[pn.IsoWeek.between(2, 37) & pn.IsoYear.isin([2025, 2026])]
t = yoy.pivot_table(index="Category", columns="IsoYear", values="qty", aggfunc="sum")
t["YoY_%"] = (t[2026] / t[2025] - 1) * 100
t.loc["RAZEM"] = [t[2025].sum(), t[2026].sum(), (t[2026].sum() / t[2025].sum() - 1) * 100]
print("Sztuki W02–W37: 2025 vs 2026 wg kategorii (uwaga: w 2025 część modeli jeszcze nie istniała; premiery nowych kolekcji):")
display(t.round(1))
t2 = yoy.pivot_table(index="store", columns="IsoYear", values="qty", aggfunc="sum")
t2["YoY_%"] = (t2[2026] / t2[2025] - 1) * 100
display(t2.round(1))

# Wzór dnia tygodnia wg kanału: udział sztuk w poszczególnych dniach (cały okres)
dd = daily.merge(dim_store[["StoreKey", "Channel"]], left_on="store_key", right_on="StoreKey")
dd["dow"] = dd["date"].dt.day_of_week
share = dd.pivot_table(index="Channel", columns="dow", values="qty", aggfunc="sum")
share = (share.div(share.sum(axis=1), axis=0) * 100).round(1)
share.columns = ["pn", "wt", "śr", "cz", "pt", "sb", "nd"]
print("Udział (%) sztuk wg dnia tygodnia i kanału:")
display(share)

**Co widzę:**
- **Skok Swetrów to premiera nowej kolekcji AW26, nie anomalia danych.** Przyczyną są cztery nowe modele (`M-KNT-001`, `M-KNT-002`, `W-KNT-001`, `W-KNT-003`), które wchodzą kolejno od połowy do końca sierpnia i **rosną jeszcze w tygodniu 09-14** (np. `M-KNT-002`: 52 → 354 → 379 → 498 → 622 szt.). Skok jest widoczny **we wszystkich 6 lokalizacjach równocześnie** (w każdej od ok. 56–131 do ok. 407–600 szt./tydz.), czyli to zdarzenie asortymentowe (premiera), a nie błąd jednego źródła. Stary model `W-KNT-002` (AW25) sprzedaje stałe ok. 7–10 szt./tydz.
- W 2025 r. kategoria Swetry w tygodniach 34–38 dochodziła do 968 szt./tydz.; w 2026 r. ok. 3 tys. Porównanie kategorii rok do roku jest jednak **zafałszowane liczbą modeli**: w 2026 r. mamy 4 nowe swetry AW26 zamiast mniejszej oferty AW25.
- Wzrosty rok do roku (W02–W37): razem +25,9%, ale w kategoriach od +9% (Jeansy) i +12,5% (T-shirty) do +66% (Koszule), +125% (Kurtki), +298% (Swetry). Różnice w ostatnich trzech to w dużej mierze **efekt nowych modeli w katalogu**, a nie wzrost popytu na ten sam asortyment.
- Wzór dnia tygodnia: online sprzedaje najwięcej w niedzielę (17%) i poniedziałek (15,9%) z płaskim środkiem tygodnia; sklepy sprzedają najwięcej w sobotę (25,3%) i piątek (18,6%), a niedziela to tylko 3,3% (13 niedziel handlowych na 90).

**Wniosek / decyzja:**
1. Skok Swetrów w sierpniu–wrześniu 2026 **zostaje w danych** (to prawdziwy wzrost popytu na nowe modele) i będzie głównym testem dla każdej metody: dopiero się rozpędza, więc prognoza „średnia z 4 tygodni" będzie je zaniżać (to H1).
2. Wskaźnik wzrostu rok do roku trzeba liczyć na **tym samym asortymencie** (modele `BASIC`, istniejące przez cały okres), inaczej wzrost katalogu jest mylony ze wzrostem popytu. Robię to niżej.
3. Tygodnie ze świętem lub niedzielą handlową mają inną liczbę dni otwarcia sklepów, więc liczba dni handlowych w tygodniu będzie cechą modelu (wpływ zmierzę w ablacji).

## 4.7. Wzrost rok do roku na porównywalnym asortymencie (`BASIC`)

**Po co:** modele `BASIC` są w ofercie przez cały czas (premiery w 2024 r.), więc ich zmiana rok do roku to czysta zmiana popytu, bez wpływu zmian katalogu. Sprawdzam, czy wzrost jest stały w czasie (miesięcznie), bo od tego zależy, czy wolno przenosić poziom z 2025 r.

In [ ]:
basic = set(merch_plan.loc[merch_plan.Collection == "BASIC", "StyleCode"])
pb = pn[pn.style_code.isin(basic) & pn.IsoYear.isin([2025, 2026]) & (pn.IsoWeek.between(2, 37))]
tb = pb.pivot_table(index="Category", columns="IsoYear", values="qty", aggfunc="sum")
tb["YoY_%"] = (tb[2026] / tb[2025] - 1) * 100
tb.loc["RAZEM BASIC"] = [tb[2025].sum(), tb[2026].sum(), (tb[2026].sum() / tb[2025].sum() - 1) * 100]
print("BASIC (12 modeli), sztuki W02–W37:")
display(tb.round(1))

# Stabilność wzrostu w czasie: YoY per kwartał ISO (po tygodniach ISO)
pb = pb.assign(q=lambda d: pd.cut(d.IsoWeek, [1, 9, 18, 27, 37], labels=["W02-09", "W10-18", "W19-27", "W28-37"]))
tq = pb.pivot_table(index="q", columns="IsoYear", values="qty", aggfunc="sum", observed=True)
tq["YoY_%"] = (tq[2026] / tq[2025] - 1) * 100
print("BASIC: YoY w kolejnych blokach tygodni ISO:")
display(tq.round(1))

## 4.8. Promocje i rabaty: co widać w `DiscountAmount`

**Po co:** nie ma kalendarza promocji, a rabaty zmieniają sprzedaż (w danych firmy: pełna cena → rabat sezonowy po ok. 8 tygodniach → wyprzedaż). Dla prognozy 4 tygodni zleceniodawca zakłada brak nowych akcji; muszę więc zrozumieć, (a) czy rabaty są powiązane z wiekiem modelu (cyklem życia), (b) czy w W39–W42 któreś modele „wejdą" automatycznie w okno rabatu sezonowego (cykl życia), mimo braku akcji, i (c) jak rabat wpływa na sprzedaż. Stopa rabatu = `SUM(DiscountAmount) / SUM(Quantity × UnitPrice)` (definicja ze słownika metryk), liczona na poziomie model × tydzień.

In [ ]:
mp = merch_plan.assign(LaunchDate=pd.to_datetime(merch_plan.LaunchDate)).set_index("StyleCode")
ws = (wk.groupby(["style_code", "week_start"]).agg(qty=("qty", "sum"), list_value=("list_value", "sum"), discount=("discount", "sum")).reset_index())
ws["disc_rate"] = ws.discount / ws.list_value
ws["age_w"] = ((ws.week_start - ws.style_code.map(mp.LaunchDate)).dt.days // 7)
ws["collection"] = ws.style_code.map(mp.Collection)
bins = [-10, 7, 11, 15, 23, 39, 999]
ws["age_bucket"] = pd.cut(ws.age_w, bins, labels=["0-7", "8-11", "12-15", "16-23", "24-39", "40+"])
print("Stopa rabatu (średnia ważona wartością przed rabatem) wg wieku modelu w tygodniach od premiery (nowe kolekcje, bez BASIC):")
nb_ = ws[ws.collection != "BASIC"]
g = nb_.groupby("age_bucket", observed=True).apply(lambda d: pd.Series({"modelo-tygodnie": len(d), "stopa_rabatu": d.discount.sum() / d.list_value.sum(),
                                                       "udzial_tyg_z_rabatem>0": (d.disc_rate > 0).mean()}), include_groups=False)
display(g.round(3))
print("BASIC: rozkład stopy rabatu w modelo-tygodniach:")
print(ws[ws.collection == "BASIC"].disc_rate.describe().round(3).to_dict())

**Co widzę:**
- Wzrost rok do roku na porównywalnym asortymencie (`BASIC`, 12 modeli): **+9,2%** (Akcesoria +8,9%, Jeansy +9,1%, Koszule +5,8%, Sukienki +9,3%, T-shirty +10,0%) i jest **stabilny w czasie**: +8,2% / +9,7% / +8,9% / +9,7% w czterech blokach tygodni. Wniosek: wcześniejsze +26% to głównie efekt nowych modeli w katalogu, a organiczny wzrost popytu to ok. +9% rocznie.
- Rabaty: w nowych kolekcjach **żadnych rabatów w pierwszych 11 tygodniach od premiery** (stopa 0,0%), pierwsze pojawiają się w wieku 12–15 tygodni (stopa 1,7%, 5% modelo-tygodni), a największe w wieku 16–39 tygodni (stopa średnio 9,8% i 11,1%). W wieku 40+ rabaty znikają (0,6%). Dla `BASIC` rabatów praktycznie nie ma (średnia 0,2%, mediana 0, max 20%).
- Prosta interpretacja „po ok. 8 tygodniach rabat sezonowy" z opisu firmy w danych **nie potwierdza się co do czasu**: pierwsze rabaty widać dopiero w wieku 12+ tygodni, a udział modelo-tygodni z rabatem jest mały (5–27%), więc rabaty są epizodami, nie stanem ciągłym.

**Wniosek / decyzja (robocza):** stopa rabatu jest cechą zależną od cyklu życia modelu (wiek 12–39 tygodni), a nie od niczego, co można przewidzieć z zewnętrznego kalendarza. Zlecenie zakłada brak nowych akcji w W39–W42, ale **rabaty „cyklu życia" starszych kolekcji (SS26 w wieku ok. 28–32 tygodni) mogą się w tym okresie pojawić**; to NIEPEWNOŚĆ, której nie rozstrzygnę (brak historii cennika i planu cen). Poniżej oglądam przebieg epizodów rabatowych, żeby zmierzyć wpływ rabatu na sprzedaż i sprawdzić, co dzieje się teraz.

In [ ]:
nonb = ws[ws.collection != "BASIC"].copy()
piv = nonb.pivot_table(index="style_code", columns="age_w", values="disc_rate")
piv = piv.reindex(columns=range(0, 70))
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(piv.values, aspect="auto", cmap="Reds", vmin=0, vmax=0.6)
ax.set_yticks(range(len(piv))); ax.set_yticklabels([f"{i} ({mp.Collection[i]})" for i in piv.index], fontsize=7)
ax.set_xlabel("wiek modelu [tygodnie od planowanej premiery]"); ax.set_title("Stopa rabatu (model × wiek) — modele nowych kolekcji; białe = brak sprzedaży lub rabatu")
plt.colorbar(im, label="stopa rabatu (udział wartości przed rabatem)"); plt.tight_layout(); plt.show()

recent = nonb[nonb.week_start >= "2026-08-17"].pivot_table(index="style_code", columns="week_start", values="disc_rate").round(2)
recent.columns = [c.strftime("%m-%d") for c in recent.columns]
print("Stopa rabatu w ostatnich 5 tygodniach, modele nowych kolekcji z jakimkolwiek rabatem:")
display(recent[(recent > 0).any(axis=1)])

**Co widzę:** żadnego modelu nowych kolekcji **nie ma dziś** w oknie rabatowym (pusta tabela: w ostatnich 5 tygodniach, do 2026-09-14, stopa rabatu wszędzie 0). Mapa pokazuje, że rabaty układają się w **wąskie pasma** (3–4 tygodnie, stopa 20–45%), ale ich położenie względem wieku modelu jest różne w różnych kolekcjach (modele AW25 w wieku ok. 20–23 tygodni, SS25 i SS26 w wieku ok. 19–25), czyli nie jest to stała „tarcza" wieku. Wygląda to raczej na **okna kalendarzowe** (wspólne daty dla wielu modeli), a nie na regułę „po N tygodniach od premiery".

**Zmiana zdania:** moja pierwsza interpretacja (wniosek wyżej: „rabat zależy od wieku modelu") jest zbyt mocna i najprawdopodobniej błędna. Sprawdzam to wprost: gdyby rabaty miały charakter kalendarzowy, w tych samych tygodniach kalendarzowych rabat miałaby wiele modeli naraz.

In [ ]:
tot_w = ws.groupby("week_start").agg(disc=("discount", "sum"), val=("list_value", "sum"))
tot_w["rate"] = tot_w.disc / tot_w.val
n_disc = ws[ws.disc_rate > 0.05].groupby("week_start").style_code.nunique().rename("modele_z_rabatem>5%")
tw = tot_w.join(n_disc).fillna({"modele_z_rabatem>5%": 0})
print("Tygodnie z łączną stopą rabatu > 1% (cała sieć) albo z modelami rabatowanymi > 5%:")
display(tw[(tw.rate > 0.01) | (tw["modele_z_rabatem>5%"] > 0)].assign(rate=lambda d: (d.rate * 100).round(1)).rename(columns={"rate": "stopa_%"})[["stopa_%", "modele_z_rabatem>5%"]].T)

**Co widzę:** rabaty są **kalendarzowe**, nie zależne od wieku modelu. Są dokładnie trzy typy zdarzeń: (1) lipiec 2025 (tygodnie 07-14…07-28): wyprzedaż końca sezonu dla 3 modeli SS25; (2) **tydzień 2025-11-24** (Black Friday): rabat ok. 20% na **20 modeli** naraz; (3) styczeń 2026 (01-12…01-26): wyprzedaż 5 modeli AW25; (4) lipiec 2026 (07-13…07-27): wyprzedaż 6 modeli SS26. Łączna stopa rabatu sieci w tych tygodniach to 2–6% (lipiec) i 14–20% (BF, styczeń). Poza nimi rabatów praktycznie nie ma. W ostatnich tygodniach (od 2026-07-28) i w tym samym okresie rok wcześniej (wrzesień–październik 2025) nie ma żadnego okna wyprzedaży (żaden tydzień z łączną stopą rabatu > 1%).

**Wniosek / decyzja:**
1. Moja wcześniejsza interpretacja „rabat zależy od wieku" była **błędna** (mapa wieku mieszała okna kalendarzowe z różnymi datami premier); poprawiam ją: **rabaty = okna wyprzedaży końca sezonu (3 tyg.) dla kolekcji schodzącej, plus Black Friday dla całej oferty**. To zmienia sposób modelowania: promocję traktuję jako zdarzenie kalendarzowe dla modelu, a nie jako funkcję wieku.
2. W badanym oknie prognozy (W39–W42, 21 IX – 18 X 2026) analogiczny okres 2025 nie miał żadnej wyprzedaży, co jest zgodne z deklaracją działu cen. Rabaty nie wpływają więc na prognozę, ale **wpływają na ocenę** w backteście (okna styczniowe i lipcowe 2026 są w próbie testowej). Dlatego: wynik główny liczę na oknach **bez** wyprzedaży modelu (to scenariusz decyzji), a wynik na wszystkich oknach raportuję obok (ustalam to przed liczeniem wyników modeli).
3. Wpływ wyprzedaży na sprzedaż mierzę poniżej metodą różnicy różnic (modele wyprzedawane vs `BASIC` w tych samych tygodniach).

## 4.9. Wpływ okien wyprzedaży na sprzedaż (różnica różnic)

**Po co:** żeby wiedzieć, czy trzeba oczyszczać dane z tych tygodni przed liczeniem średnich (dla B1 i cech opóźnionych) i czy rabat powoduje wzrost sprzedaży.

**Metoda:** dla każdego modelu z oknem wyprzedaży: średnia sprzedaż w 3 tygodniach okna ÷ średnia z 4 tygodni **przed** oknem (stosunek treatment). To samo dla wszystkich modeli `BASIC` łącznie (stosunek kontrolny, zakładam, że ich sprzedaż nie jest zmieniona rabatem, bo rabatów tam praktycznie nie ma). Iloraz stosunków ≈ efekt okna po uwzględnieniu wspólnych wahań tygodniowych. **Ograniczenie:** kontrola `BASIC` nie ma takiej samej sezonowości jak kategorie wyprzedawane (np. kurtki w styczniu), więc to szacunek orientacyjny, nie dokładny efekt przyczynowy; nie ma też losowego przydziału rabatów.

In [ ]:
windows = {"SS25_lip2025": ("2025-07-14", "2025-07-28"), "AW25_sty2026": ("2026-01-12", "2026-01-26"), "SS26_lip2026": ("2026-07-13", "2026-07-27")}
basic_week = ws[ws.collection == "BASIC"].groupby("week_start").qty.sum()
rows = []
for name, (a, b) in windows.items():
    a, b = pd.Timestamp(a), pd.Timestamp(b)
    in_w = pd.date_range(a, b, freq="W-MON"); pre_w = pd.date_range(a - pd.Timedelta(weeks=4), a - pd.Timedelta(weeks=1), freq="W-MON")
    ctrl = basic_week.reindex(in_w).mean() / basic_week.reindex(pre_w).mean()
    treated = ws[(ws.week_start.isin(in_w)) & (ws.disc_rate > 0.05)].style_code.unique()
    for sc in treated:
        s_ = ws[ws.style_code == sc].set_index("week_start").qty
        r = s_.reindex(in_w).fillna(0).mean() / max(s_.reindex(pre_w).fillna(0).mean(), 1e-9)
        d_ = ws[(ws.style_code == sc) & ws.week_start.isin(in_w)]
        rows.append((name, sc, style_info.set_index("StyleCode").Category[sc], d_.discount.sum() / d_.list_value.sum(), r, ctrl, r / ctrl))
did = pd.DataFrame(rows, columns=["okno", "model", "kategoria", "stopa_rabatu_okna", "stosunek_model", "stosunek_BASIC", "iloraz"])
display(did.round(2))
print("Mediana ilorazu (efekt okna względem BASIC):", round(did.iloraz.median(), 2), "| zakres:", round(did.iloraz.min(), 2), "-", round(did.iloraz.max(), 2))

**Co widzę:** efekt okna wyprzedaży względem `BASIC` jest **mały i niejednoznaczny**: mediana ilorazu 1,03 (zakres 0,89–1,32 przy rabatach 26–40%). Pojedyncze modele wyraźnie zyskują (`A-BAG-002`, `A-BAG-003`: ok. +30%; `W-KNT-002`: +15%), inne nie reagują lub tracą (`W-DRS-004`: −11%). Dla 14 modeli, bez losowego przydziału i z różną sezonowością kategorii, nie umiem wskazać jednego efektu.

**Wniosek / decyzja:** (1) w danych **nie ma mocnego dowodu** na silny wzrost sprzedaży pod wpływem wyprzedaży końca sezonu (ale to modele schodzące z oferty, które i tak spadałyby; prawdziwy efekt może być zamaskowany); (2) dla prognozy W39–W42, w której rabatów nie ma, ma to znaczenie drugorzędne; (3) tygodnie wyprzedaży **zostawiam w danych**, ale oznaczam flagą i raportuję wyniki z nimi i bez nich; osobnego oczyszczania nie robię, bo efekt jest niewielki względem szumu. **NIEPEWNOŚĆ:** efektu Black Friday (jeden tydzień 2025 dla 20 modeli) nie umiem tu zmierzyć, a backtest go nie obejmuje.

## 4.10. Kalendarz: święta stałe i ruchome, dni otwarcia, wydarzenia sezonowe

**Po co:** tydzień ze świętem lub niedzielą handlową ma inną liczbę dni otwarcia sklepów, a Wielkanoc i Boże Ciało przesuwają się między tygodniami ISO. To właśnie ten mechanizm psuje porównanie z rokiem poprzednim po numerze tygodnia ISO (hipoteza H3). Buduję **własną tabelę kalendarza** (hurtownia jej nie ma): święta państwowe z biblioteki `holidays` (dane zewnętrzne, Polska), niedziele handlowe z przepisów (sprawdzone wyżej na danych), Black Friday jako piątek po czwartym czwartku listopada.

In [ ]:
from dateutil.easter import easter as _easter

def build_calendar(start="2024-12-30", end="2026-12-27"):
    days = pd.date_range(start, end)
    hol = holidays.country_holidays("PL", years=sorted({d.year for d in days}), language="pl")
    cal = pd.DataFrame({"date": days})
    cal["holiday"] = cal.date.map(lambda d: hol.get(d.date()))
    cal["is_holiday"] = cal.holiday.notna()
    ts = set().union(*[trading_sundays(y) for y in sorted({d.year for d in days})])
    cal["is_sunday"] = cal.date.dt.day_of_week == 6
    cal["trading_sunday"] = cal.date.isin(ts)
    # sklep stacjonarny otwarty: nie święto i (nie niedziela albo niedziela handlowa); święto w niedzielę handlową nie występuje w danych
    cal["store_open"] = ~cal.is_holiday & (~cal.is_sunday | cal.trading_sunday)
    iso_ = cal.date.dt.isocalendar()
    cal["iso_year"], cal["iso_week"] = iso_.year.astype(int), iso_.week.astype(int)
    cal["week_start"] = cal.date - pd.to_timedelta(cal.date.dt.day_of_week, unit="D")
    return cal

cal = build_calendar()

def special_days(year):
    nov1 = pd.Timestamp(year, 11, 1)
    thursdays = [nov1 + pd.Timedelta(days=i) for i in range(30) if (nov1 + pd.Timedelta(days=i)).day_of_week == 3]
    return {"easter": pd.Timestamp(_easter(year)), "corpus": pd.Timestamp(_easter(year)) + pd.Timedelta(days=60),
            "black_friday": thursdays[3] + pd.Timedelta(days=1), "xmas_eve": pd.Timestamp(year, 12, 24)}

def week_calendar(cal):
    g = cal.groupby("week_start")
    wc = pd.DataFrame({"store_open_days": g.store_open.sum(), "holidays_monsat": g.apply(lambda d: int((d.is_holiday & ~d.is_sunday).sum()), include_groups=False),
                       "trading_sunday": g.trading_sunday.max().astype(int)})
    sd = {y: special_days(y) for y in (2024, 2025, 2026)}
    for name in ("easter", "corpus", "black_friday", "xmas_eve"):
        wk_of = {(d - pd.Timedelta(days=d.day_of_week)) for y, v in sd.items() for k, d in v.items() if k == name}
        wc[f"wk_{name}"] = wc.index.isin(wk_of).astype(int)
    # tydzień przed Wielkanocą i tydzień po (zakupy świąteczne)
    easter_wks = [d - pd.Timedelta(days=d.day_of_week) for d in (v["easter"] for v in sd.values())]
    wc["wk_before_easter"] = wc.index.isin([w - pd.Timedelta(weeks=1) for w in easter_wks]).astype(int)
    return wc

wcal = week_calendar(cal)
# Weryfikacja: dni otwarcia z reguł == dni ze sprzedażą w danych (sklepy, wszystkie 5)
obs_open = (q.drop(columns="S-ONL-01") > 0).any(axis=1)
chk = cal.set_index("date").loc[all_days, "store_open"]
print("Zgodność dni otwarcia sklepów (reguły vs dane):", bool((chk.values == obs_open.values).all()))
print("Tygodnie ze świętem pon–sob lub niedzielą handlową, 2025–2026 (fragment):")
display(wcal[(wcal.holidays_monsat > 0) | (wcal.trading_sunday > 0)].loc["2025-01-01":"2026-09-14"].head(10))
print("Wydarzenia: Wielkanoc", special_days(2025)["easter"].date(), special_days(2026)["easter"].date(),
      "| Boże Ciało", special_days(2025)["corpus"].date(), special_days(2026)["corpus"].date(),
      "| Black Friday", special_days(2025)["black_friday"].date(), special_days(2026)["black_friday"].date())

**Co widzę:** reguły otwarcia (święta państwowe + niedziele handlowe) odtwarzają **dokładnie** dni, w których sklepy faktycznie sprzedawały (zgodność = `True` dla wszystkich dni 2025-01-01…2026-09-20), więc kalendarz mogę stosować także na przyszłość (W39–W42 2026: sprawdzam to w sekcji Transform). Wielkanoc: 2025-04-20 i 2026-04-05, Boże Ciało: 2025-06-19 i 2026-06-04, Black Friday: 2025-11-28 i 2026-11-27. Z tabeli widać, że tygodnie mają 4–7 dni otwarcia sklepów (np. tydzień od 2025-04-28 tylko 4, bo 1 i 3 maja to święta).

**Wniosek:** liczba dni otwarcia (`store_open_days`) jest czytelną, prostą cechą kalendarza. Sprawdzam teraz, czy rzeczywiście tłumaczy odchylenia wzrostu rok do roku w poszczególnych tygodniach (tu H3 w wersji roboczej: czy po wyrównaniu po numerze tygodnia ISO święta ruchome wprowadzają błąd).

In [ ]:
# Stosunek sprzedaży BASIC (2026 / 2025) w tych samych tygodniach ISO: osobno sklepy i online
pbk = pn[pn.style_code.isin(basic) & pn.IsoWeek.between(2, 37) & pn.IsoYear.isin([2025, 2026])].copy()
pbk["chan"] = np.where(pbk.store == "S-ONL-01", "ONLINE", "STORE")
r = pbk.pivot_table(index="IsoWeek", columns=["chan", "IsoYear"], values="qty", aggfunc="sum")
ratio = pd.DataFrame({c: r[(c, 2026)] / r[(c, 2025)] for c in ("STORE", "ONLINE")})
od = wcal.copy(); od["iso_week"] = od.index.isocalendar().week.values; od["iso_year"] = od.index.isocalendar().year.values
od25 = od[od.iso_year == 2025].set_index("iso_week").store_open_days; od26 = od[od.iso_year == 2026].set_index("iso_week").store_open_days
ratio["open_ratio"] = (od26 / od25).reindex(ratio.index)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ratio.index, ratio.STORE, "o-", label="sklepy stacjonarne (2026/2025)")
ax.plot(ratio.index, ratio.ONLINE, "s-", label="online (2026/2025)")
ax.plot(ratio.index, ratio.open_ratio, "k--", lw=1, label="dni otwarcia sklepów 2026 / 2025")
for y, c_ in ((2025, "tab:green"), (2026, "tab:red")):
    for k, lab in (("easter", "Wielkanoc"), ("corpus", "Boże Ciało")):
        d = special_days(y)[k]; wnum = d.isocalendar().week
        ax.axvline(wnum, color=c_, alpha=.4, ls=":"); ax.text(wnum, 0.6, f"{lab} {y}", rotation=90, fontsize=7, color=c_)
ax.axhline(1.092, color="grey", lw=.8); ax.set_ylim(0.5, 1.8)
ax.set_xlabel("tydzień ISO"); ax.set_ylabel("stosunek (bez jednostki)")
ax.set_title("BASIC: sprzedaż tygodniowa 2026 vs 2025 w tym samym numerze tygodnia ISO (szara linia = średnie +9,2%)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

lr = np.log(ratio.dropna())
print("Korelacja log(YoY sklepów) z log(stosunek dni otwarcia):", round(lr.STORE.corr(lr.open_ratio), 2),
      "| online:", round(lr.ONLINE.corr(lr.open_ratio), 2))
print("Odchylenie standardowe log-YoY: sklepy", round(lr.STORE.std(), 3), "| online", round(lr.ONLINE.std(), 3))

**Co widzę:** (1) W sklepach stacjonarnych tygodniowy stosunek sprzedaży 2026/2025 (model `BASIC`) wyraźnie podąża za stosunkiem liczby dni otwarcia (korelacja logarytmów 0,81). W kanale online prawie wcale (0,05), bo online działa codziennie. Rozrzut log-stosunku jest większy w sklepach (odchylenie standardowe 0,115) niż online (0,081). (2) Największe odchylenia od średniej +9% wypadają w tygodniach ze świętami ruchomymi lub majówką: W15 (poniedziałek wielkanocny 2026 vs zwykły tydzień 2025) ok. 0,79, W18 (w 2025 tydzień z 4 dniami otwarcia sklepów, w 2026 z 5) ok. 1,56, W23 (Boże Ciało 2026) ok. 0,91 i W25 (Boże Ciało było w 2025) ok. 1,35. (3) Dni otwarcia nie tłumaczą wszystkiego: w W18 stosunek 1,56 jest wyższy niż 1,25 wynikające z samych dni otwarcia, więc zostaje szum losowy lub efekt zakupów okołoświątecznych.

**Wniosek / decyzja:** wyrównanie „ten sam numer tygodnia ISO" bez korekty kalendarza wnosi do prognozy rok do roku błąd rzędu kilkudziesięciu procent w pojedynczych tygodniach sklepów. To przemawia za hipotezą H3 (do formalnego sprawdzenia w sekcji 6). Z tego powodu: (a) liczba dni otwarcia (`store_open_days`) i znaczniki świąt ruchomych wejdą do cech modeli, (b) dla baseline'u rok do roku zbuduję i porównam wyrównanie po numerze tygodnia ISO oraz wyrównanie po wydarzeniu (z korektą dni otwarcia).

## 4.11. Audyt: stany magazynowe a sprzedaż (okres sierpień–wrzesień 2026)

**Po co:** stany są jedynym miejscem w danych, gdzie widać brak towaru wprost. Sprawdzam, czy migawki zachowują się spójnie ze sprzedażą (stan spada o sprzedane sztuki) oraz czy któreś pozycje mają zerowy stan. Jeśli tak, to zerowa sprzedaż w tych miejscach nie oznacza zerowego popytu, i nie wolno tego traktować jak popytu przy uczeniu ani przy ocenie. Pamiętam, że migawka nie jest sprzedażą: wzrost stanu to dostawa lub przesunięcie (o których nie ma osobnych danych).

In [ ]:
inv_rel = con.execute("""
WITH s AS (SELECT DateKey, ProductKey, StoreKey, SUM(Quantity) AS q FROM fact_sales GROUP BY DateKey, ProductKey, StoreKey),
j AS (
  SELECT i.DateKey, i.ProductKey, i.StoreKey, i.StockQuantity AS stock,
         LAG(i.StockQuantity) OVER (PARTITION BY i.ProductKey, i.StoreKey ORDER BY i.DateKey) AS prev,
         COALESCE(s.q, 0) AS q
  FROM fact_inventory_daily i
  LEFT JOIN s ON s.DateKey = i.DateKey AND s.ProductKey = i.ProductKey AND s.StoreKey = i.StoreKey)
SELECT CASE WHEN prev IS NULL THEN '0 pierwszy dzień migawki (brak poprzedniego stanu)'
            WHEN stock - prev = -q AND q > 0 THEN '1 stan spadł dokładnie o sprzedaż dnia'
            WHEN stock = prev AND q = 0 THEN '2 bez zmiany stanu i bez sprzedaży'
            WHEN stock > prev THEN '3 stan wzrósł (dostawa lub przesunięcie?)'
            ELSE '4 inne: spadek stanu bez pokrycia w sprzedaży' END AS zdarzenie,
       COUNT(*) AS wiersze, SUM(q) AS sprzedane_szt
FROM j GROUP BY 1 ORDER BY 1
""").df()
display(inv_rel)
print("Okres stanów:", fact_inv.DateKey.min(), "…", fact_inv.DateKey.max(), "| SKU ze stanami:", fact_inv.ProductKey.nunique(),
      "| lokalizacje:", fact_inv.StoreKey.nunique(), "| modele (StyleCode) ze stanami:",
      dim_product[dim_product.ProductKey.isin(fact_inv.ProductKey)].StyleCode.nunique(), "z", dim_product.StyleCode.nunique())

zs = con.execute("""
    SELECT st.StoreCode AS lokalizacja, p.StyleCode AS model, COUNT(DISTINCT i.ProductKey) AS sku_z_zerem,
           MIN(d."Date") AS od_dnia, MAX(d."Date") AS do_dnia, COUNT(DISTINCT i.DateKey) AS dni
    FROM fact_inventory_daily i
    JOIN dim_product p ON p.ProductKey = i.ProductKey
    JOIN dim_store st ON st.StoreKey = i.StoreKey
    JOIN dim_date d ON d.DateKey = i.DateKey
    WHERE i.StockQuantity = 0
    GROUP BY st.StoreCode, p.StyleCode ORDER BY 1, 2""").df()
zs["sku_modelu_razem"] = zs.model.map(style_info.set_index("StyleCode").n_sku)
print("\nPary model x lokalizacja, w których jakikolwiek SKU miał stan 0 na koniec dnia:")
display(zs)

# Sprzedaż tygodniowa tych par na tle średniej z 4 pozostałych sklepów stacjonarnych (ten sam model)
skey = dim_store.set_index("StoreCode").StoreKey
rows = {}
for m_, l_ in zip(zs.model, zs.lokalizacja):
    sub = pn[(pn.style_code == m_) & (pn.week_start >= "2026-06-08")]
    rows[f"{m_} @ {l_}"] = sub[sub.store_key == skey[l_]].set_index("week_start").qty
    rows[f"   (śr. 4 inne sklepy: {m_})"] = sub[~sub.store_key.isin([skey[l_], 6])].groupby("week_start").qty.sum() / 4
tz = pd.DataFrame(rows).T
tz.columns = [c.strftime("%m-%d") for c in tz.columns]
print("\nSprzedaż tygodniowa (szt.), tygodnie od poniedziałku:")
display(tz.round(0).astype(int))

**Co widzę:** (1) Migawki stanów są spójne ze sprzedażą: w 22 885 wierszach (z 59 976) stan spadł dokładnie o sprzedane sztuki dnia, w 35 709 wierszach nie zmienił się przy braku sprzedaży, 166 razy wzrósł (dostawa lub przesunięcie), a 40 wierszy ma spadek stanu bez pokrycia w sprzedaży (ubytek lub korekta, margines). (2) Stany obejmują 22 z 36 modeli; brakuje ich dla 14 modeli starszych kolekcji (ze znacznikiem `IsActive = 0`). (3) Cztery pary model × lokalizacja mają **zerowy stan wszystkich SKU** od 2026-09-07 do końca danych (14 dni): `M-SHI-001`, `M-TSH-001` i `W-DRS-002` w Gdańsku oraz `W-JNS-004` w Warszawie Arkadii. Sprzedaż tych par w tygodniach 09-07 i 09-14 wynosi 0, a w poprzednich 13 tygodniach była normalna i duża (np. `M-TSH-001` w Gdańsku 116–283 szt. tygodniowo).

**Wniosek / decyzja:** (1) w czterech seriach dwa ostatnie tygodnie to **zera z braku towaru, nie z braku popytu**. To wprost dotyczy zadania: dzisiejsza praktyka (średnia z 4 tygodni) policzy dla tych par połowę rzeczywistego popytu, bo dwa z czterech tygodni to sztuczne zera. Skalę tego błędu zmierzę w sekcji 6. (2) Stany są tylko od sierpnia 2026, więc nie rozstrzygną wcześniejszych braków. Potrzebuję reguły, która rozpoznaje brak towaru ze samej sprzedaży (spadek serii przy normalnej sprzedaży tego samego modelu w pozostałych lokalizacjach), i której poprawność mogę sprawdzić na tych czterech znanych przypadkach (sekcja 5). (3) Dla 14 modeli bez stanów (wygaszane) takiej weryfikacji nie będzie.

**NIEPEWNOŚĆ:** stan 0 oznacza brak towaru na koniec dnia; nie wiem, czy firma planuje dostawę. Czy dostawa dojdzie w W39–W42, to pytanie do zleceniodawcy (sekcja 8).

## 4.12. Przebudowa sklepu Kraków Rynek: szereg przed, w trakcie i po

**Po co:** w opisie danych jest informacja, że Kraków Rynek miał przebudowę (data w wymiarze: 2026-04-01), ale bez opisu przebiegu prac. Nie wiem, czy sklep był zamknięty, czy sprzedawał z ograniczeniem, ani czy po otwarciu sprzedaż wróciła do dawnego poziomu, czy skoczyła i potem opadła. Od tego zależy, jak potraktować Kraków w uczeniu i ocenie. Porównuję Kraków do średniej czterech pozostałych sklepów stacjonarnych na modelach `BASIC` (ten sam asortyment przez cały okres, więc stosunek nie jest zaburzony premierami). Stosunek jest bezwymiarowy: 0,6 oznacza, że Kraków sprzedaje 60% tego, co przeciętny inny sklep.

In [ ]:
st_basic = (pn[pn.style_code.isin(basic)].pivot_table(index="week_start", columns="store_key", values="qty", aggfunc="sum").iloc[1:])
krk_ratio = st_basic[3] / st_basic[[1, 2, 4, 5]].mean(axis=1)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(krk_ratio.index, krk_ratio.values, "o-", ms=3, lw=1.2, label="Kraków / średnia 4 pozostałych sklepów (BASIC)")
ax.axvline(pd.Timestamp("2026-04-01"), color="k", ls=":", lw=1); ax.text(pd.Timestamp("2026-04-04"), 0.43, "data przebudowy z wymiaru sklepu", fontsize=8)
ax.set_title("Kraków Rynek: tygodniowy stosunek sprzedaży do średniej pozostałych sklepów stacjonarnych (modele BASIC)")
ax.set_ylabel("stosunek (bez jednostki)"); ax.set_xlabel("tydzień (poniedziałek)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

def per(d):
    if d < pd.Timestamp("2026-03-09"): return "1 przed (do 2026-03-02)"
    if d <= pd.Timestamp("2026-03-23"): return "2 trzy tygodnie 03-09..03-23"
    return "3 po (od 2026-03-30)"
tab = krk_ratio.groupby(krk_ratio.index.map(per)).agg(tygodni="size", srednia="mean", mediana="median", min="min", max="max")
display(tab.round(2))
print("Okres po przebudowie wg miesięcy (średni stosunek):")
post = krk_ratio[krk_ratio.index >= "2026-03-30"]
display(post.groupby(post.index.strftime("%Y-%m")).mean().round(2).to_frame("stosunek").T)

# Czy podniesienie jest równomierne we wszystkich kategoriach BASIC?
cb = pn[pn.style_code.isin(basic)].assign(per=lambda d: np.where(d.week_start < "2026-03-09", "przed", np.where(d.week_start >= "2026-03-30", "po", "w_trakcie")))
cb = cb[cb.week_start >= "2025-01-06"]
tc = cb[cb.per != "w_trakcie"].pivot_table(index=["Category", "per"], columns="store_key", values="qty", aggfunc="mean")
tc["krk_vs_inne"] = tc[3] / tc[[1, 2, 4, 5]].mean(axis=1)
display(tc["krk_vs_inne"].unstack("per")[["przed", "po"]].assign(wzrost=lambda d: d["po"] / d["przed"]).round(2))

**Co widzę:** (1) Przed przebudową Kraków sprzedawał średnio 61% tego, co przeciętny inny sklep stacjonarny (tygodniowo od 0,52 do 0,75). (2) W trzech tygodniach 03-09, 03-16 i 03-23 stosunek spadł do ok. 0,48 (0,45–0,52), czyli o ok. 20% poniżej dawnego poziomu; sklep **nie był zamknięty** (sprzedaż nie spadła do zera). (3) Od 03-30 stosunek rośnie do 0,70, w kwietniu i maju stoi na ok. 0,73–0,74, a w sierpniu i wrześniu na 0,78–0,79. Czyli sprzedaż Krakowa po przebudowie jest o ok. 20–25% wyższa względem reszty sieci niż przed (mediana 0,76 vs 0,60), i **efekt nie wygasa** w okresie, który widzę. (4) Wzrost jest podobny we wszystkich kategoriach BASIC (od +15% T-shirty do +24% Akcesoria), więc to zmiana skali sklepu, a nie jednego asortymentu. (5) W czerwcu i lipcu stosunek spada do 0,66–0,68, w tym dwa tygodnie ok. 0,51 (06-22 i 06-29). To nie wygląda na przebudowę; sprawdzę, czy to brak towaru jednego modelu (sekcja 5).

**Wniosek / decyzja:** Kraków ma trzy fazy: przed (poziom 0,61), prace (ok. 0,48, trzy tygodnie) i po (ok. 0,75). Do modeli wprowadzę znaczniki „prace" i „po przebudowie" tylko dla tego sklepu. **ZAŁOŻENIE:** firma zna datę przebudowy z góry (to zdarzenie planowane), więc znacznik na tydzień prognozowany nie jest informacją z przyszłości. Okna prac (03-09…03-23) wyznaczyłem z danych po fakcie; przebieg prac nie jest opisany. Wrażliwość wyniku na to założenie sprawdzę w diagnostyce (usunięcie Krakowa, inny okres uczenia). Z opisu firmy wiadomo, że powierzchnia sklepu wzrosła (z 180 do 320 m²), co zgadza się z kierunkiem zmiany.

## 4.13. Krzywa życia modelu sezonowego: sprzedaż w funkcji tygodni od premiery

**Po co:** nowy model nie ma historii, a dosyłki są planowane właśnie na początku jego życia. Chcę zobaczyć, jak wygląda typowy przebieg sprzedaży od pierwszego tygodnia (rozbieg, szczyt, wygasanie), czy kształt jest podobny między modelami i czy **wczesna sprzedaż przewiduje późniejszą** (to byłaby podstawa prognozy „z analogów" dla modeli z krótką historią). Wiek liczę w tygodniach od tygodnia pierwszej sprzedaży (0 = tydzień pierwszej sprzedaży, zwykle niepełny). Poziom dopasowuję do modelu średnią z tygodni 1–4 (wczesny poziom). Modele `BASIC` pomijam, bo nie mają obserwowanej premiery.

In [ ]:
sty_w = ws.pivot_table(index="week_start", columns="style_code", values="qty", aggfunc="sum").fillna(0).loc["2025-01-06":]
new_models = [s for s in sty_w.columns if s not in basic]
curves, rows = {}, []
for m_ in new_models:
    s_ = sty_w[m_]
    f_ = s_.gt(0).idxmax()
    a_ = ((s_.index - f_).days // 7)
    s_ = s_[a_ >= 0].reset_index(drop=True)             # indeks = wiek w tygodniach
    if len(s_) >= 5:
        early = s_.iloc[1:5].mean()
        curves[m_] = s_ / early
        rows.append({"model": m_, "kolekcja": mp.Collection[m_], "kategoria": cat_of[m_], "tygodni_historii": len(s_),
                     "wczesny_poziom": early, "sr_wiek_5_8": s_.iloc[5:9].mean() if len(s_) >= 9 else np.nan,
                     "sr_wiek_9_16": s_.iloc[9:17].mean() if len(s_) >= 17 else np.nan})
lc = pd.DataFrame(rows).set_index("model")
lc["stosunek_5_8_do_wczesnego"] = lc.sr_wiek_5_8 / lc.wczesny_poziom

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
colmap = {"SS25": "tab:orange", "AW25": "tab:blue", "SS26": "tab:red", "AW26": "tab:green"}
for m_, c_ in curves.items():
    ax[0].plot(c_.index[:40], c_.values[:40], lw=0.9, alpha=0.6, color=colmap[mp.Collection[m_]])
ax[0].plot([], [], color="tab:orange", label="SS25"); ax[0].plot([], [], color="tab:blue", label="AW25")
ax[0].plot([], [], color="tab:red", label="SS26"); ax[0].plot([], [], color="tab:green", label="AW26")
ax[0].set_ylim(0, 4); ax[0].set_xlabel("wiek modelu [tygodnie od pierwszej sprzedaży]"); ax[0].set_ylabel("sprzedaż / średnia z wieku 1–4 (bez jednostki)")
ax[0].set_title("Krzywa życia: znormalizowana sprzedaż modelu (wszystkie lokalizacje)"); ax[0].legend(fontsize=8, title="kolekcja")
ok = lc.dropna(subset=["sr_wiek_5_8"])
ax[1].scatter(ok.wczesny_poziom, ok.sr_wiek_5_8, c=[colmap[k] for k in ok.kolekcja], s=35)
mx = max(ok.wczesny_poziom.max(), ok.sr_wiek_5_8.max()) * 1.1
ax[1].plot([5, mx], [5, mx], "k--", lw=0.8, label="y = x")
ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].legend(fontsize=8)
ax[1].set_xlabel("średnia tygodniowa w wieku 1–4 [szt./tydz., skala log]"); ax[1].set_ylabel("średnia tygodniowa w wieku 5–8 [szt./tydz., skala log]")
ax[1].set_title("Czy wczesna sprzedaż przewiduje późniejszą? (każdy punkt = model)")
plt.tight_layout(); plt.show()

print("Modele z >= 9 tygodniami historii:", len(ok), "| korelacja log(wczesny) vs log(wiek 5-8):", round(np.corrcoef(np.log(ok.wczesny_poziom), np.log(ok.sr_wiek_5_8))[0, 1], 3))
print("Stosunek (wiek 5-8)/(wiek 1-4): mediana", round(ok.stosunek_5_8_do_wczesnego.median(), 2),
      "| rozstęp ćwiartkowy", ok.stosunek_5_8_do_wczesnego.quantile([.25, .75]).round(2).tolist(),
      "| min-max", round(ok.stosunek_5_8_do_wczesnego.min(), 2), round(ok.stosunek_5_8_do_wczesnego.max(), 2))
display(lc.drop(columns=["sr_wiek_9_16"]).round(2).sort_values("wczesny_poziom", ascending=False))

**Co widzę:** (1) Kształt krzywej życia jest podobny w różnych modelach i kolekcjach: tydzień 0 (niepełny) to ok. 0,3 wczesnego poziomu, potem wzrost przez 3–4 tygodnie, szerokie plateau ze szczytem w wieku ok. 6–16 tygodni (zwykle 1,2–2 × wczesny poziom), spadek od wieku ok. 17–25 tygodni i prawie zero po ok. 30 tygodniach. Pojedyncze piki w wieku ok. 11 tygodni (modele AW25) przypadają na Black Friday 2025. (2) **Skala** różni się między modelami ponad dziesięciokrotnie: wczesny poziom (średnia z wieku 1–4) od ok. 64 do ok. 840 szt. tygodniowo (wszystkie lokalizacje). (3) Stosunek sprzedaży w wieku 5–8 tygodni do wczesnego poziomu jest mało zróżnicowany: mediana 1,35, rozstęp ćwiartkowy 1,21–1,46, zakres 0,98–1,68 (14 modeli ze wystarczającą historią). Korelacja 0,97 na wykresie logarytmicznym jest zawyżona przez różnice skali między modelami; ważniejszy jest rozrzut samego stosunku (ok. ±20%).

**Wniosek:** model „przebojowy" i „słaby" odróżniają się skalą, i widać to już w wieku 1–4 tygodni; kształt krzywej jest podobny. To uzasadnia prognozę z analogów (M3): wczesny poziom modelu przemnożony przez typowy kształt krzywej dla kolekcji. **NIEPEWNOŚĆ:** analogów jest mało (14 modeli z ≥ 9 tygodniami historii, w kategorii Swetry tylko jeden starszy model), krzywe zawierają wpływ pór roku i Black Friday, a nie tylko wieku modelu. Ocena M3 będzie więc obarczona dużym rozrzutem.

# 5. Transform

## 5.1. Panel tygodniowy: macierze, reguła startu serii, kalendarz

**Po co:** buduję wspólną podstawę dla wszystkich metod: macierz tygodniowa sprzedaży (89 pełnych tygodni ISO od 2025-W02 do 2026-W37 × 216 serii model × lokalizacja), obok macierze „znane z góry": liczba dni otwarcia, kalendarz świąt, stopa rabatu modelu, znaczniki Krakowa. **Reguła startu serii** (z sekcji 4.4): seria istnieje od tygodnia pierwszej sprzedaży **modelu** (we wszystkich lokalizacjach naraz, bo model jest wprowadzany do całej sieci); przed tym tygodniem wartość jest „brak" (NaN), nie zero. Od startu brak sprzedaży to zero popytu, **z wyjątkiem** tygodni z podejrzeniem braku towaru (sekcja 5.2). Modele `BASIC` traktuję jako istniejące od początku danych (premiery w 2024 r., poza zakresem). Tydzień 2025-W01 pomijam (dane od środy 2025-01-01).

In [ ]:
W = pd.date_range("2025-01-06", "2026-09-14", freq="W-MON")      # 89 pełnych tygodni (poniedziałki)
WX = pd.date_range("2025-01-06", "2026-10-19", freq="W-MON")     # + 5 przyszłych tygodni (kalendarz znany z góry)
T, TX = len(W), len(WX)
styles = sorted(style_info.StyleCode)
stores = list(dim_store.StoreKey)
SER = pd.DataFrame([(s_, k_) for s_ in styles for k_ in stores], columns=["model", "store_key"])
SER["category"] = SER.model.map(cat_of)
SER["store"] = SER.store_key.map(code)
SER["basic"] = SER.model.isin(basic)
S = len(SER)

Yraw = (pn.pivot_table(index="week_start", columns=["style_code", "store_key"], values="qty", aggfunc="sum")
          .reindex(index=W, columns=pd.MultiIndex.from_frame(SER[["model", "store_key"]])).fillna(0).to_numpy(float))
style_cols = {st_: ix for st_, ix in SER.groupby("model").indices.items()}      # model -> kolumny jego 6 serii
start = {st_: (0 if st_ in basic else int(np.argmax(Yraw[:, ix].sum(1) > 0))) for st_, ix in style_cols.items()}
START = SER.model.map(start).to_numpy()                                         # indeks tygodnia startu serii
ALIVE = np.arange(T)[:, None] >= START[None, :]
Y = np.where(ALIVE, Yraw, np.nan)                                               # NaN przed startem serii

CAL = wcal.reindex(WX)
is_online = (SER.store_key == 6).to_numpy()
OD = np.where(is_online[None, :], 7.0, CAL.store_open_days.to_numpy(float)[:, None])   # dni otwarcia w tygodniu (online = 7)
DISC = (ws.pivot_table(index="week_start", columns="style_code", values="disc_rate", aggfunc="mean")
          .reindex(index=W).reindex(columns=SER.model).fillna(0).to_numpy(float))        # stopa rabatu modelu w tygodniu (znana z góry: plan cen)
KRK = (SER.store_key == 3).to_numpy()
KRK_WORKS = np.asarray(WX.isin(pd.to_datetime(["2026-03-09", "2026-03-16", "2026-03-23"])))
KRK_POST = np.asarray(WX >= pd.Timestamp("2026-03-30"))

print(f"Macierz sprzedaży Y: {Y.shape[0]} tygodni x {Y.shape[1]} serii | {W[0].date()} … {W[-1].date()} | tygodni ISO w kalendarzu: {TX} (do {WX[-1].date()})")
print(f"Komórek 'żyjących' (od startu serii): {ALIVE.mean():.1%} | udział zer wśród żyjących: {(Yraw[ALIVE] == 0).mean():.1%} | suma sztuk: {int(np.nansum(Y)):,}")
print("Starty modeli (indeks tygodnia): BASIC=0; najpóźniejszy start:", max(start.items(), key=lambda kv: kv[1]), "| liczba modeli ze startem > 0:", sum(v > 0 for v in start.values()))
prev = pd.DataFrame({"week": W[-6:].strftime("%m-%d"), "M-KNT-001 @ Arkadia": Y[-6:, SER.index[(SER.model == "M-KNT-001") & (SER.store_key == 1)][0]],
                     "W-TSH-001 @ Arkadia": Y[-6:, SER.index[(SER.model == "W-TSH-001") & (SER.store_key == 1)][0]],
                     "dni otwarcia (sklep)": OD[T - 6:T, 0], "dni otwarcia (online)": OD[T - 6:T, 5]})
display(prev)
display(SER.head(6))

## 5.2. Reguła podejrzenia braku towaru (flaga) i jej sprawdzenie na stanach

**Po co:** wiem już z sekcji 4.11, że zera mogą wynikać z braku towaru. Stany są tylko od sierpnia 2026, więc dla całej historii potrzebuję reguły opartej wyłącznie na sprzedaży. **Pomysł:** w tygodniu *w* seria (model × lokalizacja) jest „podejrzana", jeśli sprzedała **znacznie mniej, niż wynika z udziału tej lokalizacji w sprzedaży modelu** i z tego, ile ten model sprzedał w tym tygodniu w pozostałych lokalizacjach. Oczekiwanie = udział z 8 poprzednich tygodni (bez tygodni już oflagowanych) × sprzedaż pozostałych lokalizacji ÷ (1 − udział). Flaga, gdy oczekiwanie ≥ 15 szt. i sprzedaż ≤ 20% oczekiwania (przy oczekiwanych 15 sztukach zero lub ≤ 3 sztuki to zdarzenie skrajnie mało prawdopodobne przy losowej zmienności) oraz gdy model ma ≥ 6 tygodni życia (bo w rozbiegu udziały lokalizacji dopiero się stabilizują). Reguła używa tylko danych z tygodnia *w* i wcześniejszych, więc można ją stosować w czasie rzeczywistym. Progi (15 szt., 20%) wybrałem z góry z rozumowania o szumie, a nie z dopasowania do danych; wrażliwość sprawdzam niżej.

In [ ]:
def detect_stockouts(Y, min_exp=15.0, frac=0.2, win=8, min_hist=4, min_age=6):
    """Flaga podejrzenia braku towaru: sprzedaż serii << oczekiwana z udziału lokalizacji w modelu."""
    Tn, Sn = Y.shape
    F = np.zeros((Tn, Sn), bool)
    E = np.full((Tn, Sn), np.nan)
    for st_, ix in style_cols.items():
        sub = np.nan_to_num(Y[:, ix])
        tot = sub.sum(1)
        t0 = start[st_]
        for j, c in enumerate(ix):
            hist = []                                              # tygodnie bez flagi: (sprzedaż serii, sprzedaż modelu)
            for t in range(t0, Tn):
                if len(hist) >= min_hist and (t - t0) >= min_age:
                    own = sum(h[0] for h in hist[-win:])
                    tw = sum(h[1] for h in hist[-win:])
                    if 0 < own < tw:
                        sh = own / tw
                        e = sh * (tot[t] - sub[t, j]) / (1 - sh)
                        E[t, c] = e
                        if e >= min_exp and sub[t, j] <= frac * e:
                            F[t, c] = True
                            continue                               # oflagowany tydzień nie wchodzi do udziału
                hist.append((sub[t, j], tot[t]))
    return F, E

F, E = detect_stockouts(Y)
YI = Y.copy()
YI[F] = E[F]                                                       # sprzedaż "oczyszczona": w tygodniach z flagą zastąpiona oczekiwaniem
print("Oflagowane tygodnie serii:", int(F.sum()), "z", int(ALIVE.sum()), "żyjących komórek")

# Przebiegi flag: kolejne tygodnie jednej serii
runs_f = []
for c in range(S):
    t = 0
    while t < T:
        if F[t, c]:
            u = t
            while u + 1 < T and F[u + 1, c]:
                u += 1
            runs_f.append((SER.model[c], SER.store[c], W[t].date(), W[u].date(), u - t + 1, round(float(np.nanmean(E[t:u + 1, c])), 1), int(Y[t:u + 1, c].sum())))
            t = u + 1
        else:
            t += 1
runs_f = pd.DataFrame(runs_f, columns=["model", "lokalizacja", "od_tygodnia", "do_tygodnia", "tygodni", "oczekiwane_szt_tyg", "sprzedane_szt_razem"])
display(runs_f.sort_values("od_tygodnia"))

# Sprawdzenie na stanach: tygodnie od 2026-08-03 (pełne tygodnie ze stanem na niedzielę)
inv_w = con.execute("""
SELECT p.StyleCode AS model, i.StoreKey AS store_key, DATE_TRUNC('week', d."Date")::DATE AS week_start,
       SUM(i.StockQuantity) AS stock_end, SUM(CASE WHEN i.StockQuantity = 0 THEN 1 ELSE 0 END) AS skus_zero, COUNT(*) AS skus
FROM fact_inventory_daily i JOIN dim_product p ON p.ProductKey = i.ProductKey JOIN dim_date d ON d.DateKey = i.DateKey
WHERE d.DayOfWeek = 7 GROUP BY 1, 2, 3""").df()
inv_w["week_start"] = pd.to_datetime(inv_w.week_start)
inv_w = inv_w[inv_w.week_start >= "2026-08-03"]
ser_ix = {(a, b): i for i, (a, b) in enumerate(zip(SER.model, SER.store_key))}
tw_ix = {w: i for i, w in enumerate(W)}
inv_w["flag"] = [bool(F[tw_ix[w], ser_ix[(a, b)]]) for w, a, b in zip(inv_w.week_start, inv_w.model, inv_w.store_key)]
inv_w["stock_zero"] = inv_w.stock_end == 0
print(f"\nPorównanie flagi z zerowym stanem na koniec tygodnia ({inv_w.week_start.nunique()} tygodni x {inv_w.groupby(['model', 'store_key']).ngroups} par z danymi o stanach):")
display(pd.crosstab(inv_w.flag, inv_w.stock_zero, rownames=["flaga z reguły"], colnames=["stan modelu = 0"]))
print("Oflagowane tygodnie BEZ zerowego stanu (stan łączny modelu w lokalizacji i liczba SKU ze stanem 0):")
display(inv_w[inv_w.flag & ~inv_w.stock_zero][["model", "store_key", "week_start", "stock_end", "skus_zero", "skus"]])

# Wrażliwość reguły na progi
rows = []
for me in (10.0, 15.0, 25.0):
    for fr in (0.1, 0.2, 0.3):
        Fx, _ = detect_stockouts(Y, min_exp=me, frac=fr)
        rows.append({"min_oczekiwane": me, "maks_ułamek": fr, "oflagowane_tygodnie": int(Fx.sum()),
                     "w_sierp_wrz_2026": int(Fx[W >= pd.Timestamp("2026-08-03")].sum()), "przed_2026_08": int(Fx[W < pd.Timestamp("2026-08-03")].sum())})
display(pd.DataFrame(rows))

**Co widzę:** reguła oflagowała **21 tygodni serii** z 11 118 „żyjących" komórek (0,19%). Na tygodniach, dla których są stany (7 tygodni × 132 pary model × lokalizacja), flaga i stan zerowy zgadzają się tak: 8 trafień (cztery znane pary z sekcji 4.11 × 2 tygodnie: 09-07 i 09-14), 914 poprawnych „braku flagi" i **2 fałszywe alarmy** (`W-TSH-001` w Poznaniu, tygodnie 08-10 i 08-17: sprzedaż 7 i 5 szt. przy oczekiwanych ok. 110, a stan łączny modelu to 574–579 szt. i ani jedno SKU nie ma zera). Czyli na dostępnym wycinku pełna zgodność dla przypadków z brakiem towaru (8 z 8), ale precyzja 8 z 10 oflagowanych tygodni. Pozostałe 11 oflagowanych tygodni (2025 r. i pierwsza połowa 2026 r.) leży poza okresem stanów i **nie da się ich zweryfikować**; kilka z nich wygląda tak samo (np. `A-BAG-001` w Arkadii, 2025-11: sprzedaż 2 szt. wobec ok. 65 oczekiwanych). Wrażliwość na progi: w sierpniu–wrześniu 2026 we wszystkich 9 wariantach progów flaga wskazuje te same 10 tygodni, a liczba flag przed sierpniem 2026 waha się od 8 do 21.

**Wniosek / decyzja:** (1) używam progów ustalonych z góry (≥ 15 szt. oczekiwanych, ≤ 20% oczekiwania); nie dostrajam ich pod wynik. (2) Tygodnie z flagą **wyłączam z celu uczenia i z oceny** (raportuję je osobno), a w cechach opóźnionych modeli zastępuję je oczekiwaniem (kolumna „oczyszczona" `YI`), żeby brak towaru nie obniżał poziomu serii. Baseline B1 (dzisiejsza praktyka) liczę z danych surowych, bo tak działa praktyka, i dodatkowo z danych oczyszczonych, żeby zmierzyć, ile kosztują sztuczne zera. (3) **NIEPEWNOŚĆ:** reguła nie wykrywa braku towaru, który dotyka model we wszystkich lokalizacjach naraz, ani wolnego zaniku popytu; dla 14 modeli bez stanów nie sprawdzę jej wcale. Dwa fałszywe alarmy pokazują, że flaga oznacza „nietypowo niska sprzedaż", a nie „na pewno brak towaru".

## 5.3. Cechy dostępne w chwili origin (bez informacji z przyszłości)

**Po co:** wszystkie metody uczone (M1, M2, M3) potrzebują tej samej, kontrolowanej podstawy: tabeli „stan wiedzy w poniedziałek `t`" → „sprzedaż w tygodniu `t+h-1`". Zasada: każda cecha opisująca sprzedaż jest liczona wyłącznie z tygodni **przed** `t`. Z przyszłości wolno użyć tylko rzeczy znanych z góry: kalendarza (dni otwarcia, święta, niedziele handlowe), planowanego rabatu modelu i znacznika przebudowy Krakowa. **ZAŁOŻENIE:** w backteście rabat docelowego tygodnia traktuję jako znany z planu cen (zleceniodawca potwierdza brak akcji w W39–W42, więc dla prognozy właściwej wynosi 0). Cechy podzielone na grupy, żeby później zmierzyć wkład każdej (ablacja).

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)          # nanmean pustych okien jest tu zamierzony (NaN = brak historii)
CALX = wcal.reindex(WX)
ISOW = WX.isocalendar().week.to_numpy().astype(int)
MOVING = ((CALX.wk_easter + CALX.wk_before_easter + CALX.wk_corpus + CALX.wk_easter.shift(1).fillna(0)) > 0).to_numpy()   # tygodnie wokół świąt ruchomych (Wielkanoc: przed, w, po; Boże Ciało)
LAUNCH = pd.to_datetime(merch_plan.set_index("StyleCode").LaunchDate)
LAUNCH_D = pd.to_datetime(SER.model.map(LAUNCH))
cat_codes = {c_: i for i, c_ in enumerate(sorted(SER.category.unique()))}
cat_idx = SER.category.map(cat_codes).to_numpy()
coll_codes = {c_: i for i, c_ in enumerate(["BASIC", "SS25", "AW25", "SS26", "AW26"])}
coll_idx = SER.model.map(merch_plan.set_index("StyleCode").Collection).map(coll_codes).to_numpy()
sto_idx = SER.store_key.to_numpy() - 1
START_A = START.astype(int)

def aggregates(YIx):
    """Sprzedaż modelu (wszystkie lokalizacje) i kategorii w tygodniu, przypisana do kolumn serii."""
    MT = np.full_like(YIx, np.nan)
    CT = np.zeros_like(YIx)
    for m_, ix in style_cols.items():
        MT[:, ix] = np.nansum(YIx[:, ix], axis=1, keepdims=True)
        MT[:start[m_], ix] = np.nan
    for k in range(len(cat_codes)):
        ixs = np.where(cat_idx == k)[0]
        CT[:, ixs] = np.nansum(YIx[:, ixs], axis=1, keepdims=True)
    return MT, CT

GROUPS = {
    "lag":    ["lag1", "lag2", "lag3", "lag4", "m4", "m8", "m13", "trend"],
    "ident":  ["cat", "store", "coll", "is_basic", "age_series", "age_launch"],
    "pool":   ["mdl_m4", "mdl_last", "share8", "cat_mom"],
    "cal":    ["open_days", "moving", "wk_easter", "wk_corpus", "wk_black_friday", "wk_xmas_eve", "trading_sunday", "holidays_monsat"],
    "season": ["iso_week", "lag52", "m8_ly"],
    "disc":   ["disc"],
    "krk":    ["krk_works", "krk_post"],
}

def feature_rows(t, YIx, MTx, CTx):
    """Cechy w chwili origin t (poniedziałek tygodnia t): sprzedaż tylko z tygodni < t; cechy kalendarza tygodnia docelowego znane z góry."""
    cs = np.where(START_A <= t - 1)[0]                       # serie z co najmniej jednym tygodniem historii
    win = lambda A_, n: A_[max(0, t - n):t][:, cs]
    f = {f"lag{k}": YIx[t - k, cs] for k in (1, 2, 3, 4)}
    f["m4"], f["m8"], f["m13"] = (np.nanmean(win(YIx, n), 0) for n in (4, 8, 13))
    f["trend"] = (f["m4"] + 0.5) / (f["m8"] + 0.5)
    f["cat"], f["store"], f["coll"] = cat_idx[cs], sto_idx[cs], coll_idx[cs]
    f["is_basic"] = SER.basic.to_numpy()[cs].astype(int)
    f["age_series"] = t - START_A[cs]
    f["age_launch"] = ((WX[t] - LAUNCH_D.iloc[cs]).dt.days / 7).to_numpy()
    f["mdl_m4"] = np.nanmean(win(MTx, 4), 0)
    f["mdl_last"] = MTx[t - 1, cs]
    f["share8"] = np.nansum(win(YIx, 8), 0) / (np.nansum(win(MTx, 8), 0) + 1e-9)
    f["cat_mom"] = (np.nansum(win(CTx, 4), 0) + 1) / (np.nansum(CTx[max(0, t - 8):max(0, t - 4)][:, cs], 0) + 1)
    out = []
    for h in (1, 2, 3, 4):
        tw = t + h - 1
        d = {k: v for k, v in f.items()}
        d.update(t=t, h=h, tw=tw, c=cs)
        d["open_days"] = OD[tw, cs]
        d["moving"] = float(MOVING[tw])
        for k in ("wk_easter", "wk_corpus", "wk_black_friday", "wk_xmas_eve", "trading_sunday", "holidays_monsat"):
            d[k] = float(CALX[k].iloc[tw])
        d["iso_week"] = ISOW[tw]
        d["lag52"] = YIx[tw - 52, cs] if tw - 52 >= 0 else np.full(len(cs), np.nan)
        d["m8_ly"] = np.nanmean(YIx[t - 60:t - 52][:, cs], 0) if t - 60 >= 0 else np.full(len(cs), np.nan)
        d["disc"] = DISC[tw, cs] if tw < T else np.zeros(len(cs))      # w prognozie właściwej: brak akcji
        d["krk_works"] = (KRK_WORKS[tw] & KRK[cs]).astype(float)
        d["krk_post"] = (KRK_POST[tw] & KRK[cs]).astype(float)
        d["y"] = Y[tw, cs] if tw < T else np.full(len(cs), np.nan)
        d["flag_tw"] = F[tw, cs] if tw < T else np.zeros(len(cs), bool)
        out.append(pd.DataFrame(d))
    return pd.concat(out, ignore_index=True)

MT, CT = aggregates(YI)
ROWS = pd.concat([feature_rows(t, YI, MT, CT) for t in range(8, T + 1)], ignore_index=True)
print("Wiersze cech (origin x seria x horyzont):", f"{len(ROWS):,}", "| origin od", W[8].date(), "do", WX[T].date())
print("Z czego z celem znanym:", int(ROWS.y.notna().sum()), "| wiersze prognozy właściwej (origin 2026-09-21):", int((ROWS.t == T).sum()))
print("Liczba cech:", sum(len(v) for v in GROUPS.values()) + 1, "(+ horyzont h)")
display(ROWS[ROWS.t == T].head(3).T.head(34))

# 6. Analyze

## 6.1. Hipotezy i plan walidacji (zapisane PRZED budową jakiegokolwiek modelu i przed policzeniem baseline'ów)

**Po co:** żeby wynik nie dopasowywał się do progów. Poniżej treść hipotez, oczekiwany kierunek, miara i próg decyzji. Hipotezy H1–H4 wynikają z pytań zleceniodawcy i z typowych problemów prognozowania popytu w handlu (doprecyzowałem je operacyjnie), H5 dotyczy kryterium sukcesu zgłoszonego wewnątrz zespołu. Dla każdej wskazuję też, co o danych już wiem z eksploracji (sekcja 4), żeby było jasne, które z nich nie są „czystym" testem na nowych danych. Po testach każda dostanie status: potwierdzona, odrzucona albo niejednoznaczna, z liczbą i przedziałem.

**Konwencje wspólne:** 4-tygodniowa suma prognozy na poziomie model × lokalizacja; główna ocena na „top 20" (20 modeli o największej sprzedaży w 8 tygodniach przed origin) × 6 lokalizacji; wyłączone serie z flagą braku towaru w którymkolwiek z 4 tygodni docelowych (raportowane osobno) i serie bez co najmniej jednego tygodnia historii; miary: WAPE i bias (definicje w sekcji 1.4). Niepewność: bootstrap po **punktach origin** (losowanie originów ze zwracaniem, 2000 powtórzeń, przedział 95%), a dla różnic metod wersja sparowana (ta sama próbka originów dla obu metod). Poziom istotności nie ma tu zastosowania w sensie testu p; decyzje wynikają z progów i z tego, czy przedział różnicy wyklucza 0.

| # | Hipoteza | Miara | Próg decyzji |
|---|---|---|---|
| H1 | Dzisiejsza praktyka (B1, średnia z 4 tyg.) jest obciążona systematycznie przy zmianie sezonu: zaniża przy wzroście, zawyża przy spadku. | bias B1 w komórkach (origin × kategoria), w których sprzedaż kategorii w kolejnych 4 tyg. była ≥ 15% wyższa („wzrost") lub ≥ 15% niższa („spadek") niż w ostatnich 4 tyg. | potwierdzona, gdy bias ≤ −10% we „wzroście" (górna granica przedziału 95% < 0) **i** ≥ +10% w „spadku" (dolna granica > 0); częściowo, gdy spełniony jest jeden warunek. **Zastrzeżenie:** klasyfikacja komórek używa tego, co się potem wydarzyło, więc znak biasu jest po części mechaniczną konsekwencją (zawsze, gdy sprzedaż rośnie, średnia z przeszłości jest niższa). Informacją jest **wielkość** obciążenia i to, w których kategoriach i okresach jest największe. |
| H2 | Model globalny (M2) ma niższy WAPE sumy 4 tyg. niż modele per seria (M1), bez pogorszenia obciążenia. | różnica WAPE(M1) − WAPE(M2), top 20 × lokalizacja, **originy testowe**; sparowany bootstrap po originach | potwierdzona, gdy różnica ≥ 3 p.p., dolna granica przedziału > 0 i \|bias M2\| ≤ \|bias M1\| + 3 p.p.; odrzucona, gdy górna granica < 0; w pozostałych przypadkach niejednoznaczna |
| H3 | Wyrównanie z rokiem poprzednim po numerze tygodnia ISO (B2-ISO) jest gorsze od wyrównania uwzględniającego kalendarz (B2-kal: dni otwarcia, przesunięcie wokół Wielkanocy i Bożego Ciała) w tygodniach wokół świąt ruchomych. | różnica WAPE(B2-ISO) − WAPE(B2-kal) dla pojedynczych tygodni docelowych (wszystkie serie z ≥ 1 rokiem historii) w tygodniach wokół świąt ruchomych (Wielkanoc: tydzień przed, w, po; Boże Ciało; także tygodnie, w których te święta wypadły rok wcześniej), osobno dla pozostałych tygodni; bootstrap po tygodniach docelowych | potwierdzona, gdy różnica w tygodniach świątecznych ≥ 5 p.p. i dolna granica przedziału > 0; w pozostałych tygodniach oczekuję różnicy blisko 0 (\|różnica\| < 2 p.p.). **Wcześniejsza wiedza:** z sekcji 4.10 widać, że YoY sklepów podąża za liczbą dni otwarcia (korelacja 0,81); to test na błędach prognoz, nie na surowym YoY. Tygodni świątecznych jest mało (kilka), więc moc testu jest mała. |
| H4 | Zimny start: dla serii z ≤ 6 tygodniami historii błąd jest wyraźnie wyższy, a podejście analogowe (M3) go zmniejsza względem M2. | (a) WAPE(M2) dla ≤ 6 tyg. historii minus WAPE(M2) dla dłuższej historii; (b) WAPE(M2) − WAPE(M3) na seriach z ≤ 6 tyg. historii; bootstrap po originach | (a) potwierdzona, gdy różnica ≥ 10 p.p.; (b) potwierdzona, gdy różnica ≥ 5 p.p. i dolna granica przedziału > 0. M3 nie ma strojonych parametrów (reguły ustalone z góry), więc ocenię ją na wszystkich originach; seria zimnych startów jest mała i skupiona w kilku originach |
| H5 | (własna) Wewnętrzne kryterium „WAPE sumy 4 tyg. < 30% dla top 20" jest osiągalne na poziomie model × lokalizacja. | WAPE najlepszej metody na originach testowych (z przedziałem) oraz dolna granica błędu wynikająca z samego szumu losowego sprzedaży (symulacja Poissona wokół prognozy) | osiągalne, gdy górna granica przedziału WAPE najlepszej metody < 30%; dolna granica szumu informuje, czy 30% jest w ogóle realistyczne |

### Plan walidacji

- **Rolling-origin backtest bez losowego podziału.** Originy (poniedziałki) co 2 tygodnie od 2026-01-05 do 2026-08-17 oraz dodatkowo 2026-08-24 (ostatni origin, którego 4 tygodnie mieszczą się w danych; bez niego ostatnim w siatce co 14 dni byłby 08-17). Razem 18 originów. Dla każdego originu uczenie tylko na tygodniach przed nim.
- **Strojenie vs test:** wybór cech, hiperparametrów i konfiguracji wyłącznie na originach do 2026-05-25 (11 punktów). Wynik końcowy na originach od 2026-06-08 (7 punktów). Po obejrzeniu wyniku testowego nie wracam do strojenia. Okna 4-tygodniowe sąsiednich originów (co 2 tyg.) nakładają się, więc punkty nie są niezależne, a przy 7 punktach testowych przedziały będą szerokie.
- **Prognozy przedziałowe 80%:** z reszt backtestu na originach strojeniowych (skalowanych pierwiastkiem z prognozy), sprawdzane na testowych.
- **Zakres wniosków:** backtest nie obejmuje okresu szczytu (listopad–grudzień), bo jedyną jego obserwacją jest 2025 r.; wniosków o szczycie nie wolno z niego wyciągać.

In [ ]:
def idx_of(d):
    return int(np.where(WX == pd.Timestamp(d))[0][0])

orig_dates = list(pd.date_range("2026-01-05", "2026-08-17", freq="14D")) + [pd.Timestamp("2026-08-24")]
ORIG = np.array([idx_of(d) for d in orig_dates])
N_O = len(ORIG)
TUNE = np.array([d <= pd.Timestamp("2026-05-25") for d in orig_dates])
TEST = ~TUNE
H4_ = np.arange(4)

ACT = np.stack([Y[o:o + 4].T for o in ORIG])                      # (origin, seria, h): sprzedaż rzeczywista (surowa)
FLG = np.stack([F[o:o + 4].T for o in ORIG])                      # flaga braku towaru w tygodniu docelowym
HIST_OK = np.stack([START_A <= o - 1 for o in ORIG])              # seria ma >= 1 tydzień historii w chwili origin
AGE = np.stack([o - START_A for o in ORIG])                       # tygodnie historii (o - start)
COLD = HIST_OK & (AGE <= 6)

TOP20 = np.zeros((N_O, S), bool)
for i, o in enumerate(ORIG):
    s8 = pd.Series(np.nansum(Y[o - 8:o], axis=0)).groupby(SER.model.values).sum()
    TOP20[i] = SER.model.isin(s8.nlargest(20).index).to_numpy()

ELIG = HIST_OK & ~FLG.any(2)                                     # punkty podlegające ocenie (przed ograniczeniem do metody)
print("Originy:", N_O, "| strojeniowe:", int(TUNE.sum()), f"({orig_dates[0].date()}…{[d for d, t_ in zip(orig_dates, TUNE) if t_][-1].date()})",
      "| testowe:", int(TEST.sum()), f"({[d for d, t_ in zip(orig_dates, TEST) if t_][0].date()}…{orig_dates[-1].date()})")
print("Punkty (origin x seria): z historią", int(HIST_OK.sum()), "| po wyłączeniu flag", int(ELIG.sum()), "| top 20 x lokalizacja", int((ELIG & TOP20).sum()),
      "| zimny start (<= 6 tyg.)", int((ELIG & COLD).sum()))
print("Wyłączone z oceny przez flagę braku towaru (origin x seria):", int((HIST_OK & FLG.any(2)).sum()))

# Narzędzia oceny: WAPE i bias agregowane po zbiorze (nie średnia z serii), bootstrap po originach
A4 = ACT.sum(2)

def per_origin(P4, mask):
    """Dla każdego originu: suma rzeczywista, suma błędów bezwzględnych, suma prognoz na zbiorze punktów."""
    m = mask & ~np.isnan(P4) & ~np.isnan(A4)
    a = np.where(m, A4, 0).sum(1); p = np.where(m, P4, 0).sum(1); e = np.where(m, np.abs(P4 - A4), 0).sum(1)
    return np.stack([a, e, p], 1)

def wape_bias(st, sel=None):
    s = st[sel].sum(0) if sel is not None else st.sum(0)
    return s[1] / s[0], (s[2] - s[0]) / s[0]

def boot_ci(stats_list, sel, fn, B=2000, seed=SEED):
    """Bootstrap po originach (z wektora sel), wspólna próbka originów dla wszystkich metod w stats_list."""
    rg = np.random.default_rng(seed)
    idx = np.where(sel)[0]
    res = []
    with np.errstate(divide="ignore", invalid="ignore"):
        for _ in range(B):
            pick = rg.choice(idx, size=len(idx), replace=True)
            res.append(fn(*[s[pick].sum(0) for s in stats_list]))
    return np.nanpercentile(res, [2.5, 97.5])

def summary(P4, mask, name):
    st = per_origin(P4, mask)
    rows = {}
    for lab, sel in (("strojenie", TUNE), ("test", TEST), ("wszystkie", np.ones(N_O, bool))):
        w, b = wape_bias(st, sel)
        lo, hi = boot_ci([st], sel, lambda s: s[1] / s[0])
        rows[lab] = f"WAPE {w:.1%} [{lo:.1%}; {hi:.1%}], bias {b:+.1%}"
    return pd.Series(rows, name=name)

## 6.3. Baseline'y: B1 (dzisiejsza praktyka) i B2 (rok poprzedni × poziom)

**Po co:** zanim zbuduję cokolwiek bardziej złożonego, muszę wiedzieć, jak bardzo myli się to, co firma robi dziś, i czy prosta alternatywa (ten sam tydzień rok wcześniej) już coś poprawia. Każda dalsza metoda będzie porównywana z nimi **na identycznych punktach**.

- **B1:** średnia tygodniowa z 4 ostatnich pełnych tygodni przed origin, ta sama wartość dla każdego z tygodni t+1…t+4 (suma 4 tyg. = 4 × średnia). Liczona na danych surowych (tak działa praktyka, więc zera z braku towaru obniżają średnią). Gdy seria ma mniej niż 4 tygodnie historii, bierze średnią z dostępnych tygodni (w tym niepełnego pierwszego). **B1c** to ten sam wzór na danych oczyszczonych z flag braku towaru; różnica B1 − B1c mierzy koszt sztucznych zer.
- **B2-ISO:** sprzedaż w tym samym numerze tygodnia ISO rok wcześniej (przesunięcie o 52 tygodnie) × współczynnik poziomu = suma z ostatnich 8 tygodni przed origin ÷ suma z tych samych 8 tygodni rok wcześniej. Dostępny tylko, gdy seria miała pełną historię rok temu (modele BASIC, SS25, AW25) i gdy do origin są co najmniej 4 tygodnie porównywalnej historii (od origin 2026-02-02).
- **B2-kal (wyrównanie z kalendarzem):** to samo, ale w przeliczeniu na **dzień otwarcia** (sztuki / liczba dni otwarcia danej lokalizacji), z korektą mnożnikiem dni otwarcia tygodnia docelowego, oraz z wyrównaniem **po wydarzeniu** zamiast po numerze tygodnia dla świąt ruchomych: tydzień przed Wielkanocą, tydzień Wielkanocy i tydzień po nim mapuję na odpowiednie tygodnie 2025 (a Boże Ciało na Boże Ciało 2025). Gdy sam tydzień sprzed roku przypada na takie święto, a w tym roku nie, biorę średnią z tygodni tuż poza tym blokiem. Reguły ustaliłem z góry, bez dopasowania do wyników.

In [ ]:
# --- B1 ---
fc_b1 = lambda Yx, o: np.nanmean(Yx[o - 4:o], axis=0)
P_B1 = np.stack([np.repeat(fc_b1(Y, o)[:, None], 4, 1) for o in ORIG])
P_B1c = np.stack([np.repeat(fc_b1(YI, o)[:, None], 4, 1) for o in ORIG])

# --- B2: mapowanie świąt ruchomych 2026 -> 2025 (indeksy tygodni) ---
wk_idx = lambda col, lo, hi: [i for i in np.where(CALX[col].to_numpy() == 1)[0] if lo <= i < hi]
e25, e26 = wk_idx("wk_easter", 0, 52)[0], wk_idx("wk_easter", 52, 94)[0]
c25, c26 = wk_idx("wk_corpus", 0, 52)[0], wk_idx("wk_corpus", 52, 94)[0]
EVMAP = {e26 + k: e25 + k for k in (-1, 0, 1)}
EVMAP[c26] = c25
BLOCKS = [[e25 - 1, e25, e25 + 1], [c25]]
print("Wielkanoc: tydzień", WX[e25].date(), "(2025) i", WX[e26].date(), "(2026) | Boże Ciało:", WX[c25].date(), "i", WX[c26].date())

YD = Y / OD[:T]                                                          # sztuki na dzień otwarcia (online: 7 dni)

def ref_weeks_cal(tw):
    if tw in EVMAP:
        return [EVMAP[tw]]
    r = tw - 52
    for b in BLOCKS:
        if r in b:
            return [b[0] - 1, b[-1] + 1]                                 # tydzień sprzed roku jest „świąteczny", tego roku nie: użyj sąsiadów bloku
    return [r]

def fc_b2(o, mode):
    P = np.full((S, 4), np.nan)
    w = min(8, o - 52)
    if w < 4:
        return P
    if mode == "iso":
        num, den = Y[o - w:o].sum(0), Y[o - w - 52:o - 52].sum(0)         # NaN, jeśli seria nie istniała w całym oknie
    else:
        num, den = YD[o - w:o].sum(0), YD[o - w - 52:o - 52].sum(0)
    ratio = np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)
    for h in range(4):
        tw = o + h
        if mode == "iso":
            P[:, h] = Y[tw - 52] * ratio
        else:
            P[:, h] = np.mean([YD[r] for r in ref_weeks_cal(tw)], 0) * ratio * OD[tw]
    return P

P_B2i = np.stack([fc_b2(o, "iso") for o in ORIG])
P_B2k = np.stack([fc_b2(o, "kal") for o in ORIG])
print("Punkty z prognozą B2 (origin x seria):", int((~np.isnan(P_B2i.sum(2))).sum()), "z", int(HIST_OK.sum()), "punktów z historią")

# --- Ocena na wspólnych punktach ---
S4 = lambda P: P.sum(2)
common_b2 = ELIG & ~np.isnan(S4(P_B2i)) & ~np.isnan(S4(P_B2k))
rows = []
for lab, msk in (("top 20 x lokalizacja", ELIG & TOP20), ("wszystkie serie", ELIG)):
    rows += [summary(S4(P_B1), msk, f"B1  | {lab} | wszystkie punkty"),
             summary(S4(P_B1c), msk, f"B1c | {lab} | wszystkie punkty")]
    mk = msk & common_b2
    rows += [summary(S4(P_B1), mk, f"B1  | {lab} | punkty z B2"), summary(S4(P_B2i), mk, f"B2-ISO | {lab} | punkty z B2"),
             summary(S4(P_B2k), mk, f"B2-kal | {lab} | punkty z B2")]
res_base = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 60):
    display(res_base)
print("Liczba punktów (origin x seria) w porównaniu z B2 (top 20):", int((ELIG & TOP20 & common_b2).sum()), "| wszystkie serie:", int((ELIG & common_b2).sum()))

# WAPE per horyzont dla B1 (top 20, wszystkie originy)
m_ = ELIG & TOP20
rows_h = []
for h in range(4):
    a = np.where(m_, ACT[:, :, h], 0).sum(); e = np.where(m_ & ~np.isnan(P_B1[:, :, h]), np.abs(P_B1[:, :, h] - ACT[:, :, h]), 0).sum()
    p = np.where(m_ & ~np.isnan(P_B1[:, :, h]), P_B1[:, :, h], 0).sum()
    rows_h.append({"horyzont (tydz.)": h + 1, "WAPE B1": e / a, "bias B1": (p - a) / a})
display(pd.DataFrame(rows_h).round(3))

**Co widzę.** (1) **Dzisiejsza praktyka (B1)** myli się średnio o **20,7%** sztuk sumy 4 tygodni na poziomie model × lokalizacja (top 20; przedział 95% z bootstrapu po originach 18,7–23,0%): 21,8% na originach strojeniowych i 19,1% na testowych. Błąd rośnie z horyzontem: 24,4% (tydz. 1), 25,4%, 26,0%, **30,0%** (tydz. 4). Bias jest mały (−1,9% łącznie): w sumie praktyka ani nie zawyża, ani nie zaniża, ale to średnia z okresów, w których błędy mają różne znaki (patrz H1 niżej). (2) Oczyszczenie danych z tygodni z podejrzeniem braku towaru (B1c) zmienia WAPE tylko o 0,2 p.p. (20,5% vs 20,7%), czyli sztuczne zera z braków prawie nie psują dzisiejszej średniej na tej próbie. (3) **B2** (rok poprzedni × poziom) da się policzyć tylko dla połowy punktów (1368 z 2748: modele z rocznym stażem, od origin 2026-02-02). Na punktach wspólnych (top 20, 1134 punkty) B1 ma 15,5% [13,9%; 17,4%], B2-ISO 13,8% [12,9%; 14,6%], B2-kal 14,2% [13,4%; 15,0%]; na tych punktach B1 ma bias −7,1% (zaniża w okresie wzrostu wiosennego, na originach strojeniowych −11,3%). (4) **Zaskoczenie:** wyrównanie kalendarzowe (dni otwarcia, przesunięcie świąt) **nie** poprawiło B2 ogółem, było o 0,4 p.p. gorsze od prostego wyrównania po numerze tygodnia ISO.

**Wniosek / decyzja.** Punkt odniesienia do pokonania: **WAPE ok. 20,7%** (top 20, 18 originów). B2 jest lepszy od B1 o ok. 1,7 p.p. na wspólnych punktach, ale łatwiejszy zbiór (modele BASIC o stabilnym wzorcu) sam obniża błąd B1 z 20,7% do 15,5%, więc nie wolno porównywać B2 na jego podzbiorze z B1 na całości. B2 nie jest kandydatem na metodę główną (brak prognozy dla połowy serii, w tym dla wszystkich nowych modeli); jego składniki (opóźnienie 52 tygodni, poziom sprzed roku) wchodzą do M2 jako cechy sezonowe. **Kryterium wewnętrzne „WAPE < 30%"** jest dla top 20 już spełnione przez B1 w ujęciu sumarycznym (20,7%); nie spełnia go tylko B1 w tygodniu 4 (30,0%). Wrócę do tego w H5.

## 6.4. Test H1: czy dzisiejsza praktyka jest obciążona przy zmianie sezonu?

**Po co:** H1 jest pierwszym pytaniem zleceniodawcy („raz mamy braki, raz nadmiar"). Dzielę komórki (origin × kategoria) na „wzrost" (sprzedaż kategorii w kolejnych 4 tyg. ≥ 15% wyższa niż w ostatnich 4 tyg.), „spadek" (≥ 15% niższa) i „stabilnie", i liczę bias B1 w każdej klasie z przedziałem z bootstrapu po originach. Do komórek wchodzą wszystkie serie kwalifikujące się do oceny; minimalna wielkość komórki to 100 szt. w oknie przed i po. Klasyfikacja używa tego, co się wydarzyło po origin, więc to jest opis **skali** obciążenia, a nie niezależny test jego istnienia (patrz zastrzeżenie przy H1).

In [ ]:
P4_b1 = S4(P_B1)
cell_rows = []
st_cls = {k: np.zeros((N_O, 3)) for k in ("wzrost", "spadek", "stabilnie")}
for i, o in enumerate(ORIG):
    for k, cname in enumerate(sorted(cat_codes)):
        sel = ELIG[i] & (cat_idx == k) & ~np.isnan(P4_b1[i])
        a = A4[i][sel].sum(); recent = np.nansum(Y[o - 4:o][:, sel])
        if a < 100 or recent < 100:
            continue
        ratio = a / recent
        cls = "wzrost" if ratio >= 1.15 else ("spadek" if ratio <= 0.85 else "stabilnie")
        e = np.abs(P4_b1[i][sel] - A4[i][sel]).sum(); p = P4_b1[i][sel].sum()
        st_cls[cls][i] += [a, e, p]
        cell_rows.append({"origin": orig_dates[i].date(), "miesiąc": orig_dates[i].strftime("%Y-%m"), "kategoria": cname, "zmiana_sprzedaży": ratio,
                          "klasa": cls, "rzeczywiste_4tyg": int(a), "bias_B1": (p - a) / a})
cells = pd.DataFrame(cell_rows)
out = []
for cls, st in st_cls.items():
    sel = st[:, 0] > 0
    b = wape_bias(st, sel)[1]
    lo, hi = boot_ci([st], sel, lambda s: (s[2] - s[0]) / s[0])
    out.append({"klasa": cls, "komórek (origin x kategoria)": int((cells.klasa == cls).sum()), "originów": int(sel.sum()),
                "bias B1": f"{b:+.1%}", "przedział 95% (bootstrap po originach)": f"[{lo:+.1%}; {hi:+.1%}]"})
display(pd.DataFrame(out))

# Gdzie obciążenie jest największe: bias B1 wg kategorii i miesiąca origin (komórki >= 300 szt. rzeczywistych)
cm = cells.assign(p=lambda d: d.rzeczywiste_4tyg * (1 + d.bias_B1)).groupby(["kategoria", "miesiąc"]).agg(a=("rzeczywiste_4tyg", "sum"), p=("p", "sum"))
cm["bias"] = (cm.p - cm.a) / cm.a
tab_bias = cm[cm.a >= 300].bias.unstack("miesiąc")
print("Bias B1 wg kategorii (wiersze) i miesiąca origin (kolumny), tylko komórki >= 300 szt.:")
display((tab_bias * 100).round(0).astype("Int64"))
print("Rozkład komórek wg klas:", cells.klasa.value_counts().to_dict())

**Co widzę.** Dzisiejsza praktyka (B1) ma bias **−28,0%** [−36,0%; −22,3%] w komórkach (origin × kategoria), w których sprzedaż kategorii w kolejnych 4 tygodniach wzrosła o ≥ 15% (22 komórki, 10 originów), **+37,9%** [+26,5%; +69,8%] tam, gdzie spadła o ≥ 15% (24 komórki, 11 originów) i **+0,4%** [−2,4%; +3,8%] w pozostałych 61 komórkach. Tabela po kategoriach i miesiącach pokazuje, gdzie to jest największe: **Kurtki i Swetry** na originach w lutym i marcu (bias +203% i +197% w lutym, +133% i +114% w marcu: średnia z zimowych tygodni zawyża popyt, gdy sezon wygasa), a w sierpniu odwrotnie (−70% i −75%: start kolekcji AW26, średnia z lata nie widzi rozbiegu). Poza tym: T-shirty −27% w marcu (rozbieg sezonu SS), Koszule +20% w lipcu–sierpniu, Sukienki −29% w marcu.

**Wniosek / decyzja.** **H1 potwierdzona** przy progach z 6.1 (bias ≤ −10% we wzroście z górną granicą przedziału < 0, bias ≥ +10% w spadku z dolną granicą > 0). Zastrzeżenie z 6.1 obowiązuje: znak biasu jest po części mechaniczny, bo komórki klasyfikuję wiedzą o przyszłości. Informacją jest **wielkość**: w kurtkach i swetrach dosyłka z dzisiejszej reguły potrafiła być ponad dwa razy za duża (luty) lub o trzy czwarte za mała (sierpień). Dzięki temu rozumiem, dlaczego łączny bias B1 jest mały (−1,9%) przy dużych błędach: nadwyżki i braki znoszą się w sumie. **Praktyczny wniosek dla zleceniodawcy:** bias trzeba oglądać po kategorii i okresie, nie łącznie; najbliższe okno (W39–W42) to właśnie początek wzrostu kurtek i swetrów (wejście kolekcji AW26), czyli kierunek błędu B1 jest znany: zaniżanie.

## 6.5. Test H3: wyrównanie z rokiem poprzednim po numerze tygodnia ISO a po kalendarzu

**Po co:** sprawdzam, czy wyrównanie po numerze tygodnia ISO psuje prognozę w tygodniach wokół świąt ruchomych. Oceniam **pojedyncze tygodnie docelowe** (każda para origin × horyzont daje jedną obserwację tygodnia; agreguję po tygodniu docelowym), a bootstrap losuje **tygodnie docelowe**. „Tydzień świąteczny" to tydzień, w którym wypada Wielkanoc (przed, w, po) albo Boże Ciało w danym roku **lub** rok wcześniej (bo wtedy wyrównanie po numerze tygodnia porównuje tydzień świąteczny z nieświątecznym). Próg i kierunek zapisałem w sekcji 6.1.

In [ ]:
M_w = ELIG[:, :, None] & ~np.isnan(P_B2i) & ~np.isnan(P_B2k)
TW = ORIG[:, None] + H4_[None, :]
rec = {}
for i in range(N_O):
    for h in range(4):
        m_ = M_w[i, :, h]
        if not m_.any():
            continue
        a_, pi_, pk_ = ACT[i, m_, h], P_B2i[i, m_, h], P_B2k[i, m_, h]
        r = rec.setdefault(int(TW[i, h]), np.zeros(3))
        r += [a_.sum(), np.abs(pi_ - a_).sum(), np.abs(pk_ - a_).sum()]
wk_tab = pd.DataFrame(rec, index=["A", "E_iso", "E_kal"]).T.sort_index()
wk_tab["swieto"] = [bool(MOVING[tw] or (tw - 52 >= 0 and MOVING[tw - 52])) for tw in wk_tab.index]
wk_tab.index = [WX[i].date() for i in wk_tab.index]

def boot_diff(d, B=2000):
    rg = np.random.default_rng(SEED)
    vals = []
    for _ in range(B):
        s = d.iloc[rg.integers(0, len(d), len(d))][["A", "E_iso", "E_kal"]].sum()
        vals.append((s.E_iso - s.E_kal) / s.A)
    return np.percentile(vals, [2.5, 97.5])

out = []
for lab, d in (("tygodnie świąteczne", wk_tab[wk_tab.swieto]), ("pozostałe tygodnie", wk_tab[~wk_tab.swieto])):
    s = d[["A", "E_iso", "E_kal"]].sum()
    lo, hi = boot_diff(d)
    out.append({"grupa": lab, "tygodni docelowych": len(d), "WAPE B2-ISO": s.E_iso / s.A, "WAPE B2-kal": s.E_kal / s.A,
                "różnica ISO − kal [p.p.]": 100 * (s.E_iso - s.E_kal) / s.A, "przedział 95% [p.p.]": f"[{100 * lo:+.1f}; {100 * hi:+.1f}]"})
display(pd.DataFrame(out).round(3))
hol = wk_tab[wk_tab.swieto].assign(WAPE_ISO=lambda d: d.E_iso / d.A, WAPE_kal=lambda d: d.E_kal / d.A)[["A", "WAPE_ISO", "WAPE_kal"]]
print("Tygodnie świąteczne w próbie (A = suma rzeczywista w ocenianych seriach):")
display(hol.round(3))

**Co widzę.** W 7 tygodniach docelowych wokół świąt ruchomych (Wielkanoc: tydzień przed, w, po; Boże Ciało) WAPE B2-ISO wynosi 25,4%, a B2-kal 23,6%; różnica **+1,7 p.p.** [−0,7; +5,0]. W pozostałych 26 tygodniach różnica to +0,3 p.p. [0,0; +0,9]. W podziale na tygodnie wyraźny efekt jest tylko w jednym: tydzień po Wielkanocy (2026-04-06): B2-ISO 36,5% vs B2-kal 25,3%. W czterech z siedmiu tygodni wyrównanie kalendarzowe jest lepsze (03-30, 04-06, 06-01, 06-15), w dwóch gorsze (03-23, 04-20), w jednym równe.

**Wniosek / decyzja.** **H3 niejednoznaczna:** kierunek zgodny z hipotezą, ale wielkość (1,7 p.p.) jest poniżej progu 5 p.p., a przedział zawiera 0. Efekt jest skupiony w jednym tygodniu, a próba to zaledwie 7 tygodni z jedną Wielkanocą i jednym Bożym Ciałem w 2026 r., więc **moc testu jest mała**; nie rozstrzyga to, że święta ruchome nie mają znaczenia, tylko że ten test tego nie wykazał. W pozostałych tygodniach obie wersje są praktycznie równe, jak przewidywałem. Decyzja: nie uzależniam od H3 wyboru metody głównej; kalendarz (dni otwarcia, znaczniki Wielkanocy i Bożego Ciała) wchodzi do M2 jako cechy, a wkład tej grupy mierzę w ablacji. W oknie W39–W42 nie ma świąt ruchomych, więc dla prognozy właściwej H3 jest bez znaczenia.

## 6.6. M1: statystyczna metoda per seria (ETS z tłumionym trendem + indeks sezonowy kategorii)

**Po co:** M1 to klasyczny kandydat „jedna seria = jeden model". Wybór uzasadniony wcześniej: sezonowość roczna **nie jest** estymowana per seria, bo mamy mniej niż dwa pełne cykle roczne (Holt-Winters z okresem 52 tygodni nie ma tu podstaw: ok. 90 punktów na 52 parametry sezonowe). Zamiast tego:
1. **indeks sezonowy kategorii** (osobno dla sklepów stacjonarnych i online) z 2025 r.: sprzedaż na dzień otwarcia w tygodniu ISO ÷ średnia roczna, wygładzony średnią kroczącą z 3 tygodni; używany wyłącznie przy originach w 2026 r., więc cały jest „z przeszłości";
2. seria odsezonowana (sztuki na dzień otwarcia ÷ indeks) modelowana **ETS z trendem addytywnym tłumionym** (statsmodels, parametry estymowane per seria i origin);
3. prognoza = wartość z ETS × indeks sezonowy tygodnia docelowego × dni otwarcia tygodnia docelowego.

Zasady ustalone z góry: seria z mniej niż 10 tygodniami historii (bez niepełnego pierwszego tygodnia) dostaje prognozę B1 na danych oczyszczonych (to jawny zapas, liczony osobno); prognozę ETS ograniczam do przedziału [0, 3 × maksimum z ostatnich 8 tygodni], żeby tłumiony trend nie „uciekał" przy szybkim rozbiegu; modele estymuję na danych oczyszczonych z flag braku towaru. **NIEPEWNOŚĆ:** indeks sezonowy kategorii z 2025 r. zawiera też rozbiegi premier z 2025 r. (np. start swetrów w sierpniu), więc jest dobry tylko tyle, na ile 2026 powtarza kalendarz premier 2025.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

grp_idx = is_online.astype(int)
def season_index():
    """Indeks [tydzień ISO 1..53, kategoria, grupa]: sprzedaż na dzień otwarcia z tygodni 2025-W02 … 2026-W01, wygładzona."""
    ix = np.ones((54, len(cat_codes), 2))
    for k in range(len(cat_codes)):
        for g in (0, 1):
            cols = (cat_idx == k) & (grp_idx == g)
            v = np.nansum(YD[:52][:, cols], axis=1)                       # idx 0..50 = W02..W52 (2025), idx 51 = W01 (2026-01, znane przed każdym origin w 2026)
            w_of = [2 + i for i in range(51)] + [1]
            ser = pd.Series(v, index=w_of).sort_index()                    # numery tygodni 1..52
            sm = (ser / ser.mean()).rolling(3, center=True, min_periods=1).mean().clip(lower=0.05)
            ix[1:53, k, g] = sm.to_numpy()
            ix[53, k, g] = sm.iloc[-1]
    return ix

IDXS = season_index()
ZD = YI / OD[:T] / IDXS[ISOW[:T][:, None], cat_idx[None, :], grp_idx[None, :]]    # odsezonowane sztuki na dzień otwarcia

def fc_m1(o):
    P = np.full((S, 4), np.nan)
    fb = np.zeros(S, bool)
    fidx = IDXS[ISOW[o:o + 4][:, None], cat_idx[None, :], grp_idx[None, :]]        # (4, S)
    for c in range(S):
        if START_A[c] > o - 1:
            continue
        z = ZD[START_A[c] + 1:o, c]
        if len(z) < 10:
            P[c] = np.nanmean(YI[o - 4:o, c]); fb[c] = True
            continue
        if z.sum() <= 0:
            P[c] = 0.0
            continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = ExponentialSmoothing(z, trend="add", damped_trend=True, initialization_method="estimated").fit(optimized=True)
            fz = np.clip(fit.forecast(4), 0, 3 * z[-8:].max())
        except Exception:
            fz = np.full(4, z[-4:].mean()); fb[c] = True
        P[c] = fz * fidx[:, c] * OD[o:o + 4, c]
    return P, fb

cache_m1 = CACHE / "m1_forecasts_v1.npz"
t0 = time.time()
if cache_m1.exists():
    z_ = np.load(cache_m1); P_M1, FB_M1 = z_["P"], z_["FB"]
    print("M1 wczytane z cache")
else:
    out_ = [fc_m1(o) for o in ORIG]
    P_M1 = np.stack([p for p, _ in out_]); FB_M1 = np.stack([f_ for _, f_ in out_])
    np.savez(cache_m1, P=P_M1, FB=FB_M1)
print(f"Czas M1: {time.time() - t0:.0f} s | punkty z prognozą: {int((~np.isnan(S4(P_M1))).sum())} | z zapasem B1 (za krótka historia lub błąd ETS): {int((FB_M1 & HIST_OK).sum())}")

rows = []
for lab, msk in (("top 20 x lokalizacja", ELIG & TOP20), ("wszystkie serie", ELIG)):
    rows += [summary(S4(P_B1), msk, f"B1 | {lab}"), summary(S4(P_M1), msk, f"M1 | {lab}")]
res_m1 = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 60):
    display(res_m1)

**Co widzę.** M1 (ETS z tłumionym trendem × indeks sezonowy kategorii) ma na top 20 WAPE **18,7%** [15,4%; 23,7%] na wszystkich originach, czyli o 2,0 p.p. mniej niż B1 (20,7%), ale przedział jest szeroki. Wynik jest rozjechany między okresami: na originach strojeniowych 21,2% [15,9%; 30,0%] (praktycznie bez poprawy względem B1 21,8%, bias +3,3%), na testowych 15,4% [13,3%; 17,3%] (B1 19,1%). W 258 punktach (serie krótsze niż 10 tygodni lub błąd dopasowania) M1 użył zapasu B1c, więc na zimnym starcie jest dokładnie tym samym co B1.

**Wniosek / decyzja.** Metoda per seria daje niewielką, niestabilną poprawę. Przyczyną najgorszych wyników jest wąska historia (do 52 tygodni na początku 2026 r.) i indeks sezonowy kategorii z jednego roku, który zawiera premiery z 2025 r. (rozbiegi swetrów w sierpniu), więc jest dobry tylko, o ile 2026 powtarza kalendarz premier 2025. Nie zbadałem jeszcze, dlaczego M1 myli się szczególnie mocno na początku 2026 r. (w zestawieniu per origin w 6.9 jeden origin ma błąd ponad 50%); to hipoteza do sprawdzenia w diagnostyce, nie ustalony fakt. M1 zostaje w porównaniu jako przedstawiciel rodziny „jedna seria = jeden model".

## 6.7. M2: globalny model gradientowy (LightGBM) i ablacja grup cech

**Po co:** jeden model uczony wspólnie na wszystkich seriach (pooling) powinien lepiej wykorzystać strukturę wspólną dla modeli i lokalizacji (kształt premiery, sezonowość kategorii) niż osobne modele per seria, które widzą po kilkadziesiąt punktów. Cel: sprzedaż sztuk w tygodniu `t+h−1` (jeden model na wszystkie horyzonty, `h` jako cecha), funkcja straty **Poissona** (dane to liczby sztuk, rozkład silnie prawoskośny; Tweedie sprawdzam jako wariant). Biblioteka: LightGBM (zainstalowany, wersja w sekcji 2). Dla każdego originu model uczy się **od nowa** wyłącznie na wierszach, których cel jest znany przed originem (tydzień docelowy ≤ origin − 1), z pominięciem tygodni z flagą braku towaru w celu.

**Plan strojenia (ustalony z góry, tylko originy strojeniowe do 2026-05-25):**
1. ablacja skumulowana: dodaję grupy cech po kolei (lag → tożsamość → pooling → kalendarz → sezon → rabat → Kraków) i patrzę, co poprawia WAPE sumy 4 tyg. (top 20 × lokalizacja);
2. ablacja „bez jednej grupy" od pełnego zestawu;
3. **reguła wyboru zestawu:** grupę usuwam, jeśli jej usunięcie nie pogarsza WAPE strojenia o więcej niż 0,2 p.p. (preferuję prostszy model); zredukowany zestaw akceptuję, jeśli jego WAPE jest nie gorszy niż pełnego o więcej niż 0,2 p.p.;
4. mała siatka hiperparametrów (liczba liści × liczba drzew; Poisson vs Tweedie), wybór po WAPE strojenia.

Wyniki na originach testowych będą odczytane dopiero po zamrożeniu konfiguracji.

In [ ]:
import lightgbm as lgb, pickle

CAT_F = ["cat", "store", "coll"]
ORDER = ["lag", "ident", "pool", "cal", "season", "disc", "krk"]
FEATS_ALL = ["h"] + [f for g in ORDER for f in GROUPS[g]]
BASE_PARAMS = dict(objective="poisson", learning_rate=0.05, num_leaves=15, min_child_samples=30, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, reg_lambda=1.0, n_estimators=300, random_state=SEED, n_jobs=4, verbose=-1,
                   deterministic=True, force_row_wise=True)
M2_CACHE = CACHE / "m2_runs_v1.pkl"
m2_store = pickle.loads(M2_CACHE.read_bytes()) if M2_CACHE.exists() else {}
KNOWN = ROWS[ROWS.y.notna() & ~ROWS.flag_tw]                       # wiersze z celem znanym i bez flagi braku towaru

def run_m2(feats, params, origin_mask):
    """Prognozy (origin, seria, h) dla wskazanych originów; model uczony od nowa na danych znanych przed originem."""
    key = (tuple(feats), tuple(sorted(params.items())))
    d = m2_store.setdefault(key, {})
    for i in np.where(origin_mask)[0]:
        if i in d:
            continue
        o = ORIG[i]
        tr, te = KNOWN[KNOWN.tw <= o - 1], ROWS[ROWS.t == o]
        mdl = lgb.LGBMRegressor(**params).fit(tr[feats], tr.y, categorical_feature=[f for f in CAT_F if f in feats])
        arr = np.full((S, 4), np.nan)
        arr[te.c.to_numpy(), te.h.to_numpy() - 1] = mdl.predict(te[feats])
        d[i] = arr
    M2_CACHE.write_bytes(pickle.dumps(m2_store))
    return np.stack([d.get(i, np.full((S, 4), np.nan)) for i in range(N_O)])

MAIN = ELIG & TOP20
def tune_wape(P, mask=MAIN):
    st = per_origin(S4(P), mask)
    return wape_bias(st, TUNE)

t0 = time.time()
cum, feats = [], ["h"]
for g in ORDER:
    feats = feats + GROUPS[g]
    cum.append((f"+ {g}", list(feats)))
res_cum = []
for lab, fs in cum:
    P_ = run_m2(fs, BASE_PARAMS, TUNE)
    w, b = tune_wape(P_); wa, _ = tune_wape(P_, ELIG)
    res_cum.append({"zestaw cech": lab, "liczba cech": len(fs), "WAPE strojenie top 20": w, "bias": b, "WAPE strojenie wszystkie serie": wa})
res_cum = pd.DataFrame(res_cum)
res_cum["Δ vs poprzedni [p.p.]"] = 100 * res_cum["WAPE strojenie top 20"].diff()
print(f"Czas: {time.time() - t0:.0f} s")
display(res_cum.round(4))

**Co widzę (ablacja skumulowana, WAPE na originach strojeniowych, top 20).** Sam zestaw opóźnień (9 cech) daje 20,5% (B1 na tych samych originach: 21,8%). Kolejne grupy: tożsamość serii (kategoria, lokalizacja, kolekcja, wiek) −1,6 p.p. (18,9%), pooling (poziom modelu we wszystkich lokalizacjach, udział lokalizacji, rozpęd kategorii) −0,2 p.p., **kalendarz** (dni otwarcia, święta, niedziele handlowe) **−2,4 p.p.** (16,2%), sezon (numer tygodnia, opóźnienie 52 tyg., poziom sprzed roku) −1,2 p.p. (15,1%), rabat −0,3 p.p. (14,8%), znaczniki Krakowa +0,01 p.p. (14,8%).

**Wniosek.** Samo przejście z prostej średniej na model gradientowy z opóźnieniami prawie niczego nie daje (ok. 1,3 p.p.); przewagę buduje **kontekst**: kalendarz otwarcia sklepów (liczba dni otwarcia tygodnia mocno zmienia sprzedaż sklepu, a model per seria tego nie widzi) i tożsamość serii. Znaczniki Krakowa nic nie wnoszą (podobnie szeregi po przebudowie są już opisane przez opóźnienia). Pooling wypada słabo w tej kolejności dodawania, ale kolejność wpływa na wynik, bo cechy zachodzą na siebie; dlatego druga ablacja poniżej.

In [ ]:
t0 = time.time()
w_full, _ = tune_wape(run_m2(FEATS_ALL, BASE_PARAMS, TUNE))
res_loo = []
for g in ORDER[1:]:
    fs = ["h"] + [f for gg in ORDER if gg != g for f in GROUPS[gg]]
    P_ = run_m2(fs, BASE_PARAMS, TUNE)
    w, b = tune_wape(P_)
    res_loo.append({"usunięta grupa": g, "WAPE strojenie top 20": w, "bias": b, "Δ vs pełny [p.p.]": 100 * (w - w_full)})
res_loo = pd.DataFrame(res_loo)
print(f"Czas: {time.time() - t0:.0f} s | pełny zestaw: WAPE strojenia {w_full:.4f}")
display(res_loo.round(4))

drop = [r["usunięta grupa"] for _, r in res_loo.iterrows() if r["Δ vs pełny [p.p.]"] <= 0.2]
feats_red = ["h"] + [f for g in ORDER if g not in drop for f in GROUPS[g]]
w_red, b_red = tune_wape(run_m2(feats_red, BASE_PARAMS, TUNE))
accept_red = (w_red - w_full) * 100 <= 0.2
FEATS_SEL = feats_red if accept_red else FEATS_ALL
print("Grupy do usunięcia wg reguły (Δ ≤ 0,2 p.p.):", drop or "brak", "| WAPE zredukowanego:", round(w_red, 4),
      "| zredukowany zaakceptowany:", accept_red, "| liczba cech w wybranym zestawie:", len(FEATS_SEL))

**Co widzę (ablacja „bez jednej grupy" od pełnego zestawu; pełny zestaw: WAPE strojenia 14,8%).** Usunięcie grupy pogarsza WAPE o: tożsamość +1,2 p.p., pooling +1,3 p.p., kalendarz +0,8 p.p., sezon +1,1 p.p., rabat +0,25 p.p., Kraków −0,01 p.p. Reguła wyboru z góry (usuń grupę, jeśli strata ≤ 0,2 p.p.) usuwa tylko znaczniki Krakowa; zredukowany zestaw (31 cech) ma WAPE strojenia 14,8%, więc go przyjmuję.

**Wniosek.** W tym ujęciu każda grupa poza Krakowem coś wnosi, a pooling, który w ablacji skumulowanej wyglądał na bezużyteczny (−0,2 p.p.), okazuje się jedną z najważniejszych grup (+1,3 p.p. po usunięciu): informacja o poziomie całego modelu w pozostałych lokalizacjach dubluje się z tożsamością serii i opóźnieniami, więc jej wkład widać dopiero, gdy się ją odejmie. Lekcja: **kolejność w ablacji skumulowanej potrafi zmylić; ablacja odwrotna daje pełniejszy obraz**. Rabat (+0,25 p.p.) przekracza próg 0,2 p.p. tylko nieznacznie; zostawiam go (decyzja z reguły, nie z intuicji), ale w prognozie właściwej cecha jest zerowa (brak akcji w W39–W42), więc jego wkład będzie tam nieistotny.

In [ ]:
t0 = time.time()
grid = []
for nl in (7, 15, 31):
    for ne in (150, 300, 600):
        grid.append((f"Poisson, liście={nl}, drzewa={ne}", dict(BASE_PARAMS, num_leaves=nl, n_estimators=ne)))
grid.append(("Tweedie p=1.3, liście=15, drzewa=300", dict(BASE_PARAMS, objective="tweedie", tweedie_variance_power=1.3)))
res_grid = []
for lab, pr in grid:
    w, b = tune_wape(run_m2(FEATS_SEL, pr, TUNE))
    res_grid.append({"konfiguracja": lab, "WAPE strojenie top 20": w, "bias": b})
res_grid = pd.DataFrame(res_grid)
best = res_grid["WAPE strojenie top 20"].idxmin()
PARAMS_SEL = grid[best][1]
print(f"Czas: {time.time() - t0:.0f} s")
display(res_grid.round(4))
print("Wybrana konfiguracja (wg WAPE strojenia):", grid[best][0])

**Co widzę (siatka hiperparametrów, WAPE strojenia, top 20).** Błąd spada głównie z liczbą drzew: przy 150 drzewach 16,5–18,8%, przy 300 drzewach 14,5–15,7%, przy 600 drzewach 13,8–14,1%; liczba liści ma wpływ mniejszy (przy 600 drzewach 13,8% dla 15 liści, 14,1% dla 7 i 31). Wariant Tweedie (p = 1,3, 300 drzew) daje 14,1%, czyli lepiej niż Poisson przy tej samej liczbie drzew (14,8%), ale gorzej niż Poisson z 600 drzewami. Wybrana konfiguracja: **Poisson, 15 liści, 600 drzew** (13,8%).

**Wniosek / decyzja i zastrzeżenie.** Różnice między trzema najlepszymi konfiguracjami (13,8–14,1%) są mniejsze niż rozrzut między originami, więc wybór jest mało istotny. **Uwaga metodologiczna:** wybrana wartość leży na brzegu siatki (600 to najwięcej drzew) i błąd wciąż maleje, więc przy stałej szybkości uczenia (0,05) większa liczba drzew mogłaby dać jeszcze niewielką poprawę. **Nie rozszerzam siatki:** wynik M2 na originach testowych jest już widoczny (6.9), a zasada z 6.1 zabrania wracania do strojenia po jego obejrzeniu. Skutek: M2 może być nieco niedostrojony, więc jego wynik traktuję jako ostrożny.

## 6.8. M3: zimny start z analogów (krzywa premiery z podobnych modeli)

**Po co:** dla modelu z ≤ 6 tygodniami historii seria własna jest za krótka na cokolwiek poza „jak dotąd". Natomiast kształt przebiegu po premierze jest podobny między modelami (sekcja 4.13), a różnią się skalą. M3 wykorzystuje to wprost:
1. **analogi:** modele nowych kolekcji (bez BASIC, bo nie mamy ich premier) z tej samej kategorii, które w chwili origin mają za sobą co najmniej 4 pełne tygodnie życia; gdy w kategorii jest mniej niż 2 analogi dla potrzebnego wieku, biorę wszystkie nowe modele; gdy to też za mało, M3 nie daje prognozy;
2. **krzywa** `g(a)`: sprzedaż modelu w wieku `a` tygodni (0 = tydzień pierwszej sprzedaży) podzielona przez jego średnią z wieku 1–4, a potem **mediana** po analogach (mediana, żeby jeden dziwny model nie zdominował krzywej); analog wchodzi do wieku `a` tylko wtedy, gdy ten wiek był już obserwowany przed originem;
3. **poziom** modelu `L` = suma sprzedaży z obserwowanych tygodni ÷ suma `g` z tych samych wieków (estymator ilorazowy, uwzględnia niepełny tydzień 0);
4. prognoza modelu na tydzień w wieku `a` = `L · g(a)`, rozdzielona na lokalizacje według udziałów z obserwowanych tygodni.
Wszystkie reguły ustaliłem z góry; nie ma w nim strojonych parametrów, więc ocenię go na wszystkich originach. **NIEPEWNOŚĆ:** analogów jest mało (w kategorii Swetry tylko jeden starszy model), a krzywe zawierają też efekt pory roku.

In [ ]:
models = sorted(style_cols)
start_m = np.array([start[m_] for m_ in models])
cat_m = np.array([cat_of[m_] for m_ in models])
new_m = np.array([m_ not in basic for m_ in models])
MSm = np.stack([np.nansum(YI[:, style_cols[m_]], axis=1) for m_ in models], 1)       # sprzedaż modelu (wszystkie lokalizacje), (tydzień, model)

def analog_g(o, j, ages):
    """Krzywa g(a) dla wymaganych wieków; analogi z kategorii modelu j, w razie braku wszystkie nowe modele; None, gdy brak."""
    def curves(pool):
        C = np.full((len(pool), len(ages)), np.nan)
        for r, jj in enumerate(pool):
            s = start_m[jj]
            if s + 4 > o - 1:
                continue
            E = MSm[s + 1:s + 5, jj].mean()
            if E <= 0:
                continue
            for k, a in enumerate(ages):
                if s + a <= o - 1:
                    C[r, k] = MSm[s + a, jj] / E
        return C
    pools = [[jj for jj in range(len(models)) if new_m[jj] and jj != j and cat_m[jj] == cat_m[j]], [jj for jj in range(len(models)) if new_m[jj] and jj != j]]
    for pool in pools:
        C = curves(pool)
        if C.size and ((~np.isnan(C)).sum(0) >= 2).all():
            return np.nanmedian(C, axis=0)
    return None

def fc_m3(o):
    P = np.full((S, 4), np.nan)
    for j, m_ in enumerate(models):
        s = start_m[j]; age_o = o - s
        if not (1 <= age_o <= 6) or MSm[s:o, j].sum() <= 0:
            continue
        g = analog_g(o, j, list(range(age_o + 4)))
        if g is None or g[:age_o].sum() <= 0:
            continue
        obs = MSm[s:o, j]
        fm = obs.sum() / g[:age_o].sum() * g[age_o:age_o + 4]
        ix = style_cols[m_]
        share = np.nansum(YI[s:o][:, ix], axis=0) / obs.sum()
        P[ix] = share[:, None] * fm[None, :]
    return P

# Zamrożona konfiguracja M2: prognozy na wszystkich originach (strojenie i test)
P_M2 = run_m2(FEATS_SEL, PARAMS_SEL, np.ones(N_O, bool))
P_M3 = np.stack([fc_m3(o) for o in ORIG])
has_m3 = ~np.isnan(S4(P_M3))
print("Punkty zimnego startu (<= 6 tyg., po wyłączeniu flag):", int((ELIG & COLD).sum()), "| z prognozą M3:", int((ELIG & COLD & has_m3).sum()),
      "| w tym top 20:", int((ELIG & COLD & has_m3 & TOP20).sum()), "| originów z takimi punktami:", int((ELIG & COLD & has_m3).any(1).sum()))
coldm = ELIG & COLD & has_m3
rows = []
for lab, P_ in (("B1 (praktyka)", P_B1), ("B1c (oczyszczona)", P_B1c), ("M1 (ETS)", P_M1), ("M2 (LightGBM)", P_M2), ("M3 (analogi)", P_M3)):
    st = per_origin(S4(P_), coldm)
    sel = st[:, 0] > 0
    w, b = wape_bias(st, sel); lo, hi = boot_ci([st], sel, lambda s: s[1] / s[0])
    rows.append({"metoda": lab, "WAPE (zimny start, wszystkie originy)": f"{w:.1%} [{lo:.1%}; {hi:.1%}]", "bias": f"{b:+.1%}"})
display(pd.DataFrame(rows))

# H4(a): M2 na zimnym starcie vs na dłuższej historii; H4(b): M2 vs M3 na zimnym starcie
warm = ELIG & ~COLD & ~np.isnan(S4(P_M2))
st_c, st_w = per_origin(S4(P_M2), ELIG & COLD & ~np.isnan(S4(P_M2))), per_origin(S4(P_M2), warm)
sel_cw = (st_c[:, 0] > 0) & (st_w[:, 0] > 0)
d_a = st_c[sel_cw].sum(0)[1] / st_c[sel_cw].sum(0)[0] - st_w[sel_cw].sum(0)[1] / st_w[sel_cw].sum(0)[0]
lo_a, hi_a = boot_ci([st_c, st_w], sel_cw, lambda c_, w_: c_[1] / c_[0] - w_[1] / w_[0])
st_2, st_3 = per_origin(S4(P_M2), coldm), per_origin(S4(P_M3), coldm)
sel23 = st_2[:, 0] > 0
d_b = st_2[sel23].sum(0)[1] / st_2[sel23].sum(0)[0] - st_3[sel23].sum(0)[1] / st_3[sel23].sum(0)[0]
lo_b, hi_b = boot_ci([st_2, st_3], sel23, lambda a_, b_: a_[1] / a_[0] - b_[1] / b_[0])
print(f"H4(a): WAPE M2 zimny start − WAPE M2 dłuższa historia = {100 * d_a:+.1f} p.p. [{100 * lo_a:+.1f}; {100 * hi_a:+.1f}] (originów: {int(sel_cw.sum())})")
print(f"H4(b): WAPE M2 − WAPE M3 na zimnym starcie = {100 * d_b:+.1f} p.p. [{100 * lo_b:+.1f}; {100 * hi_b:+.1f}] (originów: {int(sel23.sum())})")

**Co widzę.** Zimny start (≤ 6 tygodni historii) dotyczy 186 punktów (102 w top 20) w 8 originach. WAPE na zimnym starcie: B1 **51,5%** [33,6%; 68,1%] (bias −51,1%), M1 identyczny z B1 (zapas, bo za krótka seria), M2 **26,3%** [17,7%; 36,4%] (bias −17,8%), M3 (analogi) **32,2%** [23,4%; 43,1%] (bias −13,8%). **H4(a):** błąd M2 na zimnym starcie minus błąd M2 na dłuższej historii = **+13,7 p.p.** [+4,3; +25,0] (8 originów). **H4(b):** WAPE(M2) − WAPE(M3) = **−5,9 p.p.** [−10,5; −2,2]: M3 jest gorszy od M2.

**Wniosek / decyzja.** H4(a): **potwierdzona wg progu** (≥ 10 p.p.), ale przedział jest szeroki (dolna granica 4,3 p.p. leży poniżej progu) i opiera się na 8 originach, więc siłę potwierdzenia oceniam jako umiarkowaną: zimny start jest wyraźnie trudniejszy. H4(b): **odrzucona**: analogi nie zmniejszają błędu względem M2, tylko go zwiększają o ok. 6 p.p. (górna granica przedziału < 0). **Zaskoczenie i zmiana zdania:** spodziewałem się, że dedykowana metoda dla nowych modeli wygra, a model globalny z cechami wieku, poziomu modelu we wszystkich lokalizacjach i rozpędu kategorii radzi sobie lepiej (hipoteza robocza: uczy się kształtu premiery z wielu startów jednocześnie i nie zależy od dopasowania kategorii analogów, których jest tylko kilka). M3 ma mniejsze obciążenie (−13,8% vs −17,8%), ale większy rozrzut. Nie dodaję M3 do metody głównej. **Ważne dla zleceniodawcy:** na zimnym starcie nawet M2 zaniża o ok. 18%; to uzasadnia osobną flagę „nowy model" i ręczną korektę w górę przy tych modelach. Kombinacji M2 i M3 nie sprawdzałem (zasada: bez zmierzonego uzasadnienia).

## 6.9. Zestawienie metod na identycznych punktach i test H2 (pooling)

**Po co:** do tej pory każda metoda była oceniana osobno. Teraz zestawiam B1, M1 i M2 (oraz prostą kombinację M1+M2) na **dokładnie tych samych punktach** (origin × seria z top 20, bez serii z flagą braku towaru), osobno na originach strojeniowych i testowych, i liczę **sparowane różnice WAPE względem B1** z bootstrapem po originach. Kombinację M1+M2 (średnia arytmetyczna prognoz) dodaję dopiero teraz, gdy znam jej składniki, zgodnie z zasadą „kombinacje po zmierzeniu składników"; zachowam ją tylko wtedy, gdy na originach **strojeniowych** poprawia WAPE najlepszego składnika o co najmniej 0,5 p.p.
**H2** (próg z 6.1): różnica WAPE(M1) − WAPE(M2) na originach testowych ≥ 3 p.p., dolna granica przedziału > 0 i |bias M2| ≤ |bias M1| + 3 p.p.

In [ ]:
P_C12 = (P_M1 + P_M2) / 2
METH = {"B1 (praktyka)": P_B1, "M1 (ETS)": P_M1, "M2 (LightGBM)": P_M2, "M1+M2 (średnia)": P_C12}
ALLOK = ELIG & ~np.isnan(S4(P_M1)) & ~np.isnan(S4(P_M2)) & ~np.isnan(S4(P_B1))
MAINP = ALLOK & TOP20
print("Punkty głównego porównania (top 20 x lokalizacja, wszystkie metody mają prognozę):", int(MAINP.sum()), "| originów:", N_O)

res_all = pd.DataFrame([summary(S4(P_), MAINP, n_) for n_, P_ in METH.items()])
with pd.option_context("display.max_colwidth", 60):
    display(res_all)

def paired(Pa, Pb, mask, sel):
    sa, sb = per_origin(S4(Pa), mask), per_origin(S4(Pb), mask)
    wa, ba = wape_bias(sa, sel); wb, bb = wape_bias(sb, sel)
    lo, hi = boot_ci([sa, sb], sel, lambda a_, b_: a_[1] / a_[0] - b_[1] / b_[0])
    return 100 * (wa - wb), 100 * lo, 100 * hi

rows = []
for n_ in ("M1 (ETS)", "M2 (LightGBM)", "M1+M2 (średnia)"):
    r = {"metoda": n_}
    for lab, sel in (("strojenie", TUNE), ("test", TEST), ("wszystkie", np.ones(N_O, bool))):
        d, lo, hi = paired(METH[n_], P_B1, MAINP, sel)
        r[f"Δ WAPE vs B1, {lab} [p.p.]"] = f"{d:+.1f} [{lo:+.1f}; {hi:+.1f}]"
    rows.append(r)
display(pd.DataFrame(rows))

# H2 na originach testowych
d_h2, lo_h2, hi_h2 = paired(P_M1, P_M2, MAINP, TEST)
b1_ = wape_bias(per_origin(S4(P_M1), MAINP), TEST)[1]; b2_ = wape_bias(per_origin(S4(P_M2), MAINP), TEST)[1]
print(f"H2 (test): WAPE(M1) − WAPE(M2) = {d_h2:+.1f} p.p. [{lo_h2:+.1f}; {hi_h2:+.1f}] | bias M1 {b1_:+.1%}, bias M2 {b2_:+.1%}")
d_h2t, lo_h2t, hi_h2t = paired(P_M1, P_M2, MAINP, TUNE)
print(f"(dla porównania, strojenie): {d_h2t:+.1f} p.p. [{lo_h2t:+.1f}; {hi_h2t:+.1f}]")

# WAPE per horyzont (wszystkie originy) i per origin (rozrzut)
rows = []
for n_, P_ in METH.items():
    r = {"metoda": n_}
    for h in range(4):
        m_ = MAINP
        a = np.where(m_, ACT[:, :, h], 0).sum(); e = np.where(m_, np.abs(P_[:, :, h] - ACT[:, :, h]), 0).sum()
        r[f"h={h + 1}"] = e / a
    rows.append(r)
print("WAPE per horyzont (top 20, wszystkie originy):")
display(pd.DataFrame(rows).round(3))

po = pd.DataFrame({n_: np.where(per_origin(S4(P_), MAINP)[:, 0] > 0, per_origin(S4(P_), MAINP)[:, 1] / per_origin(S4(P_), MAINP)[:, 0], np.nan)
                   for n_, P_ in METH.items()}, index=[d.strftime("%m-%d") for d in orig_dates])
po["zbiór"] = np.where(TUNE, "strojenie", "test")
print("WAPE per origin (rozrzut między punktami backtestu):")
display(po.round(3))

**Co widzę.** Na tych samych 2140 punktach (top 20 × lokalizacja): **M2 (LightGBM): WAPE 12,6%** [11,6%; 14,1%] na wszystkich originach, 13,8% na strojeniowych, **11,1%** [10,4%; 11,8%] na **testowych** (bias +0,7%); B1 20,7% [18,7%; 23,0%] (test 19,1%); M1 18,7% [15,4%; 23,7%]; średnia M1+M2 14,1%. Sparowana przewaga nad B1 (wszystkie originy): M2 **−8,0 p.p.** [−9,7; −6,5], M1 −2,0 p.p. [−4,7; +2,2] (niejednoznaczne), M1+M2 −6,6 p.p. [−8,7; −4,1]. Kombinacja M1+M2 na originach strojeniowych (15,7%) jest **gorsza** od samego M2 (13,8%), czyli nie spełnia z góry ustalonego warunku (poprawa ≥ 0,5 p.p. względem najlepszego składnika) i **odrzucam ją**. WAPE per horyzont: M2 17,5% / 18,9% / 19,6% / 20,7% (B1: 24,4% / 25,4% / 26,0% / 30,0%). Per origin M2 jest lepszy od B1 w **18 z 18** originów (różnice od 2,6 do 15,5 p.p.).

**H2 (pooling): potwierdzona** wg progów z 6.1: WAPE(M1) − WAPE(M2) = **+4,4 p.p.** [+2,3; +6,3] na originach testowych (próg ≥ 3 p.p., dolna granica > 0), obciążenie M2 +0,7% vs M1 −2,1% (warunek |bias M2| ≤ |bias M1| + 3 p.p. spełniony). **Zastrzeżenie:** M1 i M2 różnią się nie tylko poolingiem, ale też dostępem do cech (kalendarz, poziom modelu, sezon), których M1 nie widzi; H2 pokazuje przewagę „modelu globalnego z kontekstem" nad „modelem per seria", a udział samego poolingu pokazuje dopiero ablacja z 6.7 (+1,3 p.p. po usunięciu grupy pooling). Różnica na originach strojeniowych jest większa (+7,4 p.p.), ale przedział szerszy ([+3,2; +14,5]).

**Wniosek / decyzja.** Metodą główną jest **M2 z konfiguracją zamrożoną w 6.7**. Poprawa względem dzisiejszej praktyki jest duża (ok. 8 p.p. WAPE, błąd względny mniejszy o ok. 40%) i stabilna między originami. **Sceptycznie:** tak duża poprawa wymaga sprawdzenia wycieku informacji z przyszłości; robię to w przeglądzie krytycznym (6.13: test zatrucia cech, koncentracja błędu, leave-one-origin-out).

## 6.10. H5 (kryterium 30%) i przedziały prognozy 80%

**Po co:** (1) Zespół zgłosił kryterium „WAPE sumy 4 tygodni < 30% dla top 20". Sprawdzam, czy jest sensowne: ile błędu wynika z samego losowego szumu sprzedaży (liczby sztuk są małe, więc nawet idealna prognoza średniej myli się o losową wielkość) oraz jak ma się do wyniku dzisiejszej praktyki. **Dolna granica szumu:** symuluję sprzedaż jako rozkład Poissona wokół prognozy M2 (czyli zakładam, że M2 trafia w prawdziwą średnią) i liczę WAPE takiej symulacji; to oszacowanie błędu, poniżej którego nie zejdzie żadna metoda. Dodatkowo mierzę nadmiar rozrzutu (overdispersion) `φ = Σ(A−P)² / ΣP` (dla czystego Poissona φ ≈ 1; wyższa wartość to błąd modelu lub dodatkowa zmienność popytu).
(2) **Przedziały 80%:** kalibruję je wyłącznie na originach strojeniowych: błąd skalowany `z = (A − P) / √(P + 1)` dla wszystkich ocenianych serii, kwantyle 10% i 90% tego błędu, przedział = `P + q · √(P + 1)` (obcięty od dołu do 0). Pokrycie sprawdzam na originach testowych. Pierwiastek z prognozy wynika z tego, że rozrzut sprzedaży rośnie wolniej niż jej poziom.

In [ ]:
rg_ = np.random.default_rng(SEED)

def noise_floor(P4, mask, phi=1.0, reps=300):
    mu = np.clip(P4[mask], 1e-6, None)
    out = []
    for _ in range(reps):
        if phi <= 1.0001:
            a = rg_.poisson(mu)
        else:
            n_ = mu / (phi - 1.0)
            a = rg_.negative_binomial(n_, 1.0 / phi)
        out.append(np.abs(a - mu).sum() / a.sum())
    return float(np.mean(out)), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

P4_M2, P4_B1 = S4(P_M2), S4(P_B1)
for lab, sel in (("strojenie", TUNE), ("test", TEST)):
    m_ = MAINP & sel[:, None]
    phi = ((A4[m_] - P4_M2[m_]) ** 2).sum() / P4_M2[m_].sum()
    f1 = noise_floor(P4_M2, m_)
    f2 = noise_floor(P4_M2, m_, phi=phi)
    w_m2 = per_origin(P4_M2, MAINP)[sel].sum(0); w_b1 = per_origin(P4_B1, MAINP)[sel].sum(0)
    print(f"{lab:9s}| WAPE M2 {w_m2[1] / w_m2[0]:.1%} | WAPE B1 {w_b1[1] / w_b1[0]:.1%} | próg szumu Poissona (prawdziwa średnia = M2): {f1[0]:.1%} [{f1[1]:.1%}; {f1[2]:.1%}] | φ obserwowane = {phi:.2f} -> szum przy tym rozrzucie {f2[0]:.1%}")
print("Średnia prognoza 4 tyg. na punkt (top 20 x lokalizacja):", round(float(P4_M2[MAINP].mean()), 1), "szt. | mediana:", round(float(np.median(P4_M2[MAINP])), 1))

# Czy 30% jest „trudne": ile punktów (origin) B1 przekracza 30% i jaki jest WAPE na pojedynczych punktach origin
pb = per_origin(P4_B1, MAINP); pm = per_origin(P4_M2, MAINP)
print("Originy, w których WAPE B1 >= 30%:", int((pb[:, 1] / pb[:, 0] >= 0.30).sum()), "z", N_O, "| M2:", int((pm[:, 1] / pm[:, 0] >= 0.30).sum()))
for h in range(4):
    m_ = MAINP
    ab = np.where(m_, ACT[:, :, h], 0).sum(); eb = np.where(m_, np.abs(P_B1[:, :, h] - ACT[:, :, h]), 0).sum()
    print(f"  B1 horyzont {h + 1}: WAPE {eb / ab:.1%}", end=";")
print()

# --- przedziały 80% ---
def interval_calib(P4, mask_cal, q=(0.10, 0.90)):
    z = (A4[mask_cal] - P4[mask_cal]) / np.sqrt(P4[mask_cal] + 1.0)
    return np.quantile(z, q)

def interval(P4, qs):
    return np.clip(P4 + qs[0] * np.sqrt(P4 + 1.0), 0, None), P4 + qs[1] * np.sqrt(P4 + 1.0)

cal_mask = ELIG & TUNE[:, None] & ~np.isnan(P4_M2)
Q_M2 = interval_calib(P4_M2, cal_mask); Q_B1 = interval_calib(P4_B1, cal_mask)
print(f"Kwantyle błędu skalowanego (strojenie, wszystkie serie): M2 {np.round(Q_M2, 2)} | B1 {np.round(Q_B1, 2)}")
rows = []
for lab, P4_, qs in (("M2", P4_M2, Q_M2), ("B1", P4_B1, Q_B1)):
    lo_, hi_ = interval(P4_, qs)
    for zb, sel in (("strojenie (kalibracja)", TUNE), ("test", TEST)):
        for seg, mk in (("top 20", MAINP), ("wszystkie serie", ELIG & ~np.isnan(P4_))):
            m_ = mk & sel[:, None]
            cov = ((A4[m_] >= lo_[m_]) & (A4[m_] <= hi_[m_])).mean()
            rows.append({"metoda": lab, "zbiór": zb, "serie": seg, "pokrycie": cov, "mediana szerokości / prognoza": np.median((hi_[m_] - lo_[m_]) / np.maximum(P4_[m_], 1)), "punktów": int(m_.sum())})
display(pd.DataFrame(rows).round(3))

# Pokrycie wg wielkości prognozy (test, M2, wszystkie serie)
lo_, hi_ = interval(P4_M2, Q_M2)
m_ = ELIG & TEST[:, None] & ~np.isnan(P4_M2)
bins = pd.cut(P4_M2[m_], [0, 10, 40, 150, 1e9], labels=["<=10", "10-40", "40-150", ">150"])
cb = pd.DataFrame({"bin": bins, "in": (A4[m_] >= lo_[m_]) & (A4[m_] <= hi_[m_]), "below": A4[m_] < lo_[m_], "above": A4[m_] > hi_[m_]}).groupby("bin", observed=True).agg(punktów=("in", "size"), pokrycie=("in", "mean"), poniżej=("below", "mean"), powyżej=("above", "mean"))
print("Pokrycie przedziału 80% wg wielkości prognozy 4 tyg. (M2, test, wszystkie serie):")
display(cb.round(3))

**Co widzę (przedziały 80%).** Przedział oparty na błędzie skalowanym `√(P+1)` ma łączne pokrycie bliskie nominalnemu (M2, test: 78,9% dla top 20 i 82,1% dla wszystkich serii; B1 z szerokością dwukrotnie większą: 76%), ale pokrycie **zależy od wielkości prognozy**: dla serii o prognozie ≤ 10 szt. pokrycie to 99,7% (przedział za szeroki), a dla prognoz > 150 szt. tylko **72,7%** (za wąski; poza przedziałem 15% obserwacji poniżej i 12% powyżej). To znaczy, że rozrzut błędu rośnie szybciej niż pierwiastek z poziomu (prawdziwy popyt ma dodatkową zmienność, nie tylko losowość Poissona; potwierdza to φ ≈ 9–11 powyżej). **H5 (kryterium 30%):** M2 ma WAPE 11,1% na originach testowych i 12,6% ogółem, B1 20,7%; dolna granica szumu czystego Poissona wokół prognozy to **4–5%** (średnia prognoza na punkt ok. 240 szt. na 4 tygodnie), a przy rozrzucie obserwowanym (φ ≈ 9–11, czyli z uwzględnieniem błędu modelu i dodatkowej zmienności popytu) szum odpowiadałby ok. 12–16%. Granica nieredukowalnego błędu leży więc gdzieś między 4% a ok. 12%, i nie umiem jej zawęzić z tych danych.

**Wniosek / decyzja.** (1) **H5: kryterium „WAPE < 30%" jest osiągalne, ale nie jest wymagające:** dzisiejsza praktyka już je spełnia w ujęciu sumarycznym (20,7%) i nie spełnia go tylko dla horyzontu 4 tygodni (30,0%) oraz w 2 z 18 originów. Jako cel dla nowej metody jest zbyt luźne; sensowniejsze są kryteria względne: **poprawa WAPE o ≥ 5 p.p. względem B1 przy bias w zakresie ±5%** i **pokrycie przedziału 80% w zakresie 75–85%**, osobno dla grup (duże serie, nowe modele). (2) Przedziały potrzebują poprawki skalowania; poprawiam je poniżej, wybierając wykładnik skalowania wyłącznie na originach strojeniowych. **Przyznaję, że zauważyłem problem dopiero po obejrzeniu pokrycia na teście**, więc poprawka jest wprowadzona po fakcie (choć jej parametr dobieram bez testu) i traktuję jej wynik na teście jako sprawdzenie, a nie ocenę niezależną.

In [ ]:
bin_edges = [0, 10, 40, 150, 1e9]
def calib_gamma(P4, mask_cal, gamma, q=(0.10, 0.90)):
    return np.quantile((A4[mask_cal] - P4[mask_cal]) / (P4[mask_cal] + 1.0) ** gamma, q)

def cov_by_bin(P4, qs, gamma, mask):
    lo_, hi_ = np.clip(P4 + qs[0] * (P4 + 1.0) ** gamma, 0, None), P4 + qs[1] * (P4 + 1.0) ** gamma
    b = np.digitize(P4[mask], bin_edges[1:-1])
    ins = (A4[mask] >= lo_[mask]) & (A4[mask] <= hi_[mask])
    return np.array([ins[b == k].mean() if (b == k).sum() >= 30 else np.nan for k in range(4)]), ins.mean(), np.median((hi_[mask] - lo_[mask]) / np.maximum(P4[mask], 1))

rows = []
for gm in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
    qs = calib_gamma(P4_M2, cal_mask, gm)
    cb_, ov, wd = cov_by_bin(P4_M2, qs, gm, cal_mask)
    rows.append({"wykładnik γ": gm, "pokrycie ≤10": cb_[0], "10-40": cb_[1], "40-150": cb_[2], ">150": cb_[3], "pokrycie ogółem": ov,
                 "maks. odchylenie od 80%": np.nanmax(np.abs(cb_ - 0.8)), "mediana szerokości / prognoza": wd, "q10": qs[0], "q90": qs[1]})
res_gamma = pd.DataFrame(rows)
GAMMA = float(res_gamma.loc[res_gamma["maks. odchylenie od 80%"].idxmin(), "wykładnik γ"])
Q_G = calib_gamma(P4_M2, cal_mask, GAMMA)
print("Kalibracja WYŁĄCZNIE na originach strojeniowych (wszystkie serie); wybrany wykładnik γ =", GAMMA)
display(res_gamma.round(3))

def interval_g(P4):
    return np.clip(P4 + Q_G[0] * (P4 + 1.0) ** GAMMA, 0, None), P4 + Q_G[1] * (P4 + 1.0) ** GAMMA

rows = []
for zb, sel in (("strojenie (kalibracja)", TUNE), ("test", TEST)):
    for seg, mk in (("top 20", MAINP), ("wszystkie serie", ELIG & ~np.isnan(P4_M2))):
        m_ = mk & sel[:, None]
        cb_, ov, wd = cov_by_bin(P4_M2, Q_G, GAMMA, m_)
        rows.append({"zbiór": zb, "serie": seg, "pokrycie ogółem": ov, "≤10": cb_[0], "10-40": cb_[1], "40-150": cb_[2], ">150": cb_[3], "mediana szerokości / prognoza": wd, "punktów": int(m_.sum())})
print("Pokrycie przedziału 80% po poprawce skalowania (M2):")
display(pd.DataFrame(rows).round(3))

## 6.11. Błąd i obciążenie w przekrojach (M2 vs B1)

**Po co:** jedna liczba WAPE ukrywa, gdzie metoda zawodzi. Sprawdzam, czy przewaga M2 nad B1 jest wszędzie, czy tylko w części przekrojów, oraz czy któraś grupa ma wyraźne obciążenie (to byłaby gotowa wskazówka, gdzie planiści powinni korygować prognozę). Przekroje liczę na **wszystkich originach** (z konieczności: w jednej grupie na samych originach testowych jest za mało punktów); M2 miał dobierane cechy i hiperparametry na originach strojeniowych, więc to diagnostyka, nie ocena końcowa. Grupy z małą sprzedażą (mniej niż 500 szt. w przekroju) oznaczam jako niepewne.

In [ ]:
bc = lambda a: np.broadcast_to(a, (N_O, S))
CATN = np.array(sorted(cat_codes, key=cat_codes.get))
seg_defs = {
    "kategoria": (bc(cat_idx[None, :]), {k: CATN[k] for k in range(len(CATN))}),
    "lokalizacja": (bc(sto_idx[None, :]), {k: SER.store[SER.store_key == k + 1].iloc[0] for k in range(6)}),
    "wiek serii (tyg.)": (np.digitize(AGE, [7, 13, 27]), {0: "1-6", 1: "7-12", 2: "13-26", 3: ">=27"}),
    "miesiąc origin": (bc(np.array([d.month for d in orig_dates])[:, None]), {m_: f"2026-{m_:02d}" for m_ in range(1, 9)}),
}
PROMO = np.stack([DISC[o:o + 4].max(0) > 0.05 for o in ORIG])
seg_defs["rabat w tyg. docelowych"] = (PROMO.astype(int), {0: "bez rabatu", 1: "z rabatem > 5%"})

def seg_table(name, grp, labels, mask, methods=(("B1", P_B1), ("M2", P_M2))):
    rows = []
    for k, lab in labels.items():
        m_ = mask & (grp == k)
        if not m_.any():
            continue
        a = A4[m_].sum()
        r = {"przekrój": name, "grupa": lab, "punktów": int(m_.sum()), "sprzedaż 4-tyg. [szt.]": int(a)}
        for n_, P_ in methods:
            p = S4(P_)
            r[f"WAPE {n_}"] = np.abs(p[m_] - A4[m_]).sum() / a
            r[f"bias {n_}"] = (p[m_].sum() - a) / a
        r["niepewne"] = "tak" if a < 500 else ""
        rows.append(r)
    return pd.DataFrame(rows)

seg_all = pd.concat([seg_table(n_, g_, l_, MAINP) for n_, (g_, l_) in seg_defs.items()], ignore_index=True)
with pd.option_context("display.width", 200, "display.max_columns", 20):
    display(seg_all.round(3))

**Co widzę (wszystkie 18 originów, top 20 × lokalizacja; WAPE M2 / B1, bias M2).** **M2 jest lepszy od B1 w każdym z 27 przekrojów.** Gdzie jest najtrudniej: (1) **Kurtki i Swetry**: WAPE M2 38,9% i 37,7% (B1: 78,6% i 75,7%), małe wolumeny (6–9 tys. szt. łącznie w sumach 4-tygodniowych), bias M2 +6,1% i +13,9% (B1: +16,2% i +44,0%); pozostałe kategorie 10,7–12,8% (Jeansy, T-shirty, Sukienki, Koszule), Akcesoria 19,5%. (2) **Wiek serii 13–26 tygodni**: WAPE 27,0%, **bias +19,7%** (B1: 49,5% i +47,5%): modele po szczycie sprzedaży, schodzące w kierunku wyprzedaży, M2 zawyża. Wiek 1–6 tyg.: 16,4%, bias −6,4% (B1: 38,4%, −37,8%); wiek ≥ 27 tyg.: 11,0%. (3) **Miesiące:** najtrudniejszy luty (20,2%) i styczeń (15,8%), reszta 10,7–14,0%; bias M2 −6,1% w marcu i +6,5% w lipcu. (4) **Lokalizacje:** najsłabiej Kraków Rynek (14,7%, bias −3,5%; B1 22,5%), pozostałe 11,7–13,7%. (5) **Tygodnie docelowe z rabatem > 5%** (167 punktów, 8% próby): WAPE 21,0%, bias +9,7% wobec 12,2% i −0,5% bez rabatu.

**Wniosek.** Przewaga M2 nad B1 jest powszechna, a nie wynikiem jednego przekroju. Najsłabsze miejsca to **kurtki i swetry** (sezonowość znana z jednego roku, mała sprzedaż), **modele schodzące ze szczytu** (zawyżanie o ok. 20%) i **tygodnie z rabatem** (zawyżanie o ok. 10%: rabatów w historii jest tylko kilka okien, więc model nie nauczył się ich wpływu; z sekcji 4.8 wiemy zresztą, że efekt rabatu na ilość jest mały i niejednoznaczny). Dla zleceniodawcy: prognozie dla kurtek i swetrów warto dawać mniejszą wagę i większy margines, a modele po szczycie sezonu przeglądać ręcznie. Rabaty nie dotyczą okna W39–W42 (brak akcji), ale **kurtki i swetry** dotyczą: to początek ich sezonu. Przekroje liczone na wszystkich originach, więc nie są ocenianiem końcowym.

## 6.12. Stabilność i wrażliwość na założenia

**Po co:** wynik jest wiarygodny, jeśli nie zależy od jednego okresu, jednej serii ani jednej decyzji analityka. Sprawdzam (wszystko na punktach głównego porównania, top 20 × lokalizacja, jeśli nie napisano inaczej): (a) wykluczenie serii „podejrzanych" (z flagą braku towaru lub długą przerwą w sprzedaży); (b) wykluczenie Krakowa (przebudowa); (c) inna definicja „top N" (10, 20, 40, oraz wg 13 tygodni zamiast 8); (d) inne okno uczenia M2 (tylko ostatnie 26 tygodni zamiast całej historii); (e) trzy różne ziarna losowe M2; (f) ocena z tygodniami z flagą braku towaru włączonymi do oceny.

In [ ]:
def wape_mask(P, mask, sel=None):
    st = per_origin(S4(P), mask)
    s = st.sum(0) if sel is None else st[sel].sum(0)
    return s[1] / s[0], (s[2] - s[0]) / s[0]

rows = []
def add_row(lab, mask, Pb=P_B1, Pm=P_M2):
    for zb, sel in (("wszystkie", None), ("test", TEST)):
        wb, bb = wape_mask(Pb, mask, sel); wm, bm = wape_mask(Pm, mask, sel)
        rows.append({"wariant": lab, "zbiór": zb, "punktów": int(mask.sum()), "WAPE B1": wb, "WAPE M2": wm, "Δ (B1−M2) [p.p.]": 100 * (wb - wm), "bias M2": bm})

add_row("bazowy (top 20, bez flag)", MAINP)
susp_series = np.zeros(S, bool)
susp_series[np.where(F.any(0))[0]] = True
for c in range(S):                                  # serie z przebiegiem >= 5 zer wewnątrz życia
    z_ = (Y[START_A[c]:, c] == 0).astype(int)
    run = 0
    for v in z_:
        run = run + 1 if v else 0
        if run >= 5:
            susp_series[c] = True
            break
print("Serie podejrzane (flaga lub przebieg >= 5 zer):", int(susp_series.sum()), "z", S, "| z tego w top 20 (dowolny origin):", int((susp_series & TOP20.any(0)).sum()))
add_row("bez serii podejrzanych", MAINP & ~susp_series[None, :])
add_row("bez Krakowa", MAINP & ~KRK[None, :])

def top_mask(n, weeks):
    tm = np.zeros((N_O, S), bool)
    for i, o in enumerate(ORIG):
        s_ = pd.Series(np.nansum(Y[o - weeks:o], axis=0)).groupby(SER.model.values).sum()
        tm[i] = SER.model.isin(s_.nlargest(n).index).to_numpy()
    return tm
for n_, w_ in ((10, 8), (40, 8), (20, 13)):
    add_row(f"top {n_} wg {w_} tyg.", ALLOK & top_mask(n_, w_))
add_row("wszystkie serie (nie tylko top 20)", ALLOK)
# tygodnie z flagą włączone do oceny (cel surowy, bez wyłączania)
mask_with_flags = HIST_OK & TOP20 & ~np.isnan(S4(P_M2))
add_row("top 20 z włączonymi tyg. z flagą", mask_with_flags)

# (d) krótkie okno uczenia i (e) ziarna
def run_m2_custom(feats, params, window=None):
    out = np.full((N_O, S, 4), np.nan)
    for i, o in enumerate(ORIG):
        tr = KNOWN[(KNOWN.tw <= o - 1) & ((KNOWN.tw >= o - window) if window else True)]
        te = ROWS[ROWS.t == o]
        mdl = lgb.LGBMRegressor(**params).fit(tr[feats], tr.y, categorical_feature=[f for f in CAT_F if f in feats])
        out[i, te.c.to_numpy(), te.h.to_numpy() - 1] = mdl.predict(te[feats])
    return out
t0 = time.time()
P_M2_w26 = run_m2_custom(FEATS_SEL, PARAMS_SEL, window=26)
add_row("M2 uczony na ostatnich 26 tyg.", MAINP, Pm=P_M2_w26)
P_M2_seeds = []
for sd in (1, 2, 3):
    P_M2_seeds.append(run_m2_custom(FEATS_SEL, dict(PARAMS_SEL, random_state=SEED + sd)))
    add_row(f"M2, ziarno +{sd}", MAINP, Pm=P_M2_seeds[-1])
print(f"Czas: {time.time() - t0:.0f} s")
sens = pd.DataFrame(rows)
display(sens.round(3))

**Co widzę (przewaga B1 − M2 w p.p. WAPE; wszystkie originy, test w nawiasie).** Wariant bazowy 8,0 (8,0); bez serii podejrzanych (42 z 216 serii ma flagę braku towaru lub przebieg ≥ 5 zer) 7,9 (8,1); bez Krakowa 8,1 (8,2); top 10 6,8 (6,3); „top 40" 8,6 (9,1) (uwaga: modeli jest 36, więc to to samo, co wszystkie serie); top 20 wg 13 tygodni 8,0 (7,9); z włączonymi tygodniami z flagą braku towaru do oceny 7,8 (7,7) (WAPE M2 13,2% vs B1 21,0%); M2 uczony tylko na ostatnich 26 tygodniach 5,6 (5,4) (WAPE M2 15,1% zamiast 12,6%); trzy inne ziarna losowe M2: 7,9–8,1 (WAPE M2 12,6–12,7%, praktycznie bez zmian). Bias M2 mieści się w granicach −1% … +2%.

**Wniosek.** Wynik jest **odporny** na wykluczenie podejrzanych serii, Krakowa, definicję top N i na ziarno losowe. Jedyne istotne pogorszenie to krótsze okno uczenia (M2 traci ok. 2,5 p.p.), czyli **historia ma znaczenie** (sezonowość wymaga długiego okresu), a dla top 10 przewaga jest mniejsza (6,8 p.p.), bo największe modele mają bardziej stabilne serie i B1 jest tam lepszy. Włączenie tygodni z flagą braku towaru do oceny nie zmienia wniosku (7,8 p.p.).

## 6.13. Przegląd krytyczny: próba obalenia własnego wyniku

**Po co:** przed sformułowaniem wniosków sprawdzam pięć rzeczy, które najczęściej psują wyniki prognoz: (1) czy jakakolwiek cecha używa informacji z po origin (test „zatrucia": w kopii danych podmieniam wszystkie tygodnie od origin na absurdalną wartość 10^6 i sprawdzam, czy cechy się zmieniły); (2) czy WAPE nie jest zdominowany kilkoma seriami; (3) czy wynik nie zależy od jednego originu; (4) czy tygodnie z podejrzeniem braku towaru nie fałszują oceny (wariant z włączonymi flagami jest w 6.12); (5) czy porównania są na identycznych punktach.

In [ ]:
leak_rows = []
for t_ in (int(ORIG[0]), int(ORIG[len(ORIG) // 2]), int(ORIG[-1])):
    YI2 = YI.copy(); YI2[t_:] = 1e6
    MT2, CT2 = aggregates(YI2)
    R2 = feature_rows(t_, YI2, MT2, CT2)
    R1 = ROWS[ROWS.t == t_].reset_index(drop=True)
    bad = [f for f in FEATS_ALL if not np.allclose(R1[f].to_numpy(float), R2[f].to_numpy(float), equal_nan=True)]
    leak_rows.append({"origin": WX[t_].date(), "cechy zmienione po zatruciu": bad or "brak"})
print("Test zatrucia cech (wartości od origin podmienione na 1e6):")
display(pd.DataFrame(leak_rows))

# (2) koncentracja błędu
m_ = MAINP
err_c = pd.DataFrame({"model": np.broadcast_to(SER.model.to_numpy()[None, :], (N_O, S))[m_], "store": np.broadcast_to(SER.store.to_numpy()[None, :], (N_O, S))[m_],
                      "err_M2": np.abs(P4_M2 - A4)[m_], "err_B1": np.abs(P4_B1 - A4)[m_], "act": A4[m_]})
g = err_c.groupby(["model", "store"]).sum(numeric_only=True)
for n_ in ("M2", "B1"):
    s_ = g[f"err_{n_}"].sort_values(ascending=False)
    print(f"{n_}: serie (model x lokalizacja) w ocenie: {len(s_)}; 5 największych serii = {s_.head(5).sum() / s_.sum():.1%} błędu, 10 = {s_.head(10).sum() / s_.sum():.1%}; udział 5 największych serii w sprzedaży: {g.act.sort_values(ascending=False).head(5).sum() / g.act.sum():.1%}")
top5 = g.err_M2.sort_values(ascending=False).head(5).index
g2 = g.drop(top5)
print(f"WAPE po usunięciu 5 serii o największym błędzie M2: M2 {g2.err_M2.sum() / g2.act.sum():.1%} | B1 {g2.err_B1.sum() / g2.act.sum():.1%} (przed: M2 {g.err_M2.sum() / g.act.sum():.1%}, B1 {g.err_B1.sum() / g.act.sum():.1%})")

# (3) zależność od jednego originu: różnica WAPE B1 − M2 per origin i leave-one-origin-out
sb, sm_ = per_origin(P4_B1, MAINP), per_origin(P4_M2, MAINP)
d_o = sb[:, 1] / sb[:, 0] - sm_[:, 1] / sm_[:, 0]
loo = [(sb.sum(0)[1] - sb[i, 1]) / (sb.sum(0)[0] - sb[i, 0]) - (sm_.sum(0)[1] - sm_[i, 1]) / (sm_.sum(0)[0] - sm_[i, 0]) for i in range(N_O)]
print(f"Przewaga M2 nad B1 (p.p. WAPE) per origin: min {100 * d_o.min():.1f}, mediana {100 * np.median(d_o):.1f}, max {100 * d_o.max():.1f}; M2 lepszy w {int((d_o > 0).sum())} z {N_O} originów")
print(f"Leave-one-origin-out: przewaga M2 od {100 * min(loo):.1f} do {100 * max(loo):.1f} p.p.")

# (5) identyczność punktów
print("Punkty porównania identyczne dla wszystkich metod:", int(MAINP.sum()), "| NaN w prognozach na tych punktach: B1", int(np.isnan(P4_B1[MAINP]).sum()), "M1", int(np.isnan(S4(P_M1)[MAINP]).sum()), "M2", int(np.isnan(P4_M2[MAINP]).sum()))

# 7. Results & Conclusions

## 7.1. Prognoza na tygodnie 2026-W39 … W42 (top 20 modeli × lokalizacja)

**Po co:** to jest produkt dla zleceniodawcy. Konfiguracja M2 (zestaw cech, hiperparametry) jest ta sama, którą zamroziłem na originach strojeniowych; jedyna różnica to dane uczące: teraz model uczy się na całej dostępnej historii (do niedzieli 2026-09-20). Przedziały 80% pochodzą z kalibracji na originach strojeniowych (sekcja 6.10). Obok wartości centralnej podaję prognozę dzisiejszej praktyki (B1), żeby było widać, gdzie decyzja się zmieni. Flagi: **nowy model** (≤ 6 tygodni sprzedaży: wyższe ryzyko błędu, backtest dał tu WAPE ok. 26%); **podejrzenie braku towaru** (flaga z 5.2 w jednym z dwóch ostatnich tygodni: ostatnia sprzedaż nie odzwierciedla popytu, a prognoza używa oczekiwania zamiast zera); **niska pewność** (szeroki przedział względem prognozy lub prognoza < 20 szt.). **ZAŁOŻENIE:** w W39–W42 nie ma akcji rabatowych (potwierdzone przez dział cen), więc cecha rabatu = 0; kalendarz: brak świąt i niedziel handlowych w tym oknie (sprawdzam niżej).

In [ ]:
print("Origin:", WX[T].date(), "| ISO:", WX[T].isocalendar().week, "| tygodnie prognozy:", [str(d.date()) for d in WX[T:T + 4]])
print("Kalendarz W39–W42: dni otwarcia sklepów", list(OD[T:T + 4, 0].astype(int)), "| święta pon-sob:", list(CALX.holidays_monsat.iloc[T:T + 4].astype(int)), "| niedziele handlowe:", list(CALX.trading_sunday.iloc[T:T + 4].astype(int)))

tr = KNOWN[KNOWN.tw <= T - 1]
te = ROWS[ROWS.t == T].copy()
mdl_fin = lgb.LGBMRegressor(**PARAMS_SEL).fit(tr[FEATS_SEL], tr.y, categorical_feature=[f for f in CAT_F if f in FEATS_SEL])
te["pred"] = mdl_fin.predict(te[FEATS_SEL])
fc_w = te.pivot_table(index="c", columns="h", values="pred").reindex(range(S))
assert fc_w.notna().all().all(), "brak prognozy dla części serii"
P4_fin = fc_w.sum(1).to_numpy()

s8 = pd.Series(np.nansum(Y[T - 8:T], axis=0)).groupby(SER.model.values).sum()
top_models = list(s8.nlargest(20).index)
lo_f, hi_f = interval_g(P4_fin)
age_T = T - START_A
fin = pd.DataFrame({"model": SER.model, "lokalizacja": SER.store, "kategoria": SER.category,
                    "B1_4tyg": 4 * np.nanmean(Y[T - 4:T], axis=0), "M2_4tyg": P4_fin, "p10": lo_f, "p90": hi_f,
                    "ost_4tyg_rzecz": np.nansum(Y[T - 4:T], axis=0)})
for h in range(1, 5):
    fin[f"W{38 + h}"] = fc_w[h].to_numpy()
fin["nowy_model"] = age_T <= 6
fin["podejrzenie_braku_towaru"] = F[T - 2:T].any(0)
fin["niska_pewnosc"] = ((fin.p90 - fin.p10) / np.maximum(fin.M2_4tyg, 1) > 0.6) | (fin.M2_4tyg < 20)
top = fin[fin.model.isin(top_models)].copy()
print("Top 20 modeli (sprzedaż w 8 tyg. przed origin):", ", ".join(top_models))
print("Modele z ≤ 6 tyg. sprzedaży wśród top 20:", sorted(top[top.nowy_model].model.unique()), "| serie z podejrzeniem braku towaru:", top[top.podejrzenie_braku_towaru][["model", "lokalizacja"]].values.tolist())
print(f"Suma 4 tyg. (top 20, wszystkie lokalizacje): M2 {top.M2_4tyg.sum():,.0f} szt. [pojedyncze przedziały nie sumują się] | B1 {top.B1_4tyg.sum():,.0f} | ostatnie 4 tyg. rzeczywiste {top.ost_4tyg_rzecz.sum():,.0f}")

cat_cmp = top.groupby("kategoria")[["ost_4tyg_rzecz", "B1_4tyg", "M2_4tyg"]].sum()
cat_cmp["M2 / ostatnie 4 tyg."] = cat_cmp.M2_4tyg / cat_cmp.ost_4tyg_rzecz
print("Prognoza vs ostatnie 4 tygodnie wg kategorii (top 20):")
display(cat_cmp.round(2))

# Tabela model x lokalizacja: wartość centralna [p10; p90]
def fmt(r):
    s = f"{r.M2_4tyg:.0f} [{r.p10:.0f}; {r.p90:.0f}]"
    return s + ("*" if r.nowy_model else "") + ("^" if r.podejrzenie_braku_towaru else "")
top["txt"] = top.apply(fmt, axis=1)
tab = top.pivot(index="model", columns="lokalizacja", values="txt").reindex(top_models)
tab.insert(0, "kategoria", [cat_of[m_] for m_ in tab.index])
tab["razem M2"] = top.groupby("model").M2_4tyg.sum().reindex(top_models).round(0).astype(int)
tab["razem B1"] = top.groupby("model").B1_4tyg.sum().reindex(top_models).round(0).astype(int)
print("Prognoza sumy 4 tygodni (2026-W39…W42): wartość centralna [przedział 80%]; * nowy model, ^ podejrzenie braku towaru")
with pd.option_context("display.width", 250, "display.max_columns", 20):
    display(tab)
fin_out = top.sort_values(["model", "lokalizacja"])[["model", "lokalizacja", "kategoria", "W39", "W40", "W41", "W42", "M2_4tyg", "p10", "p90", "B1_4tyg", "nowy_model", "podejrzenie_braku_towaru", "niska_pewnosc"]].round(1)
fin_out.to_csv(CACHE.parent / "prognoza-W39-W42-top20.csv", index=False, encoding="utf-8")
print("Zapisano prognoza-W39-W42-top20.csv:", len(fin_out), "wierszy | flagi: nowy model", int(top.nowy_model.sum()), "| podejrzenie braku towaru", int(top.podejrzenie_braku_towaru.sum()), "| niska pewność", int(top.niska_pewnosc.sum()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
x = np.arange(N_O)
for n_, st_ in (("B1 (praktyka)", "o--"), ("M1 (ETS)", "s--"), ("M2 (LightGBM)", "o-")):
    ax[0].plot(x, 100 * po[n_], st_, label=n_)
ax[0].axvspan(TUNE.sum() - 0.5, N_O - 0.5, color="grey", alpha=0.15, label="originy testowe")
ax[0].set_xticks(x); ax[0].set_xticklabels(po.index, rotation=60, fontsize=8)
ax[0].set_xlabel("Origin backtestu (poniedziałek, 2026)"); ax[0].set_ylabel("WAPE sumy 4 tygodni [%]")
ax[0].set_title("Backtest: błąd na każdym origin (top 20 × lokalizacja)"); ax[0].legend(fontsize=8)
show_m = top_models[:6]
colors = plt.cm.tab10(np.arange(6))
for k, m_ in enumerate(show_m):
    ser = np.nansum(Y[:, style_cols[m_]], axis=1)[-26:]
    ax[1].plot(WX[T - 26:T], ser, color=colors[k], label=m_)
    fc_m = fin[fin.model == m_][["W39", "W40", "W41", "W42"]].sum().to_numpy()
    ax[1].plot(WX[T:T + 4], fc_m, "--", color=colors[k])
ax[1].axvline(WX[T], color="k", lw=0.8)
ax[1].set_xlabel("Tydzień (poniedziałek)"); ax[1].set_ylabel("Sprzedaż modelu, wszystkie lokalizacje [szt./tydz.]")
ax[1].set_title("6 największych modeli: ostatnie 26 tygodni i prognoza M2 (linia przerywana)"); ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()